<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/VIB_PLACEBO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LOAD DỮ LIỆU

XỬ LÝ FILE TARGET

In [ ]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

# =========================
# MOUNT GOOGLE DRIVE
# =========================
drive.mount("/content/drive")

# =========================
# THƯ MỤC GOOGLE DRIVE
# =========================
BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(BASE_DIR, exist_ok=True)

# File đầu vào và đầu ra
INPUT_FILE = os.path.join(BASE_DIR, "6.Data_Card.xlsx")
OUTPUT_FILE = os.path.join(BASE_DIR, "target_processed.csv")

# =========================
# KIỂM TRA FILE ĐẦU VÀO
# =========================
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Không tìm thấy file:\n{INPUT_FILE}\n"
        "Hãy kiểm tra file 6.Data_Card.xlsx đã nằm trong thư mục ELAPSPLACEBO chưa."
    )

# =========================
# LOAD FILE CARD
# =========================
card = pd.read_excel(INPUT_FILE)

# Chuẩn hóa tên cột
card.columns = card.columns.str.strip().str.upper()

# Kiểm tra các cột bắt buộc
required_columns = ["CUSTOMER_NUMBER", "COUNT_CREDITCARD", "MONTH"]

missing_columns = [
    col for col in required_columns
    if col not in card.columns
]

if missing_columns:
    raise ValueError(
        f"Thiếu các cột bắt buộc: {missing_columns}\n"
        f"Các cột hiện có: {card.columns.tolist()}"
    )

# =========================
# CHỌN CỘT CẦN DÙNG
# =========================
target = card[
    ["CUSTOMER_NUMBER", "COUNT_CREDITCARD", "MONTH"]
].copy()

# Chuyển kiểu dữ liệu
target["COUNT_CREDITCARD"] = pd.to_numeric(
    target["COUNT_CREDITCARD"],
    errors="coerce"
).fillna(0)

target["MONTH"] = pd.to_datetime(
    target["MONTH"],
    errors="coerce"
)

# Loại các dòng không có CUSTOMER_NUMBER
target = target.dropna(subset=["CUSTOMER_NUMBER"])

# =========================
# GOM THEO KHÁCH HÀNG
# =========================
summary = (
    target
    .groupby("CUSTOMER_NUMBER", as_index=False)
    .agg(
        MAX_CARD=("COUNT_CREDITCARD", "max"),
        MIN_MONTH=("MONTH", "min"),
        MAX_MONTH=("MONTH", "max")
    )
)

# Có thẻ = 1, không có thẻ = 0
summary["TARGET"] = (
    summary["MAX_CARD"] > 0
).astype(int)

# Có thẻ: lấy tháng sớm nhất
# Không có thẻ: lấy tháng muộn nhất
summary["TARGET_MONTH"] = summary["MIN_MONTH"]

summary.loc[
    summary["TARGET"] == 0,
    "TARGET_MONTH"
] = summary.loc[
    summary["TARGET"] == 0,
    "MAX_MONTH"
]

# =========================
# GIỮ CỘT CẦN THIẾT
# =========================
target_final = summary[
    ["CUSTOMER_NUMBER", "TARGET", "TARGET_MONTH"]
].copy()

# =========================
# SAVE FILE VÀO GOOGLE DRIVE
# =========================
target_final.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# =========================
# HIỂN THỊ KẾT QUẢ
# =========================
print(target_final.head())
print("Shape:", target_final.shape)
print("Số adopter:", (target_final["TARGET"] == 1).sum())
print("Số non-adopter:", (target_final["TARGET"] == 0).sum())
print(f"Saved: {OUTPUT_FILE}")

Mounted at /content/drive
   CUSTOMER_NUMBER  TARGET TARGET_MONTH
0                0       0   2019-12-31
1                3       0   2019-12-31
2                8       1   2019-05-31
3                9       0   2019-12-31
4               13       0   2019-12-31
Shape: (150459, 3)
Số adopter: 42374
Số non-adopter: 108085
Saved: /content/drive/MyDrive/ELAPSPLACEBO/target_processed.csv


4️⃣ Nối CUSTOMER với TARGET

In [ ]:
import os
import pandas as pd
from google.colab import drive

# =========================
# MOUNT GOOGLE DRIVE
# =========================
drive.mount("/content/drive")

# =========================
# THƯ MỤC LÀM VIỆC
# =========================
BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

CUSTOMER_FILE = os.path.join(BASE_DIR, "1.Data_Customer.csv")
TARGET_FILE = os.path.join(BASE_DIR, "target_processed.csv")
OUTPUT_FILE = os.path.join(
    BASE_DIR,
    "step2_customer_target_filtered.csv"
)

# =========================
# KIỂM TRA FILE ĐẦU VÀO
# =========================
for file_path in [CUSTOMER_FILE, TARGET_FILE]:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Không tìm thấy file:\n{file_path}"
        )

# =========================
# LOAD FILES
# =========================
customer = pd.read_csv(
    CUSTOMER_FILE,
    low_memory=False
)

target = pd.read_csv(
    TARGET_FILE,
    low_memory=False
)

# =========================
# CHUẨN HÓA TÊN CỘT
# =========================
customer.columns = (
    customer.columns
    .str.strip()
    .str.upper()
)

target.columns = (
    target.columns
    .str.strip()
    .str.upper()
)

# =========================
# KIỂM TRA CỘT BẮT BUỘC
# =========================
required_customer_columns = [
    "CUSTOMER_NUMBER",
    "CLIENT_CREATE_DATE"
]

required_target_columns = [
    "CUSTOMER_NUMBER",
    "TARGET",
    "TARGET_MONTH"
]

missing_customer = [
    col for col in required_customer_columns
    if col not in customer.columns
]

missing_target = [
    col for col in required_target_columns
    if col not in target.columns
]

if missing_customer:
    raise ValueError(
        f"File customer thiếu cột: {missing_customer}"
    )

if missing_target:
    raise ValueError(
        f"File target thiếu cột: {missing_target}"
    )

# =========================
# CHUẨN HÓA CUSTOMER_NUMBER
# Tránh lỗi một file là số, file kia là chuỗi
# =========================
customer["CUSTOMER_NUMBER"] = (
    customer["CUSTOMER_NUMBER"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

target["CUSTOMER_NUMBER"] = (
    target["CUSTOMER_NUMBER"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# =========================
# CHUYỂN KIỂU NGÀY THÁNG
# =========================
customer["CLIENT_CREATE_DATE"] = pd.to_datetime(
    customer["CLIENT_CREATE_DATE"],
    errors="coerce"
)

target["TARGET_MONTH"] = pd.to_datetime(
    target["TARGET_MONTH"],
    errors="coerce"
)

# =========================
# LOẠI DÒNG THIẾU THÔNG TIN CỐT LÕI
# =========================
customer = customer.dropna(
    subset=[
        "CUSTOMER_NUMBER",
        "CLIENT_CREATE_DATE"
    ]
)

target = target.dropna(
    subset=[
        "CUSTOMER_NUMBER",
        "TARGET_MONTH"
    ]
)

# Nếu một khách xuất hiện nhiều lần trong customer,
# giữ dòng có CLIENT_CREATE_DATE sớm nhất
customer = (
    customer
    .sort_values("CLIENT_CREATE_DATE")
    .drop_duplicates(
        subset=["CUSTOMER_NUMBER"],
        keep="first"
    )
)

# Nếu target có trùng khách, giữ dòng đầu tiên
target = target.drop_duplicates(
    subset=["CUSTOMER_NUMBER"],
    keep="first"
)

# =========================
# INNER JOIN
# GIỮ KHÁCH CÓ TRONG CẢ HAI FILE
# =========================
df = customer.merge(
    target,
    on="CUSTOMER_NUMBER",
    how="inner",
    validate="one_to_one"
)

print("Before filtering:", df.shape)

# =========================
# KIỂM TRA BẤT THƯỜNG THỜI GIAN
# =========================
invalid_mask = (
    df["TARGET_MONTH"] <
    df["CLIENT_CREATE_DATE"]
)

invalid_count = invalid_mask.sum()

print(
    "Customers with TARGET_MONTH before "
    "CLIENT_CREATE_DATE:",
    invalid_count
)

# Lưu riêng các dòng bất thường để kiểm tra
if invalid_count > 0:
    invalid_file = os.path.join(
        BASE_DIR,
        "step2_invalid_target_before_customer_creation.csv"
    )

    df.loc[invalid_mask].to_csv(
        invalid_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved invalid rows:", invalid_file)

# =========================
# XÓA KHÁCH CÓ TARGET_MONTH
# TRƯỚC CLIENT_CREATE_DATE
# =========================
df = df.loc[~invalid_mask].copy()

print("After filtering:", df.shape)

# =========================
# TẠO BIẾN THỜI GIAN QUAN HỆ
# Hữu ích cho bước placebo sau này
# =========================
df["TENURE_DAYS_AT_TARGET"] = (
    df["TARGET_MONTH"] -
    df["CLIENT_CREATE_DATE"]
).dt.days

# =========================
# SẮP XẾP DỮ LIỆU
# =========================
df = df.sort_values(
    by=["CUSTOMER_NUMBER"]
).reset_index(drop=True)

# =========================
# SAVE FILE
# =========================
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# =========================
# THỐNG KÊ
# =========================
print("Saved:", OUTPUT_FILE)
print()
print(df.head())

print("\nTarget distribution:")
print(
    df["TARGET"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nTenure summary:")
print(
    df["TENURE_DAYS_AT_TARGET"]
    .describe()
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Before filtering: (150459, 11)
Customers with TARGET_MONTH before CLIENT_CREATE_DATE: 51
Saved invalid rows: /content/drive/MyDrive/ELAPSPLACEBO/step2_invalid_target_before_customer_creation.csv
After filtering: (150408, 11)
Saved: /content/drive/MyDrive/ELAPSPLACEBO/step2_customer_target_filtered.csv

  CUSTOMER_NUMBER CLIENT_SEX CLIENT_CREATE_DATE        DATE_OF_BIRTH  \
0               0          F         2019-09-16  1988-10-08 00:00:00   
1          100009          M         2019-08-23  1995-09-09 00:00:00   
2          100018          M         2019-10-02  1989-01-16 00:00:00   
3          100019          M         2019-05-18  1972-07-28 00:00:00   
4          100026          F         2019-04-23  1975-05-26 00:00:00   

  STAFF_VIB IB_REGISTER_DATE EB_REGISTER_CHANNEL  SMS VERIFY_METHOD  TARGET  \
0         N       2019-09-17              BRANCH    N  

5️⃣ Loại khách mở thẻ trước khi mở tài khoản

In [ ]:
import os
import pandas as pd
from google.colab import drive

# =========================
# MOUNT GOOGLE DRIVE
# =========================
drive.mount("/content/drive")

# =========================
# THƯ MỤC GOOGLE DRIVE
# =========================
BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

CUSTOMER_FILE = os.path.join(BASE_DIR, "1.Data_Customer.csv")
TARGET_FILE = os.path.join(BASE_DIR, "target_processed.csv")
OUTPUT_FILE = os.path.join(
    BASE_DIR,
    "step2_customer_target_filtered.csv"
)

# =========================
# KIỂM TRA FILE CÓ TỒN TẠI
# =========================
for file_path in [CUSTOMER_FILE, TARGET_FILE]:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Không tìm thấy file:\n{file_path}"
        )

# =========================
# LOAD FILES
# =========================
customer = pd.read_csv(
    CUSTOMER_FILE,
    low_memory=False
)

target = pd.read_csv(
    TARGET_FILE,
    low_memory=False
)

# =========================
# CHUẨN HÓA TÊN CỘT
# =========================
customer.columns = (
    customer.columns
    .str.strip()
    .str.upper()
)

target.columns = (
    target.columns
    .str.strip()
    .str.upper()
)

# =========================
# KIỂM TRA CỘT BẮT BUỘC
# =========================
required_customer_cols = [
    "CUSTOMER_NUMBER",
    "CLIENT_CREATE_DATE"
]

required_target_cols = [
    "CUSTOMER_NUMBER",
    "TARGET",
    "TARGET_MONTH"
]

missing_customer_cols = [
    col for col in required_customer_cols
    if col not in customer.columns
]

missing_target_cols = [
    col for col in required_target_cols
    if col not in target.columns
]

if missing_customer_cols:
    raise ValueError(
        f"File 1.Data_Customer.csv thiếu cột: "
        f"{missing_customer_cols}"
    )

if missing_target_cols:
    raise ValueError(
        f"File target_processed.csv thiếu cột: "
        f"{missing_target_cols}"
    )

# =========================
# CHUẨN HÓA CUSTOMER_NUMBER
# Tránh một file là số, một file là chuỗi
# =========================
customer["CUSTOMER_NUMBER"] = (
    customer["CUSTOMER_NUMBER"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

target["CUSTOMER_NUMBER"] = (
    target["CUSTOMER_NUMBER"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# =========================
# CHUYỂN KIỂU DỮ LIỆU
# =========================
customer["CLIENT_CREATE_DATE"] = pd.to_datetime(
    customer["CLIENT_CREATE_DATE"],
    errors="coerce"
)

target["TARGET_MONTH"] = pd.to_datetime(
    target["TARGET_MONTH"],
    errors="coerce"
)

target["TARGET"] = pd.to_numeric(
    target["TARGET"],
    errors="coerce"
)

# =========================
# LOẠI DÒNG THIẾU DỮ LIỆU CỐT LÕI
# =========================
customer = customer.dropna(
    subset=[
        "CUSTOMER_NUMBER",
        "CLIENT_CREATE_DATE"
    ]
)

target = target.dropna(
    subset=[
        "CUSTOMER_NUMBER",
        "TARGET",
        "TARGET_MONTH"
    ]
)

target["TARGET"] = target["TARGET"].astype(int)

# =========================
# XỬ LÝ KHÁCH HÀNG BỊ TRÙNG
# =========================

# Nếu customer có nhiều dòng cho một khách,
# giữ ngày tạo quan hệ sớm nhất
customer = (
    customer
    .sort_values("CLIENT_CREATE_DATE")
    .drop_duplicates(
        subset="CUSTOMER_NUMBER",
        keep="first"
    )
)

# target_processed lý tưởng chỉ có 1 dòng/khách
target = target.drop_duplicates(
    subset="CUSTOMER_NUMBER",
    keep="first"
)

# =========================
# INNER JOIN
# GIỮ KHÁCH CÓ TRONG CẢ HAI FILE
# =========================
df = customer.merge(
    target,
    on="CUSTOMER_NUMBER",
    how="inner",
    validate="one_to_one"
)

print("Before filtering:", df.shape)

# =========================
# XÁC ĐỊNH DÒNG BẤT THƯỜNG
# TARGET_MONTH TRƯỚC NGÀY TẠO QUAN HỆ
# =========================
invalid_mask = (
    df["TARGET_MONTH"] <
    df["CLIENT_CREATE_DATE"]
)

print(
    "Số khách có TARGET_MONTH trước CLIENT_CREATE_DATE:",
    int(invalid_mask.sum())
)

# Lưu riêng dòng lỗi để kiểm tra
if invalid_mask.any():
    INVALID_FILE = os.path.join(
        BASE_DIR,
        "step2_invalid_dates.csv"
    )

    df.loc[invalid_mask].to_csv(
        INVALID_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("Đã lưu dòng bất thường tại:", INVALID_FILE)

# =========================
# XÓA DÒNG BẤT THƯỜNG
# =========================
df = df.loc[~invalid_mask].copy()

print("After filtering:", df.shape)

# =========================
# TẠO TENURE ĐỂ DÙNG CHO PLACEBO
# =========================
df["TENURE_DAYS_AT_TARGET"] = (
    df["TARGET_MONTH"] -
    df["CLIENT_CREATE_DATE"]
).dt.days

# =========================
# SẮP XẾP
# =========================
df = (
    df
    .sort_values("CUSTOMER_NUMBER")
    .reset_index(drop=True)
)

# =========================
# SAVE FILE
# =========================
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# =========================
# HIỂN THỊ KẾT QUẢ
# =========================
print("\nSaved:", OUTPUT_FILE)
print(df.head())

print("\nTarget distribution:")
print(
    df["TARGET"]
    .value_counts()
    .sort_index()
)

print("\nTenure summary:")
print(
    df.groupby("TARGET")["TENURE_DAYS_AT_TARGET"]
    .describe()
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Before filtering: (150459, 11)
Số khách có TARGET_MONTH trước CLIENT_CREATE_DATE: 51
Đã lưu dòng bất thường tại: /content/drive/MyDrive/ELAPSPLACEBO/step2_invalid_dates.csv
After filtering: (150408, 11)

Saved: /content/drive/MyDrive/ELAPSPLACEBO/step2_customer_target_filtered.csv
  CUSTOMER_NUMBER CLIENT_SEX CLIENT_CREATE_DATE        DATE_OF_BIRTH  \
0               0          F         2019-09-16  1988-10-08 00:00:00   
1          100009          M         2019-08-23  1995-09-09 00:00:00   
2          100018          M         2019-10-02  1989-01-16 00:00:00   
3          100019          M         2019-05-18  1972-07-28 00:00:00   
4          100026          F         2019-04-23  1975-05-26 00:00:00   

  STAFF_VIB IB_REGISTER_DATE EB_REGISTER_CHANNEL  SMS VERIFY_METHOD  TARGET  \
0         N       2019-09-17              BRANCH    N     SMART_OTP       0  

TẠO BẢNG PLACEBO

In [ ]:
# ============================================================
# PLACEBO CUTOFF GENERATION - 5 SEEDS
# Đọc và lưu trực tiếp trên Google Drive:
# /content/drive/MyDrive/ELAPSPLACEBO
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


# ============================================================
# 0. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/ELAPSPLACEBO"
)

INPUT_FILE = (
    BASE_DIR
    / "step2_customer_target_filtered.csv"
)

OUTPUT_DIR = (
    BASE_DIR
    / "placebo_cutoff_outputs"
)

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"
START_DATE_COL = "CLIENT_CREATE_DATE"
EVENT_DATE_COL = "TARGET_MONTH"

# Chạy 5 lần lấy mẫu độc lập
SEEDS = [1, 2, 3, 4, 5]

# Cho phép placebo cutoff đúng bằng T_END
ALLOW_CUTOFF_AT_T_END = True

# Tạo thư mục nếu chưa tồn tại
BASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project folder:", BASE_DIR)
print("Input file:", INPUT_FILE)
print("Output folder:", OUTPUT_DIR)


# ============================================================
# 2. CHECK INPUT FILE
# ============================================================

if not INPUT_FILE.exists():

    print("\nCác file hiện có trong thư mục ELAPSPLACEBO:")

    for item in BASE_DIR.iterdir():
        print("-", item.name)

    raise FileNotFoundError(
        "\nKhông tìm thấy file đầu vào:\n"
        f"{INPUT_FILE}\n\n"
        "Hãy kiểm tra file "
        "'step2_customer_target_filtered.csv' "
        "đã nằm trong thư mục ELAPSPLACEBO chưa."
    )


# ============================================================
# 3. LOAD DATA
# ============================================================

df_original = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

# Chuẩn hóa tên cột
df_original.columns = (
    df_original.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_cols = [
    ID_COL,
    TARGET_COL,
    START_DATE_COL,
    EVENT_DATE_COL
]

missing_cols = [
    col
    for col in required_cols
    if col not in df_original.columns
]

if missing_cols:
    raise ValueError(
        f"Thiếu các cột bắt buộc: {missing_cols}\n"
        f"Các cột hiện có: "
        f"{df_original.columns.tolist()}"
    )

print("\n===== ORIGINAL DATA =====")
print("Shape:", df_original.shape)
print("Columns:")
print(df_original.columns.tolist())


# ============================================================
# 4. STANDARDIZE DATA TYPES
# ============================================================

df = df_original.copy()

# Chuẩn hóa CUSTOMER_NUMBER
df[ID_COL] = (
    df[ID_COL]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Ngày bắt đầu quan hệ
df[START_DATE_COL] = pd.to_datetime(
    df[START_DATE_COL],
    errors="coerce"
)

# Ngày mở thẻ thật đối với adopter
# hoặc ngày cuối quan sát đối với non-adopter
df[EVENT_DATE_COL] = pd.to_datetime(
    df[EVENT_DATE_COL],
    errors="coerce"
)

# Target
df[TARGET_COL] = pd.to_numeric(
    df[TARGET_COL],
    errors="coerce"
)

invalid_target = ~df[TARGET_COL].isin([0, 1])

if invalid_target.any():

    print("\nCác TARGET không hợp lệ:")

    print(
        df.loc[
            invalid_target,
            [ID_COL, TARGET_COL]
        ].head(20)
    )

    raise ValueError(
        "TARGET phải chỉ gồm 0 và 1."
    )

df[TARGET_COL] = (
    df[TARGET_COL]
    .astype(int)
)


# ============================================================
# 5. CHECK DUPLICATE CUSTOMERS
# ============================================================

duplicate_count = (
    df[ID_COL]
    .duplicated()
    .sum()
)

if duplicate_count > 0:

    duplicate_examples = df.loc[
        df[ID_COL].duplicated(keep=False),
        [ID_COL]
    ].head(20)

    print("\nVí dụ khách hàng bị trùng:")
    print(duplicate_examples)

    raise ValueError(
        f"Phát hiện {duplicate_count} CUSTOMER_NUMBER bị trùng. "
        "File đầu vào phải có đúng một dòng cho mỗi khách hàng."
    )


# ============================================================
# 6. DETERMINE T_END
# ============================================================

# Với non-adopter, TARGET_MONTH được xem là ngày cuối cửa sổ quan sát
non_adopter_end_dates = (
    df.loc[
        df[TARGET_COL] == 0,
        EVENT_DATE_COL
    ]
    .dropna()
)

if non_adopter_end_dates.empty:
    raise ValueError(
        "Không tìm thấy TARGET_MONTH hợp lệ "
        "ở nhóm non-adopter."
    )

unique_end_dates = (
    non_adopter_end_dates
    .drop_duplicates()
    .sort_values()
)

T_END = (
    non_adopter_end_dates
    .max()
)

print("\n===== OBSERVATION END =====")
print("Unique non-adopter TARGET_MONTH values:")
print(unique_end_dates.tolist())
print("Selected T_END:", T_END.date())

if len(unique_end_dates) > 1:
    print(
        "CẢNH BÁO: Non-adopter có nhiều TARGET_MONTH khác nhau. "
        "Code đang dùng ngày lớn nhất làm T_END."
    )


# ============================================================
# 7. COMPUTE ACTUAL d* FOR ADOPTERS
# ============================================================

# d* = thời gian từ ngày bắt đầu quan hệ đến ngày mở thẻ
df["ACTUAL_D_STAR"] = (
    df[EVENT_DATE_COL]
    - df[START_DATE_COL]
).dt.days

# Số ngày tối đa có thể quan sát từ ngày bắt đầu đến T_END
df["AVAILABLE_DAYS"] = (
    T_END
    - df[START_DATE_COL]
).dt.days

invalid_start = (
    df[START_DATE_COL].isna()
    | df["AVAILABLE_DAYS"].isna()
    | (df["AVAILABLE_DAYS"] < 0)
)

print("\n===== BASIC DATA CHECK =====")
print("Total customers:", len(df))
print(
    "Adopters:",
    int((df[TARGET_COL] == 1).sum())
)
print(
    "Non-adopters:",
    int((df[TARGET_COL] == 0).sum())
)
print(
    "Missing/invalid relationship-start date:",
    int(invalid_start.sum())
)

# d* hợp lệ của adopter
valid_adopter_mask = (
    (df[TARGET_COL] == 1)
    & df["ACTUAL_D_STAR"].notna()
    & (df["ACTUAL_D_STAR"] >= 0)
    & (
        df["ACTUAL_D_STAR"]
        <= df["AVAILABLE_DAYS"]
    )
)

invalid_adopter_mask = (
    (df[TARGET_COL] == 1)
    & ~valid_adopter_mask
)

print(
    "Valid adopter d*:",
    int(valid_adopter_mask.sum())
)

print(
    "Invalid adopter d*:",
    int(invalid_adopter_mask.sum())
)

if invalid_adopter_mask.any():

    print("\nVí dụ adopter có d* không hợp lệ:")

    print(
        df.loc[
            invalid_adopter_mask,
            [
                ID_COL,
                START_DATE_COL,
                EVENT_DATE_COL,
                "ACTUAL_D_STAR",
                "AVAILABLE_DAYS"
            ]
        ].head(20)
    )

d_star_pool = (
    df.loc[
        valid_adopter_mask,
        "ACTUAL_D_STAR"
    ]
    .round()
    .astype(int)
    .sort_values()
    .to_numpy()
)

if len(d_star_pool) == 0:
    raise ValueError(
        "Không có d* hợp lệ trong nhóm adopter."
    )

print("\n===== ADOPTER d* DISTRIBUTION =====")
print(
    pd.Series(
        d_star_pool,
        name="Adopter d*"
    ).describe()
)

print("\nPercentiles:")

print(
    pd.Series(d_star_pool).quantile(
        [
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ============================================================
# 8. PLACEBO SAMPLING FUNCTION
# ============================================================

def create_placebo_dataset(
    source_df: pd.DataFrame,
    seed: int
) -> tuple[pd.DataFrame, pd.DataFrame]:

    """
    Tạo một bộ cutoff placebo.

    Adopter:
        - giữ ACTUAL_D_STAR;
        - cutoff là TARGET_MONTH thực.

    Non-adopter:
        - lấy ngẫu nhiên d* từ phân bố adopter;
        - chỉ lấy d* <= AVAILABLE_DAYS;
        - cutoff giả = CLIENT_CREATE_DATE + d*.

    Trả về:
        full_df:
            toàn bộ thuộc tính gốc + cột placebo.

        mapping_df:
            bảng cutoff gọn để nối với dữ liệu thô.
    """

    result = source_df.copy()

    rng = np.random.default_rng(seed)

    result["ASSIGNED_CUTOFF_DAYS"] = np.nan

    result["PLACEBO_CUTOFF_DATE"] = pd.NaT

    result["CUTOFF_TYPE"] = ""

    result["ELIGIBLE_DSTAR_COUNT"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64"
    )

    result["SAMPLING_STATUS"] = ""

    result["PLACEBO_SEED"] = seed


    # --------------------------------------------------------
    # 8.1 ADOPTERS: GIỮ CUTOFF THẬT
    # --------------------------------------------------------

    adopter_indices = result.index[
        result[TARGET_COL] == 1
    ]

    result.loc[
        adopter_indices,
        "ASSIGNED_CUTOFF_DAYS"
    ] = result.loc[
        adopter_indices,
        "ACTUAL_D_STAR"
    ]

    result.loc[
        adopter_indices,
        "PLACEBO_CUTOFF_DATE"
    ] = result.loc[
        adopter_indices,
        EVENT_DATE_COL
    ]

    result.loc[
        adopter_indices,
        "CUTOFF_TYPE"
    ] = "Actual acquisition cutoff"

    adopter_valid_flags = (
        valid_adopter_mask
        .reindex(adopter_indices)
        .fillna(False)
        .to_numpy()
    )

    result.loc[
        adopter_indices,
        "SAMPLING_STATUS"
    ] = np.where(
        adopter_valid_flags,
        "Valid actual cutoff",
        "Invalid actual cutoff"
    )


    # --------------------------------------------------------
    # 8.2 NON-ADOPTERS: GÁN PLACEBO d*
    # --------------------------------------------------------

    non_adopter_indices = result.index[
        result[TARGET_COL] == 0
    ]

    search_side = (
        "right"
        if ALLOW_CUTOFF_AT_T_END
        else "left"
    )

    for idx in non_adopter_indices:

        available_days = result.at[
            idx,
            "AVAILABLE_DAYS"
        ]

        if pd.isna(available_days):

            result.at[
                idx,
                "SAMPLING_STATUS"
            ] = "Missing relationship-start date"

            result.at[
                idx,
                "CUTOFF_TYPE"
            ] = "Placebo cutoff unavailable"

            continue

        available_days = int(
            available_days
        )

        eligible_count = np.searchsorted(
            d_star_pool,
            available_days,
            side=search_side
        )

        result.at[
            idx,
            "ELIGIBLE_DSTAR_COUNT"
        ] = eligible_count

        if eligible_count == 0:

            result.at[
                idx,
                "SAMPLING_STATUS"
            ] = "No eligible adopter d*"

            result.at[
                idx,
                "CUTOFF_TYPE"
            ] = "Placebo cutoff unavailable"

            continue

        sampled_position = rng.integers(
            low=0,
            high=eligible_count
        )

        sampled_d_star = int(
            d_star_pool[
                sampled_position
            ]
        )

        placebo_cutoff = (
            result.at[
                idx,
                START_DATE_COL
            ]
            + pd.Timedelta(
                days=sampled_d_star
            )
        )

        result.at[
            idx,
            "ASSIGNED_CUTOFF_DAYS"
        ] = sampled_d_star

        result.at[
            idx,
            "PLACEBO_CUTOFF_DATE"
        ] = placebo_cutoff

        result.at[
            idx,
            "CUTOFF_TYPE"
        ] = "Placebo cutoff"

        result.at[
            idx,
            "SAMPLING_STATUS"
        ] = "Sampled from eligible adopter d*"


    # --------------------------------------------------------
    # 8.3 CONVERT DATA TYPES
    # --------------------------------------------------------

    result["ASSIGNED_CUTOFF_DAYS"] = (
        pd.to_numeric(
            result["ASSIGNED_CUTOFF_DAYS"],
            errors="coerce"
        )
        .round()
        .astype("Int64")
    )

    result["AVAILABLE_DAYS"] = (
        pd.to_numeric(
            result["AVAILABLE_DAYS"],
            errors="coerce"
        )
        .round()
        .astype("Int64")
    )

    result["ACTUAL_D_STAR"] = (
        pd.to_numeric(
            result["ACTUAL_D_STAR"],
            errors="coerce"
        )
        .round()
        .astype("Int64")
    )


    # --------------------------------------------------------
    # 8.4 VALIDATION
    # --------------------------------------------------------

    cutoff_after_end = (
        result["PLACEBO_CUTOFF_DATE"]
        > T_END
    )

    cutoff_before_start = (
        result["PLACEBO_CUTOFF_DATE"]
        < result[START_DATE_COL]
    )

    invalid_non_adopter = (
        (result[TARGET_COL] == 0)
        & result[
            "ASSIGNED_CUTOFF_DAYS"
        ].notna()
        & (
            result[
                "ASSIGNED_CUTOFF_DAYS"
            ]
            > result[
                "AVAILABLE_DAYS"
            ]
        )
    )

    if cutoff_after_end.any():
        raise ValueError(
            f"Seed {seed}: phát hiện cutoff sau T_END."
        )

    if cutoff_before_start.any():
        raise ValueError(
            f"Seed {seed}: phát hiện cutoff "
            "trước ngày bắt đầu quan hệ."
        )

    if invalid_non_adopter.any():
        raise ValueError(
            f"Seed {seed}: non-adopter có placebo d* "
            "lớn hơn AVAILABLE_DAYS."
        )


    # --------------------------------------------------------
    # 8.5 MAPPING TABLE
    # --------------------------------------------------------

    mapping_columns = [
        ID_COL,
        TARGET_COL,
        START_DATE_COL,
        EVENT_DATE_COL,
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "ASSIGNED_CUTOFF_DAYS",
        "PLACEBO_CUTOFF_DATE",
        "CUTOFF_TYPE",
        "ELIGIBLE_DSTAR_COUNT",
        "SAMPLING_STATUS",
        "PLACEBO_SEED"
    ]

    mapping_df = (
        result[
            mapping_columns
        ]
        .copy()
    )

    return result, mapping_df


# ============================================================
# 9. RUN FIVE SEEDS
# ============================================================

all_mapping_tables = []

seed_summary_rows = []

distribution_rows = []

for seed in SEEDS:

    print("\n" + "=" * 60)
    print(f"RUNNING PLACEBO SEED {seed}")
    print("=" * 60)

    full_seed_df, mapping_seed_df = (
        create_placebo_dataset(
            source_df=df,
            seed=seed
        )
    )


    # --------------------------------------------------------
    # 9.1 SAVE FULL FILE
    # --------------------------------------------------------

    full_output_file = (
        OUTPUT_DIR
        / (
            "step2_customer_target_filtered_"
            f"placebo_seed_{seed}.csv"
        )
    )

    full_seed_df.to_csv(
        full_output_file,
        index=False,
        encoding="utf-8-sig"
    )


    # --------------------------------------------------------
    # 9.2 SAVE MAPPING FILE
    # --------------------------------------------------------

    mapping_output_file = (
        OUTPUT_DIR
        / f"placebo_table_seed_{seed}.csv"
    )

    mapping_seed_df.to_csv(
        mapping_output_file,
        index=False,
        encoding="utf-8-sig"
    )

    all_mapping_tables.append(
        mapping_seed_df
    )


    # --------------------------------------------------------
    # 9.3 SAMPLING STATUS
    # --------------------------------------------------------

    non_adopter_status = (
        mapping_seed_df.loc[
            mapping_seed_df[
                TARGET_COL
            ] == 0,
            "SAMPLING_STATUS"
        ]
        .value_counts(
            dropna=False
        )
    )

    print("\nNon-adopter sampling status:")
    print(non_adopter_status)


    valid_non = mapping_seed_df[
        (
            mapping_seed_df[
                TARGET_COL
            ] == 0
        )
        & (
            mapping_seed_df[
                "SAMPLING_STATUS"
            ]
            == "Sampled from eligible adopter d*"
        )
    ]


    valid_adopter = mapping_seed_df[
        (
            mapping_seed_df[
                TARGET_COL
            ] == 1
        )
        & (
            mapping_seed_df[
                "SAMPLING_STATUS"
            ]
            == "Valid actual cutoff"
        )
    ]


    seed_summary_rows.append({

        "Seed": seed,

        "Total customers": len(
            mapping_seed_df
        ),

        "Valid adopters": len(
            valid_adopter
        ),

        "Valid non-adopters": len(
            valid_non
        ),

        "Non-adopters without eligible d*": int(
            (
                mapping_seed_df[
                    "SAMPLING_STATUS"
                ]
                == "No eligible adopter d*"
            ).sum()
        ),

        "Missing start date": int(
            (
                mapping_seed_df[
                    "SAMPLING_STATUS"
                ]
                == "Missing relationship-start date"
            ).sum()
        ),

        "Adopter mean d*": valid_adopter[
            "ASSIGNED_CUTOFF_DAYS"
        ].mean(),

        "Adopter median d*": valid_adopter[
            "ASSIGNED_CUTOFF_DAYS"
        ].median(),

        "Placebo mean d*": valid_non[
            "ASSIGNED_CUTOFF_DAYS"
        ].mean(),

        "Placebo median d*": valid_non[
            "ASSIGNED_CUTOFF_DAYS"
        ].median()
    })


    # --------------------------------------------------------
    # 9.4 DISTRIBUTION SUMMARY
    # --------------------------------------------------------

    for group_name, group_df in [

        (
            "Adopter actual",
            valid_adopter
        ),

        (
            "Non-adopter placebo",
            valid_non
        )
    ]:

        series = (
            group_df[
                "ASSIGNED_CUTOFF_DAYS"
            ]
            .dropna()
            .astype(float)
        )

        distribution_rows.append({

            "Seed": seed,

            "Group": group_name,

            "n": len(series),

            "Mean": series.mean(),

            "Std": series.std(),

            "Min": series.min(),

            "Q05": series.quantile(
                0.05
            ),

            "Q25": series.quantile(
                0.25
            ),

            "Median": series.median(),

            "Q75": series.quantile(
                0.75
            ),

            "Q95": series.quantile(
                0.95
            ),

            "Max": series.max()
        })


    print("Full file:", full_output_file)
    print("Mapping file:", mapping_output_file)


# ============================================================
# 10. COMBINE RESULTS
# ============================================================

all_mapping_df = pd.concat(
    all_mapping_tables,
    ignore_index=True
)

seed_summary_df = pd.DataFrame(
    seed_summary_rows
)

distribution_summary_df = pd.DataFrame(
    distribution_rows
)

all_mapping_df.to_csv(
    OUTPUT_DIR
    / "all_placebo_tables_5_seeds.csv",
    index=False,
    encoding="utf-8-sig"
)

seed_summary_df.to_csv(
    OUTPUT_DIR
    / "placebo_seed_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

distribution_summary_df.to_csv(
    OUTPUT_DIR
    / "placebo_distribution_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. EXPORT EXCEL SUMMARY
# ============================================================

excel_output = (
    OUTPUT_DIR
    / "placebo_cutoff_summary_5_seeds.xlsx"
)

with pd.ExcelWriter(
    excel_output,
    engine="openpyxl"
) as writer:

    seed_summary_df.to_excel(
        writer,
        sheet_name="Seed summary",
        index=False
    )

    distribution_summary_df.to_excel(
        writer,
        sheet_name="Distribution summary",
        index=False
    )

    pd.DataFrame({
        "Adopter d*": pd.Series(
            d_star_pool
        )
    }).to_excel(
        writer,
        sheet_name="Adopter d-star pool",
        index=False
    )

    for seed, mapping_df in zip(
        SEEDS,
        all_mapping_tables
    ):

        mapping_df.to_excel(
            writer,
            sheet_name=f"Seed {seed}",
            index=False
        )


# ============================================================
# 12. CREATE DISTRIBUTION PLOTS
# ============================================================

for seed, mapping_df in zip(
    SEEDS,
    all_mapping_tables
):

    adopter_values = mapping_df.loc[
        (
            mapping_df[
                TARGET_COL
            ] == 1
        )
        & (
            mapping_df[
                "SAMPLING_STATUS"
            ]
            == "Valid actual cutoff"
        ),
        "ASSIGNED_CUTOFF_DAYS"
    ].dropna()


    placebo_values = mapping_df.loc[
        (
            mapping_df[
                TARGET_COL
            ] == 0
        )
        & (
            mapping_df[
                "SAMPLING_STATUS"
            ]
            == "Sampled from eligible adopter d*"
        ),
        "ASSIGNED_CUTOFF_DAYS"
    ].dropna()


    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        adopter_values,
        bins=50,
        density=True,
        alpha=0.6,
        label="Adopter actual d*"
    )

    plt.hist(
        placebo_values,
        bins=50,
        density=True,
        alpha=0.6,
        label="Non-adopter placebo d*"
    )

    plt.xlabel(
        "Admissible-window length (days)"
    )

    plt.ylabel(
        "Density"
    )

    plt.title(
        "Actual and placebo cutoff distributions "
        f"— Seed {seed}"
    )

    plt.legend()

    plt.tight_layout()

    plot_file = (
        OUTPUT_DIR
        / f"placebo_distribution_seed_{seed}.png"
    )

    plt.savefig(
        plot_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 13. CREATE FILTERED CUSTOMER LISTS
#     FOR RAW DATA PROCESSING
# ============================================================

for seed, mapping_df in zip(
    SEEDS,
    all_mapping_tables
):

    valid_for_processing = mapping_df[
        (
            (
                mapping_df[
                    TARGET_COL
                ] == 1
            )
            & (
                mapping_df[
                    "SAMPLING_STATUS"
                ]
                == "Valid actual cutoff"
            )
        )
        |
        (
            (
                mapping_df[
                    TARGET_COL
                ] == 0
            )
            & (
                mapping_df[
                    "SAMPLING_STATUS"
                ]
                == "Sampled from eligible adopter d*"
            )
        )
    ].copy()


    processing_file = (
        OUTPUT_DIR
        / (
            "customer_cutoff_for_raw_processing_"
            f"seed_{seed}.csv"
        )
    )


    valid_for_processing[
        [
            ID_COL,
            TARGET_COL,
            START_DATE_COL,
            "ASSIGNED_CUTOFF_DAYS",
            "PLACEBO_CUTOFF_DATE",
            "CUTOFF_TYPE",
            "PLACEBO_SEED"
        ]
    ].to_csv(
        processing_file,
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 14. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("PLACEBO TABLE GENERATION COMPLETED")
print("=" * 70)

print("\n===== SEED SUMMARY =====")
print(seed_summary_df)

print("\nKết quả đã được lưu tại:")
print(OUTPUT_DIR)

print("\nCác file đã tạo:")

for file_path in sorted(
    OUTPUT_DIR.iterdir()
):
    print("-", file_path.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/ELAPSPLACEBO
Input file: /content/drive/MyDrive/ELAPSPLACEBO/step2_customer_target_filtered.csv
Output folder: /content/drive/MyDrive/ELAPSPLACEBO/placebo_cutoff_outputs

===== ORIGINAL DATA =====
Shape: (150408, 12)
Columns:
['CUSTOMER_NUMBER', 'CLIENT_SEX', 'CLIENT_CREATE_DATE', 'DATE_OF_BIRTH', 'STAFF_VIB', 'IB_REGISTER_DATE', 'EB_REGISTER_CHANNEL', 'SMS', 'VERIFY_METHOD', 'TARGET', 'TARGET_MONTH', 'TENURE_DAYS_AT_TARGET']

===== OBSERVATION END =====
Unique non-adopter TARGET_MONTH values:
[Timestamp('2019-12-31 00:00:00')]
Selected T_END: 2019-12-31

===== BASIC DATA CHECK =====
Total customers: 150408
Adopters: 42323
Non-adopters: 108085
Missing/invalid relationship-start date: 0
Valid adopter d*: 42323
Invalid adopter d*: 0

===== ADOPTER d* DISTRIBUTION =====
count    42323.000000
mean        37.259741
std       

6️⃣ Nối TRANSACTION

In [ ]:
import os
from pathlib import Path

import pandas as pd
from google.colab import drive


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

PLACEBO_DIR = os.path.join(
    BASE_DIR,
    "placebo_cutoff_outputs"
)

TRANSACTION_FILE = os.path.join(
    BASE_DIR,
    "2.Data_MyVIB_Transaction.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "transaction_placebo_outputs"
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"
CUTOFF_COL = "PLACEBO_CUTOFF_DATE"
TRANS_DATE_COL = "TRANS_DATE"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. CHECK BASE FILES
# ============================================================

if not Path(BASE_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục:\n{BASE_DIR}"
    )

if not Path(TRANSACTION_FILE).exists():
    raise FileNotFoundError(
        f"Không tìm thấy file giao dịch:\n{TRANSACTION_FILE}"
    )

print("BASE_DIR:", BASE_DIR)
print("PLACEBO_DIR:", PLACEBO_DIR)
print("TRANSACTION_FILE:", TRANSACTION_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)


# ============================================================
# 4. LOAD TRANSACTION DATA ONCE
# ============================================================

trans = pd.read_csv(
    TRANSACTION_FILE,
    low_memory=False
)

trans.columns = (
    trans.columns
    .str.strip()
    .str.upper()
)

required_trans_cols = [
    ID_COL,
    TRANS_DATE_COL
]

missing_trans_cols = [
    col for col in required_trans_cols
    if col not in trans.columns
]

if missing_trans_cols:
    raise ValueError(
        f"File transaction thiếu cột: {missing_trans_cols}\n"
        f"Các cột hiện có: {trans.columns.tolist()}"
    )

# Chuẩn hóa CUSTOMER_NUMBER
trans[ID_COL] = (
    trans[ID_COL]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
)

# Chuẩn hóa ngày giao dịch
trans[TRANS_DATE_COL] = pd.to_datetime(
    trans[TRANS_DATE_COL],
    errors="coerce"
)

print("\n===== TRANSACTION DATA =====")
print("Rows:", len(trans))
print("Customers:", trans[ID_COL].nunique())
print(
    "Missing TRANS_DATE:",
    trans[TRANS_DATE_COL].isna().sum()
)


# ============================================================
# 5. PROCESS ONE SEED
# ============================================================

def process_transaction_seed(seed: int) -> dict:

    print("\n" + "=" * 70)
    print(f"PROCESSING TRANSACTION — SEED {seed}")
    print("=" * 70)

    # --------------------------------------------------------
    # 5.1 Input master placebo file
    # --------------------------------------------------------

    master_file = os.path.join(
        PLACEBO_DIR,
        f"step2_customer_target_filtered_placebo_seed_{seed}.csv"
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"step3_transaction_before_placebo_cutoff_seed_{seed}.csv"
    )

    summary_file = os.path.join(
        OUTPUT_DIR,
        f"step3_transaction_summary_seed_{seed}.csv"
    )

    customer_check_file = os.path.join(
        OUTPUT_DIR,
        f"step3_transaction_customer_check_seed_{seed}.csv"
    )

    if not Path(master_file).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file master placebo seed {seed}:\n"
            f"{master_file}"
        )

    # --------------------------------------------------------
    # 5.2 Load master placebo file
    # --------------------------------------------------------

    master = pd.read_csv(
        master_file,
        low_memory=False
    )

    master.columns = (
        master.columns
        .str.strip()
        .str.upper()
    )

    required_master_cols = [
        ID_COL,
        TARGET_COL,
        "CLIENT_CREATE_DATE",
        CUTOFF_COL
    ]

    missing_master_cols = [
        col for col in required_master_cols
        if col not in master.columns
    ]

    if missing_master_cols:
        raise ValueError(
            f"Seed {seed}: file master thiếu cột "
            f"{missing_master_cols}\n"
            f"Các cột hiện có: {master.columns.tolist()}"
        )

    # --------------------------------------------------------
    # 5.3 Standardize master data types
    # --------------------------------------------------------

    master[ID_COL] = (
        master[ID_COL]
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    master[CUTOFF_COL] = pd.to_datetime(
        master[CUTOFF_COL],
        errors="coerce"
    )

    master["CLIENT_CREATE_DATE"] = pd.to_datetime(
        master["CLIENT_CREATE_DATE"],
        errors="coerce"
    )

    master[TARGET_COL] = pd.to_numeric(
        master[TARGET_COL],
        errors="coerce"
    )

    optional_numeric_cols = [
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "PLACEBO_SEED"
    ]

    for col in optional_numeric_cols:
        if col in master.columns:
            master[col] = pd.to_numeric(
                master[col],
                errors="coerce"
            )

    # --------------------------------------------------------
    # 5.4 Check duplicate customers
    # --------------------------------------------------------

    duplicate_count = (
        master[ID_COL]
        .duplicated()
        .sum()
    )

    print(
        "Duplicate customers in master:",
        duplicate_count
    )

    if duplicate_count > 0:

        duplicate_file = os.path.join(
            OUTPUT_DIR,
            f"duplicate_master_customers_seed_{seed}.csv"
        )

        master.loc[
            master[ID_COL].duplicated(
                keep=False
            )
        ].to_csv(
            duplicate_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: file master có "
            "CUSTOMER_NUMBER bị trùng."
        )

    # --------------------------------------------------------
    # 5.5 Check invalid cutoff
    # --------------------------------------------------------

    invalid_cutoff = master[
        master[CUTOFF_COL].isna()
        | master["CLIENT_CREATE_DATE"].isna()
        | (
            master[CUTOFF_COL]
            < master["CLIENT_CREATE_DATE"]
        )
    ]

    print(
        "Customers with invalid cutoff:",
        len(invalid_cutoff)
    )

    if not invalid_cutoff.empty:

        invalid_file = os.path.join(
            OUTPUT_DIR,
            f"invalid_cutoff_seed_{seed}.csv"
        )

        invalid_cutoff.to_csv(
            invalid_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: có khách hàng thiếu "
            "hoặc sai PLACEBO_CUTOFF_DATE."
        )

    # --------------------------------------------------------
    # 5.6 Avoid duplicated columns after merge
    #
    # Nếu master đã có cột trùng tên với transaction,
    # chỉ giữ CUSTOMER_NUMBER từ master và cột giao dịch từ trans.
    # --------------------------------------------------------

    overlapping_cols = [
        col for col in trans.columns
        if col in master.columns
        and col != ID_COL
    ]

    if overlapping_cols:
        print(
            "Các cột giao dịch đã tồn tại trong master "
            "và sẽ bị loại khỏi master trước merge:",
            overlapping_cols
        )

        master_for_merge = master.drop(
            columns=overlapping_cols
        ).copy()
    else:
        master_for_merge = master.copy()

    # --------------------------------------------------------
    # 5.7 LEFT JOIN
    # Giữ toàn bộ khách hàng trong master
    # --------------------------------------------------------

    df = master_for_merge.merge(
        trans,
        on=ID_COL,
        how="left",
        validate="one_to_many"
    )

    expected_customers = (
        master_for_merge[ID_COL]
        .nunique()
    )

    print("\n===== BEFORE FILTERING =====")
    print("Rows:", len(df))
    print(
        "Customers:",
        df[ID_COL].nunique()
    )
    print(
        "Expected customers:",
        expected_customers
    )
    print(
        "Adopters:",
        df.loc[
            df[TARGET_COL] == 1,
            ID_COL
        ].nunique()
    )
    print(
        "Non-adopters:",
        df.loc[
            df[TARGET_COL] == 0,
            ID_COL
        ].nunique()
    )

    # --------------------------------------------------------
    # 5.8 Filter transaction records
    #
    # Giữ:
    # - khách không có giao dịch;
    # - giao dịch trước PLACEBO_CUTOFF_DATE.
    # --------------------------------------------------------

    valid_transaction_mask = (
        df[TRANS_DATE_COL].isna()
        |
        (
            df[CUTOFF_COL].notna()
            & (
                df[TRANS_DATE_COL]
                < df[CUTOFF_COL]
            )
        )
    )

    df_filtered = df.loc[
        valid_transaction_mask
    ].copy()

    # --------------------------------------------------------
    # 5.9 Find customers lost after filtering
    #
    # Khách có giao dịch nhưng tất cả giao dịch đều sau cutoff
    # sẽ biến mất hoàn toàn.
    # --------------------------------------------------------

    customers_before = set(
        master_for_merge[
            ID_COL
        ].unique()
    )

    customers_after_raw_filter = set(
        df_filtered[
            ID_COL
        ].unique()
    )

    lost_customers = (
        customers_before
        - customers_after_raw_filter
    )

    print("\n===== AFTER RAW FILTERING =====")
    print("Rows:", len(df_filtered))
    print(
        "Customers after raw filter:",
        df_filtered[ID_COL].nunique()
    )
    print(
        "Customers temporarily lost:",
        len(lost_customers)
    )

    # --------------------------------------------------------
    # 5.10 Restore lost customers
    #
    # Thêm lại một dòng customer-level với các cột
    # transaction để trống.
    # --------------------------------------------------------

    if lost_customers:

        missing_rows = master_for_merge[
            master_for_merge[ID_COL].isin(
                lost_customers
            )
        ].copy()

        transaction_only_cols = [
            col for col in trans.columns
            if col != ID_COL
        ]

        for col in transaction_only_cols:
            if col not in missing_rows.columns:
                missing_rows[col] = pd.NA

        missing_rows = missing_rows[
            df_filtered.columns
        ]

        df_filtered = pd.concat(
            [
                df_filtered,
                missing_rows
            ],
            ignore_index=True
        )

    # --------------------------------------------------------
    # 5.11 Final customer consistency check
    # --------------------------------------------------------

    final_customers = (
        df_filtered[ID_COL]
        .nunique()
    )

    print(
        "Customers after restoration:",
        final_customers
    )

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách cuối không khớp. "
            f"Expected={expected_customers}, "
            f"Final={final_customers}"
        )

    # --------------------------------------------------------
    # 5.12 Leakage check
    # --------------------------------------------------------

    leak = df_filtered[
        df_filtered[TRANS_DATE_COL].notna()
        & df_filtered[CUTOFF_COL].notna()
        & (
            df_filtered[TRANS_DATE_COL]
            >= df_filtered[CUTOFF_COL]
        )
    ]

    print(
        "Remaining leakage rows:",
        len(leak)
    )

    if not leak.empty:

        leak_file = os.path.join(
            OUTPUT_DIR,
            f"transaction_leakage_seed_{seed}.csv"
        )

        leak.to_csv(
            leak_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: vẫn còn giao dịch "
            "tại hoặc sau PLACEBO_CUTOFF_DATE."
        )

    # --------------------------------------------------------
    # 5.13 Customers with no valid pre-cutoff transaction
    # --------------------------------------------------------

    no_transaction_flag = (
        df_filtered
        .groupby(ID_COL)[TRANS_DATE_COL]
        .apply(
            lambda x: x.isna().all()
        )
        .rename(
            "NO_PRE_CUTOFF_TRANSACTION"
        )
        .reset_index()
    )

    no_transaction_count = int(
        no_transaction_flag[
            "NO_PRE_CUTOFF_TRANSACTION"
        ].sum()
    )

    print(
        "Customers with no pre-cutoff transaction:",
        no_transaction_count
    )

    # --------------------------------------------------------
    # 5.14 Count valid transaction rows per customer
    # --------------------------------------------------------

    transaction_count = (
        df_filtered
        .groupby(ID_COL)[TRANS_DATE_COL]
        .count()
        .rename(
            "PRE_CUTOFF_TRANSACTION_COUNT"
        )
        .reset_index()
    )

    customer_check_cols = [
        ID_COL,
        TARGET_COL,
        "CLIENT_CREATE_DATE",
        CUTOFF_COL
    ]

    for optional_col in [
        "CUTOFF_TYPE",
        "PLACEBO_SEED",
        "ASSIGNED_CUTOFF_DAYS"
    ]:
        if optional_col in master_for_merge.columns:
            customer_check_cols.append(
                optional_col
            )

    customer_check = (
        master_for_merge[
            customer_check_cols
        ]
        .merge(
            no_transaction_flag,
            on=ID_COL,
            how="left",
            validate="one_to_one"
        )
        .merge(
            transaction_count,
            on=ID_COL,
            how="left",
            validate="one_to_one"
        )
    )

    customer_check[
        "PRE_CUTOFF_TRANSACTION_COUNT"
    ] = (
        customer_check[
            "PRE_CUTOFF_TRANSACTION_COUNT"
        ]
        .fillna(0)
        .astype(int)
    )

    # --------------------------------------------------------
    # 5.15 Summary by target
    # --------------------------------------------------------

    aggregation_dict = {
        "CUSTOMERS": (
            ID_COL,
            "nunique"
        ),
        "TRANSACTION_ROWS": (
            TRANS_DATE_COL,
            "count"
        ),
        "EARLIEST_TRANSACTION": (
            TRANS_DATE_COL,
            "min"
        ),
        "LATEST_TRANSACTION": (
            TRANS_DATE_COL,
            "max"
        )
    }

    if "ASSIGNED_CUTOFF_DAYS" in df_filtered.columns:
        aggregation_dict[
            "MEAN_CUTOFF_DAYS"
        ] = (
            "ASSIGNED_CUTOFF_DAYS",
            "mean"
        )

        aggregation_dict[
            "MEDIAN_CUTOFF_DAYS"
        ] = (
            "ASSIGNED_CUTOFF_DAYS",
            "median"
        )

    summary = (
        df_filtered
        .groupby(TARGET_COL)
        .agg(
            **aggregation_dict
        )
        .reset_index()
    )

    summary["SEED"] = seed

    # --------------------------------------------------------
    # 5.16 Sort
    # --------------------------------------------------------

    df_filtered = (
        df_filtered
        .sort_values(
            by=[
                ID_COL,
                TRANS_DATE_COL
            ],
            na_position="last"
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 5.17 Save files
    # --------------------------------------------------------

    df_filtered.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    customer_check.to_csv(
        customer_check_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== SAVED =====")
    print(output_file)
    print(summary_file)
    print(customer_check_file)

    return {
        "SEED": seed,
        "INPUT_CUSTOMERS": expected_customers,
        "FINAL_CUSTOMERS": final_customers,
        "OUTPUT_ROWS": len(df_filtered),
        "NO_TRANSACTION_CUSTOMERS": no_transaction_count,
        "LEAKAGE_ROWS": len(leak),
        "OUTPUT_FILE": output_file
    }


# ============================================================
# 6. RUN ALL FIVE SEEDS
# ============================================================

run_results = []

for seed in SEEDS:
    seed_result = process_transaction_seed(
        seed
    )

    run_results.append(
        seed_result
    )


# ============================================================
# 7. COMBINE SEED SUMMARIES
# ============================================================

run_summary = pd.DataFrame(
    run_results
)

run_summary_file = os.path.join(
    OUTPUT_DIR,
    "step3_transaction_all_seeds_summary.csv"
)

run_summary.to_csv(
    run_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("ALL FIVE SEEDS COMPLETED")
print("=" * 70)

print(run_summary)

print("\nSaved combined summary:")
print(run_summary_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/ELAPSPLACEBO
PLACEBO_DIR: /content/drive/MyDrive/ELAPSPLACEBO/placebo_cutoff_outputs
TRANSACTION_FILE: /content/drive/MyDrive/ELAPSPLACEBO/2.Data_MyVIB_Transaction.csv
OUTPUT_DIR: /content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs

===== TRANSACTION DATA =====
Rows: 1418030
Customers: 52488
Missing TRANS_DATE: 0

PROCESSING TRANSACTION — SEED 1
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 1404243
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 218974
Customers after raw filter: 122595
Customers temporarily lost: 27813


/tmp/ipykernel_54151/12545281.py:476: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff transaction: 132355

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_before_placebo_cutoff_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_summary_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_customer_check_seed_1.csv

PROCESSING TRANSACTION — SEED 2
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 1404243
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 219998
Customers after raw filter: 122664
Customers temporarily lost: 27744


/tmp/ipykernel_54151/12545281.py:476: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff transaction: 132286

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_before_placebo_cutoff_seed_2.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_summary_seed_2.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_customer_check_seed_2.csv

PROCESSING TRANSACTION — SEED 3
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 1404243
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 219188
Customers after raw filter: 122558
Customers temporarily lost: 27850


...

Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff transaction: 132392

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_before_placebo_cutoff_seed_3.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_summary_seed_3.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_customer_check_seed_3.csv

PROCESSING TRANSACTION — SEED 4
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 1404243
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 218117
Customers after raw filter: 122539
Customers temporarily lost: 27869


/tmp/ipykernel_54151/12545281.py:476: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff transaction: 132411

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_before_placebo_cutoff_seed_4.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_summary_seed_4.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_customer_check_seed_4.csv

PROCESSING TRANSACTION — SEED 5
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 1404243
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 220094
Customers after raw filter: 122546
Customers temporarily lost: 27862


/tmp/ipykernel_54151/12545281.py:476: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff transaction: 132404

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_before_placebo_cutoff_seed_5.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_summary_seed_5.csv
/content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs/step3_transaction_customer_check_seed_5.csv

ALL FIVE SEEDS COMPLETED
   SEED  INPUT_CUSTOMERS  FINAL_CUSTOMERS  OUTPUT_ROWS  \
0     1           150408           150408       246787   
1     2           150408           150408       247742   
2     3           150408           150408       247038   
3     4           150408           150408       245986   
4     5           150408           150408       247956   

   NO_TRANSACTION_CUSTOMERS  LEAKAGE_ROWS  \
0                    132355             0   
1                    132286             0   
2                    132392         

TẠO FEATURE CHO TRANSACTION

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

INPUT_DIR = os.path.join(
    BASE_DIR,
    "transaction_placebo_outputs"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "transaction_features"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"

DATE_COL = "TRANS_DATE"
AMOUNT_COL = "TRANS_AMOUNT"
LV1_COL = "TRANS_LV1"
LV2_COL = "TRANS_LV2"

FINAL_FEATURE_COLS = [
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
    "TRANS_LV1_MODE_CODE",
    "TRANS_LV2_MODE_CODE"
]


# ============================================================
# 3. CHECK INPUT DIRECTORY
# ============================================================

if not Path(INPUT_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục đầu vào:\n{INPUT_DIR}"
    )

print("INPUT_DIR :", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


# ============================================================
# 4. HELPER: STANDARDIZE CUSTOMER NUMBER
# ============================================================

def standardize_customer_id(
    series: pd.Series
) -> pd.Series:

    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )


# ============================================================
# 5. HELPER: CLEAN CATEGORY
# ============================================================

def clean_category(
    series: pd.Series
) -> pd.Series:

    result = (
        series
        .astype("string")
        .str.strip()
        .str.upper()
    )

    result = result.replace({
        "": pd.NA,
        "NAN": pd.NA,
        "<NA>": pd.NA,
        "NONE": pd.NA,
        "NULL": pd.NA
    })

    return result


# ============================================================
# 6. HELPER: MODE WITH DETERMINISTIC TIE BREAKING
# ============================================================

def get_mode_value(
    series: pd.Series
):

    valid = clean_category(
        series
    ).dropna()

    if valid.empty:
        return "NO_TRANSACTION"

    counts = valid.value_counts()

    max_count = counts.max()

    candidates = (
        counts[
            counts == max_count
        ]
        .index
        .astype(str)
        .tolist()
    )

    # Nếu có nhiều mode cùng tần suất,
    # chọn theo thứ tự chữ để kết quả tái lập
    return sorted(candidates)[0]


# ============================================================
# 7. CREATE RAW FEATURES FOR ONE SEED
# ============================================================

def create_raw_features(
    seed: int
) -> pd.DataFrame:

    print("\n" + "=" * 75)
    print(f"CREATING TRANSACTION FEATURES — SEED {seed}")
    print("=" * 75)

    input_file = os.path.join(
        INPUT_DIR,
        f"step3_transaction_before_placebo_cutoff_seed_{seed}.csv"
    )

    if not Path(input_file).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file seed {seed}:\n"
            f"{input_file}"
        )

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    df = pd.read_csv(
        input_file,
        low_memory=False
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )

    required_cols = [
        ID_COL,
        TARGET_COL,
        DATE_COL,
        AMOUNT_COL,
        LV1_COL,
        LV2_COL
    ]

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Seed {seed} thiếu các cột: {missing_cols}\n"
            f"Các cột hiện có:\n{df.columns.tolist()}"
        )

    # --------------------------------------------------------
    # Standardize
    # --------------------------------------------------------

    df[ID_COL] = standardize_customer_id(
        df[ID_COL]
    )

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce"
    )

    df[DATE_COL] = pd.to_datetime(
        df[DATE_COL],
        errors="coerce"
    )

    df[AMOUNT_COL] = (
        df[AMOUNT_COL]
        .astype("string")
        .str.replace(
            ",",
            "",
            regex=False
        )
        .str.strip()
    )

    df[AMOUNT_COL] = pd.to_numeric(
        df[AMOUNT_COL],
        errors="coerce"
    )

    df[LV1_COL] = clean_category(
        df[LV1_COL]
    )

    df[LV2_COL] = clean_category(
        df[LV2_COL]
    )

    df = df.dropna(
        subset=[
            ID_COL,
            TARGET_COL
        ]
    ).copy()

    df[TARGET_COL] = (
        df[TARGET_COL]
        .astype(int)
    )

    if not df[TARGET_COL].isin([0, 1]).all():
        raise ValueError(
            f"Seed {seed}: TARGET phải chỉ gồm 0 và 1."
        )

    # --------------------------------------------------------
    # Check target consistency
    # --------------------------------------------------------

    target_check = (
        df.groupby(ID_COL)[TARGET_COL]
        .nunique()
    )

    if (target_check > 1).any():
        raise ValueError(
            f"Seed {seed}: một khách hàng có nhiều TARGET."
        )

    # --------------------------------------------------------
    # Customer base
    #
    # Giữ đủ khách hàng, kể cả khách không có transaction
    # --------------------------------------------------------

    customer_base = (
        df[
            [
                ID_COL,
                TARGET_COL
            ]
        ]
        .drop_duplicates(
            subset=[ID_COL],
            keep="first"
        )
        .copy()
    )

    expected_customers = (
        customer_base[ID_COL]
        .nunique()
    )

    # --------------------------------------------------------
    # Only valid transaction rows
    #
    # Các dòng TRANS_DATE = NaT chỉ để giữ khách,
    # không dùng để tính feature
    # --------------------------------------------------------

    valid_transactions = df.loc[
        df[DATE_COL].notna()
    ].copy()

    print("Input rows:", len(df))
    print("Expected customers:", expected_customers)
    print(
        "Valid transaction rows:",
        len(valid_transactions)
    )
    print(
        "Customers with transaction:",
        valid_transactions[ID_COL].nunique()
    )

    # --------------------------------------------------------
    # Aggregate
    # --------------------------------------------------------

    if not valid_transactions.empty:

        aggregated = (
            valid_transactions
            .groupby(ID_COL)
            .agg(
                TRANS_LV1_MODE=(
                    LV1_COL,
                    get_mode_value
                ),
                TRANS_LV2_MODE=(
                    LV2_COL,
                    get_mode_value
                ),
                TRANS_AMOUNT_MAX=(
                    AMOUNT_COL,
                    "max"
                ),
                TRANS_AMOUNT_MEAN=(
                    AMOUNT_COL,
                    "mean"
                ),
                TRANSACTION_COUNT=(
                    DATE_COL,
                    "count"
                )
            )
            .reset_index()
        )

    else:

        aggregated = pd.DataFrame(
            columns=[
                ID_COL,
                "TRANS_LV1_MODE",
                "TRANS_LV2_MODE",
                "TRANS_AMOUNT_MAX",
                "TRANS_AMOUNT_MEAN",
                "TRANSACTION_COUNT"
            ]
        )

    # --------------------------------------------------------
    # Merge with all customers
    # --------------------------------------------------------

    feature_full = customer_base.merge(
        aggregated,
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )

    feature_full["TRANS_LV1_MODE"] = (
        feature_full["TRANS_LV1_MODE"]
        .fillna("NO_TRANSACTION")
    )

    feature_full["TRANS_LV2_MODE"] = (
        feature_full["TRANS_LV2_MODE"]
        .fillna("NO_TRANSACTION")
    )

    feature_full[
        [
            "TRANS_AMOUNT_MAX",
            "TRANS_AMOUNT_MEAN"
        ]
    ] = (
        feature_full[
            [
                "TRANS_AMOUNT_MAX",
                "TRANS_AMOUNT_MEAN"
            ]
        ]
        .fillna(0)
    )

    feature_full["TRANSACTION_COUNT"] = (
        feature_full["TRANSACTION_COUNT"]
        .fillna(0)
        .astype(int)
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    final_customers = (
        feature_full[ID_COL]
        .nunique()
    )

    duplicates = int(
        feature_full[ID_COL]
        .duplicated()
        .sum()
    )

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách cuối không khớp."
        )

    if duplicates > 0:
        raise ValueError(
            f"Seed {seed}: còn CUSTOMER_NUMBER bị trùng."
        )

    return feature_full


# ============================================================
# 8. CREATE RAW FEATURES FOR ALL FIVE SEEDS
# ============================================================

raw_features = {}

all_lv1_categories = {
    "NO_TRANSACTION"
}

all_lv2_categories = {
    "NO_TRANSACTION"
}

for seed in SEEDS:

    feature_seed = create_raw_features(
        seed
    )

    raw_features[seed] = feature_seed

    all_lv1_categories.update(
        feature_seed[
            "TRANS_LV1_MODE"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    all_lv2_categories.update(
        feature_seed[
            "TRANS_LV2_MODE"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )


# ============================================================
# 9. CREATE ONE GLOBAL MAPPING FOR ALL FIVE SEEDS
#
# Giúp cùng một category nhận cùng một code ở mọi seed
# ============================================================

lv1_categories = [
    value
    for value in sorted(
        all_lv1_categories
    )
    if value != "NO_TRANSACTION"
]

lv2_categories = [
    value
    for value in sorted(
        all_lv2_categories
    )
    if value != "NO_TRANSACTION"
]

LV1_MAPPING = {
    "NO_TRANSACTION": 0
}

LV1_MAPPING.update({
    category: code
    for code, category in enumerate(
        lv1_categories,
        start=1
    )
})

LV2_MAPPING = {
    "NO_TRANSACTION": 0
}

LV2_MAPPING.update({
    category: code
    for code, category in enumerate(
        lv2_categories,
        start=1
    )
})

print("\n===== GLOBAL CATEGORY MAPPING =====")
print(
    "Number of LV1 categories:",
    len(LV1_MAPPING)
)
print(
    "Number of LV2 categories:",
    len(LV2_MAPPING)
)


# ============================================================
# 10. SAVE MAPPING TABLES
# ============================================================

lv1_mapping_df = pd.DataFrame({
    "TRANS_LV1_MODE": list(
        LV1_MAPPING.keys()
    ),
    "TRANS_LV1_MODE_CODE": list(
        LV1_MAPPING.values()
    )
})

lv2_mapping_df = pd.DataFrame({
    "TRANS_LV2_MODE": list(
        LV2_MAPPING.keys()
    ),
    "TRANS_LV2_MODE_CODE": list(
        LV2_MAPPING.values()
    )
})

lv1_mapping_file = os.path.join(
    OUTPUT_DIR,
    "transaction_lv1_mapping_all_seeds.csv"
)

lv2_mapping_file = os.path.join(
    OUTPUT_DIR,
    "transaction_lv2_mapping_all_seeds.csv"
)

lv1_mapping_df.to_csv(
    lv1_mapping_file,
    index=False,
    encoding="utf-8-sig"
)

lv2_mapping_df.to_csv(
    lv2_mapping_file,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. APPLY MAPPING AND SAVE EACH SEED
# ============================================================

summary_rows = []

for seed in SEEDS:

    feature_full = raw_features[
        seed
    ].copy()

    # --------------------------------------------------------
    # Encode MODE
    # --------------------------------------------------------

    feature_full[
        "TRANS_LV1_MODE_CODE"
    ] = (
        feature_full[
            "TRANS_LV1_MODE"
        ]
        .map(
            LV1_MAPPING
        )
    )

    feature_full[
        "TRANS_LV2_MODE_CODE"
    ] = (
        feature_full[
            "TRANS_LV2_MODE"
        ]
        .map(
            LV2_MAPPING
        )
    )

    if feature_full[
        "TRANS_LV1_MODE_CODE"
    ].isna().any():
        raise ValueError(
            f"Seed {seed}: còn LV1 chưa được mã hóa."
        )

    if feature_full[
        "TRANS_LV2_MODE_CODE"
    ].isna().any():
        raise ValueError(
            f"Seed {seed}: còn LV2 chưa được mã hóa."
        )

    feature_full[
        [
            "TRANS_LV1_MODE_CODE",
            "TRANS_LV2_MODE_CODE"
        ]
    ] = (
        feature_full[
            [
                "TRANS_LV1_MODE_CODE",
                "TRANS_LV2_MODE_CODE"
            ]
        ]
        .astype(int)
    )

    # --------------------------------------------------------
    # Final transaction feature table
    #
    # Không giữ TARGET vì TARGET đã có ở bảng customer
    # --------------------------------------------------------

    feature_final = feature_full[
        [
            ID_COL,
            "TRANS_LV1_MODE",
            "TRANS_LV2_MODE",
            "TRANS_AMOUNT_MAX",
            "TRANS_AMOUNT_MEAN",
            "TRANS_LV1_MODE_CODE",
            "TRANS_LV2_MODE_CODE"
        ]
    ].copy()

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    expected_customers = (
        feature_full[ID_COL]
        .nunique()
    )

    final_customers = (
        feature_final[ID_COL]
        .nunique()
    )

    duplicate_count = int(
        feature_final[ID_COL]
        .duplicated()
        .sum()
    )

    missing_features = int(
        feature_final[
            FINAL_FEATURE_COLS
        ]
        .isna()
        .sum()
        .sum()
    )

    print("\n" + "-" * 70)
    print(f"VALIDATION — SEED {seed}")
    print("-" * 70)
    print("Expected customers :", expected_customers)
    print("Final customers    :", final_customers)
    print("Duplicate customers:", duplicate_count)
    print("Missing features   :", missing_features)
    print("Columns:")
    print(feature_final.columns.tolist())

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách không khớp."
        )

    if duplicate_count > 0:
        raise ValueError(
            f"Seed {seed}: còn khách bị trùng."
        )

    if missing_features > 0:
        raise ValueError(
            f"Seed {seed}: còn feature bị thiếu."
        )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_file = os.path.join(
        OUTPUT_DIR,
        f"feature_transaction_mode_placebo_seed_{seed}.csv"
    )

    audit_file = os.path.join(
        OUTPUT_DIR,
        f"feature_transaction_audit_placebo_seed_{seed}.csv"
    )

    summary_file = os.path.join(
        OUTPUT_DIR,
        f"feature_transaction_summary_placebo_seed_{seed}.csv"
    )

    feature_final = (
        feature_final
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    feature_full = (
        feature_full
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    feature_final.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    feature_full.to_csv(
        audit_file,
        index=False,
        encoding="utf-8-sig"
    )

    summary = (
        feature_full
        .groupby(TARGET_COL)
        .agg(
            CUSTOMERS=(
                ID_COL,
                "nunique"
            ),
            CUSTOMERS_WITH_TRANSACTION=(
                "TRANSACTION_COUNT",
                lambda x: int(
                    (x > 0).sum()
                )
            ),
            MEAN_TRANSACTION_COUNT=(
                "TRANSACTION_COUNT",
                "mean"
            ),
            MEAN_TRANS_AMOUNT_MAX=(
                "TRANS_AMOUNT_MAX",
                "mean"
            ),
            MEAN_TRANS_AMOUNT_MEAN=(
                "TRANS_AMOUNT_MEAN",
                "mean"
            )
        )
        .reset_index()
    )

    summary[
        "CUSTOMERS_WITHOUT_TRANSACTION"
    ] = (
        summary["CUSTOMERS"]
        - summary[
            "CUSTOMERS_WITH_TRANSACTION"
        ]
    )

    summary["SEED"] = seed

    summary.to_csv(
        summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    summary_rows.append({
        "SEED": seed,
        "CUSTOMERS": final_customers,
        "WITH_TRANSACTION": int(
            (
                feature_full[
                    "TRANSACTION_COUNT"
                ] > 0
            ).sum()
        ),
        "WITHOUT_TRANSACTION": int(
            (
                feature_full[
                    "TRANSACTION_COUNT"
                ] == 0
            ).sum()
        ),
        "LV1_CATEGORIES": len(
            LV1_MAPPING
        ),
        "LV2_CATEGORIES": len(
            LV2_MAPPING
        ),
        "DUPLICATES": duplicate_count,
        "MISSING_FEATURES": (
            missing_features
        ),
        "OUTPUT_FILE": output_file
    })

    print("Saved:", output_file)


# ============================================================
# 12. SAVE ALL-SEED SUMMARY
# ============================================================

summary_all = pd.DataFrame(
    summary_rows
)

summary_all_file = os.path.join(
    OUTPUT_DIR,
    "feature_transaction_all_seeds_summary.csv"
)

summary_all.to_csv(
    summary_all_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("TRANSACTION FEATURE EXTRACTION COMPLETED")
print("=" * 75)

print(summary_all)

print("\nMapping files:")
print(lv1_mapping_file)
print(lv2_mapping_file)

print("\nSummary file:")
print(summary_all_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
INPUT_DIR : /content/drive/MyDrive/ELAPSPLACEBO/transaction_placebo_outputs
OUTPUT_DIR: /content/drive/MyDrive/ELAPSPLACEBO/transaction_features

CREATING TRANSACTION FEATURES — SEED 1
Input rows: 246787
Expected customers: 150408
Valid transaction rows: 114432
Customers with transaction: 18053

CREATING TRANSACTION FEATURES — SEED 2
Input rows: 247742
Expected customers: 150408
Valid transaction rows: 115456
Customers with transaction: 18122

CREATING TRANSACTION FEATURES — SEED 3
Input rows: 247038
Expected customers: 150408
Valid transaction rows: 114646
Customers with transaction: 18016

CREATING TRANSACTION FEATURES — SEED 4
Input rows: 245986
Expected customers: 150408
Valid transaction rows: 113575
Customers with transaction: 17997

CREATING TRANSACTION FEATURES — SEED 5
Input rows: 247956
Expected customers: 150408
Valid transaction rows: 115552
Custo

Activity → nối với TARGET

In [ ]:
import os
from pathlib import Path

import pandas as pd
from google.colab import drive


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

PLACEBO_DIR = os.path.join(
    BASE_DIR,
    "placebo_cutoff_outputs"
)

ACTIVITY_FILE = os.path.join(
    BASE_DIR,
    "3.Data_MyVIB_Activity.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "activity_placebo_outputs"
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"
START_DATE_COL = "CLIENT_CREATE_DATE"
CUTOFF_COL = "PLACEBO_CUTOFF_DATE"
ACTIVITY_DATE_COL = "ACTIVITY_DATE"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. CHECK DIRECTORIES AND RAW ACTIVITY FILE
# ============================================================

if not Path(BASE_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục:\n{BASE_DIR}"
    )

if not Path(PLACEBO_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục placebo:\n{PLACEBO_DIR}"
    )

if not Path(ACTIVITY_FILE).exists():
    raise FileNotFoundError(
        f"Không tìm thấy file activity:\n{ACTIVITY_FILE}"
    )

print("BASE_DIR:", BASE_DIR)
print("PLACEBO_DIR:", PLACEBO_DIR)
print("ACTIVITY_FILE:", ACTIVITY_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)


# ============================================================
# 4. LOAD RAW ACTIVITY DATA ONCE
# ============================================================

activity = pd.read_csv(
    ACTIVITY_FILE,
    low_memory=False
)

activity.columns = (
    activity.columns
    .str.strip()
    .str.upper()
)

required_activity_cols = [
    ID_COL,
    ACTIVITY_DATE_COL
]

missing_activity_cols = [
    col for col in required_activity_cols
    if col not in activity.columns
]

if missing_activity_cols:
    raise ValueError(
        f"File activity thiếu cột: {missing_activity_cols}\n"
        f"Các cột hiện có: {activity.columns.tolist()}"
    )

# Chuẩn hóa mã khách hàng
activity[ID_COL] = (
    activity[ID_COL]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
)

# Chuẩn hóa ngày activity
activity[ACTIVITY_DATE_COL] = pd.to_datetime(
    activity[ACTIVITY_DATE_COL],
    errors="coerce"
)

# Loại bản ghi thiếu CUSTOMER_NUMBER
activity = activity.dropna(
    subset=[ID_COL]
).copy()

print("\n===== RAW ACTIVITY DATA =====")
print("Rows:", len(activity))
print("Customers:", activity[ID_COL].nunique())
print(
    "Missing ACTIVITY_DATE:",
    int(activity[ACTIVITY_DATE_COL].isna().sum())
)


# ============================================================
# 5. FUNCTION: PROCESS ONE SEED
# ============================================================

def process_activity_seed(seed: int) -> dict:

    print("\n" + "=" * 75)
    print(f"PROCESSING ACTIVITY — SEED {seed}")
    print("=" * 75)

    # --------------------------------------------------------
    # 5.1 File paths
    # --------------------------------------------------------

    master_file = os.path.join(
        PLACEBO_DIR,
        f"step2_customer_target_filtered_placebo_seed_{seed}.csv"
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"step4_activity_before_placebo_cutoff_seed_{seed}.csv"
    )

    summary_file = os.path.join(
        OUTPUT_DIR,
        f"step4_activity_summary_seed_{seed}.csv"
    )

    customer_check_file = os.path.join(
        OUTPUT_DIR,
        f"step4_activity_customer_check_seed_{seed}.csv"
    )

    if not Path(master_file).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file master placebo seed {seed}:\n"
            f"{master_file}"
        )

    # --------------------------------------------------------
    # 5.2 Load master placebo
    # --------------------------------------------------------

    master = pd.read_csv(
        master_file,
        low_memory=False
    )

    master.columns = (
        master.columns
        .str.strip()
        .str.upper()
    )

    required_master_cols = [
        ID_COL,
        TARGET_COL,
        START_DATE_COL,
        CUTOFF_COL
    ]

    missing_master_cols = [
        col for col in required_master_cols
        if col not in master.columns
    ]

    if missing_master_cols:
        raise ValueError(
            f"Seed {seed}: file master thiếu cột: "
            f"{missing_master_cols}\n"
            f"Các cột hiện có: {master.columns.tolist()}"
        )

    # --------------------------------------------------------
    # 5.3 Standardize master data types
    # --------------------------------------------------------

    master[ID_COL] = (
        master[ID_COL]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    master[START_DATE_COL] = pd.to_datetime(
        master[START_DATE_COL],
        errors="coerce"
    )

    master[CUTOFF_COL] = pd.to_datetime(
        master[CUTOFF_COL],
        errors="coerce"
    )

    master[TARGET_COL] = pd.to_numeric(
        master[TARGET_COL],
        errors="coerce"
    )

    for col in [
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "PLACEBO_SEED"
    ]:
        if col in master.columns:
            master[col] = pd.to_numeric(
                master[col],
                errors="coerce"
            )

    master = master.dropna(
        subset=[
            ID_COL,
            TARGET_COL
        ]
    ).copy()

    master[TARGET_COL] = (
        master[TARGET_COL]
        .astype(int)
    )

    invalid_target = ~master[
        TARGET_COL
    ].isin([0, 1])

    if invalid_target.any():
        raise ValueError(
            f"Seed {seed}: TARGET phải chỉ gồm 0 và 1."
        )

    # --------------------------------------------------------
    # 5.4 Check duplicate customers
    # --------------------------------------------------------

    duplicate_count = int(
        master[ID_COL]
        .duplicated()
        .sum()
    )

    print(
        "Duplicate customers in master:",
        duplicate_count
    )

    if duplicate_count > 0:

        duplicate_file = os.path.join(
            OUTPUT_DIR,
            f"duplicate_master_customers_seed_{seed}.csv"
        )

        master.loc[
            master[ID_COL].duplicated(
                keep=False
            )
        ].to_csv(
            duplicate_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: file master có "
            "CUSTOMER_NUMBER bị trùng."
        )

    # --------------------------------------------------------
    # 5.5 Check invalid cutoff
    # --------------------------------------------------------

    invalid_cutoff = master[
        master[CUTOFF_COL].isna()
        | master[START_DATE_COL].isna()
        | (
            master[CUTOFF_COL]
            < master[START_DATE_COL]
        )
    ].copy()

    print(
        "Customers with invalid cutoff:",
        len(invalid_cutoff)
    )

    if not invalid_cutoff.empty:

        invalid_file = os.path.join(
            OUTPUT_DIR,
            f"invalid_cutoff_seed_{seed}.csv"
        )

        invalid_cutoff.to_csv(
            invalid_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: có khách hàng thiếu cutoff "
            "hoặc cutoff trước ngày bắt đầu quan hệ."
        )

    # --------------------------------------------------------
    # 5.6 Remove overlapping activity columns from master
    # --------------------------------------------------------

    overlapping_cols = [
        col for col in activity.columns
        if col in master.columns
        and col != ID_COL
    ]

    if overlapping_cols:
        print(
            "Các cột activity trùng trong master sẽ bị bỏ trước merge:",
            overlapping_cols
        )

        master_for_merge = master.drop(
            columns=overlapping_cols
        ).copy()
    else:
        master_for_merge = master.copy()

    # --------------------------------------------------------
    # 5.7 LEFT JOIN
    # --------------------------------------------------------

    df = master_for_merge.merge(
        activity,
        on=ID_COL,
        how="left",
        validate="one_to_many"
    )

    expected_customers = (
        master_for_merge[ID_COL]
        .nunique()
    )

    print("\n===== BEFORE FILTERING =====")
    print("Rows:", len(df))
    print(
        "Customers:",
        df[ID_COL].nunique()
    )
    print(
        "Expected customers:",
        expected_customers
    )
    print(
        "Adopters:",
        df.loc[
            df[TARGET_COL] == 1,
            ID_COL
        ].nunique()
    )
    print(
        "Non-adopters:",
        df.loc[
            df[TARGET_COL] == 0,
            ID_COL
        ].nunique()
    )

    # --------------------------------------------------------
    # 5.8 Filter activity before cutoff
    # --------------------------------------------------------

    valid_activity_mask = (
        df[ACTIVITY_DATE_COL].isna()
        |
        (
            df[CUTOFF_COL].notna()
            & (
                df[ACTIVITY_DATE_COL]
                < df[CUTOFF_COL]
            )
        )
    )

    df_filtered = df.loc[
        valid_activity_mask
    ].copy()

    # --------------------------------------------------------
    # 5.9 Detect lost customers
    #
    # Khách có activity nhưng toàn bộ đều sau cutoff
    # sẽ bị mất tạm thời.
    # --------------------------------------------------------

    customers_before = set(
        master_for_merge[
            ID_COL
        ].unique()
    )

    customers_after_filter = set(
        df_filtered[
            ID_COL
        ].unique()
    )

    lost_customers = (
        customers_before
        - customers_after_filter
    )

    print("\n===== AFTER RAW FILTERING =====")
    print("Rows:", len(df_filtered))
    print(
        "Customers:",
        df_filtered[ID_COL].nunique()
    )
    print(
        "Customers temporarily lost:",
        len(lost_customers)
    )

    # --------------------------------------------------------
    # 5.10 Restore lost customers
    # --------------------------------------------------------

    if lost_customers:

        missing_rows = master_for_merge[
            master_for_merge[ID_COL].isin(
                lost_customers
            )
        ].copy()

        activity_only_cols = [
            col for col in activity.columns
            if col != ID_COL
        ]

        for col in activity_only_cols:
            if col not in missing_rows.columns:
                missing_rows[col] = pd.NA

        missing_rows = missing_rows[
            df_filtered.columns
        ]

        df_filtered = pd.concat(
            [
                df_filtered,
                missing_rows
            ],
            ignore_index=True
        )

    # --------------------------------------------------------
    # 5.11 Final customer consistency
    # --------------------------------------------------------

    final_customers = (
        df_filtered[ID_COL]
        .nunique()
    )

    print(
        "Customers after restoration:",
        final_customers
    )

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách cuối không khớp. "
            f"Expected={expected_customers}, "
            f"Final={final_customers}"
        )

    # --------------------------------------------------------
    # 5.12 Leakage check
    # --------------------------------------------------------

    leak = df_filtered[
        df_filtered[
            ACTIVITY_DATE_COL
        ].notna()
        & df_filtered[
            CUTOFF_COL
        ].notna()
        & (
            df_filtered[ACTIVITY_DATE_COL]
            >= df_filtered[CUTOFF_COL]
        )
    ].copy()

    print(
        "Remaining leakage rows:",
        len(leak)
    )

    if not leak.empty:

        leak_file = os.path.join(
            OUTPUT_DIR,
            f"activity_leakage_seed_{seed}.csv"
        )

        leak.to_csv(
            leak_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: vẫn còn activity "
            "tại hoặc sau PLACEBO_CUTOFF_DATE."
        )

    # --------------------------------------------------------
    # 5.13 Count activity per customer
    # --------------------------------------------------------

    activity_count = (
        df_filtered
        .groupby(ID_COL)[ACTIVITY_DATE_COL]
        .count()
        .rename(
            "PRE_CUTOFF_ACTIVITY_COUNT"
        )
        .reset_index()
    )

    no_activity_count = int(
        (
            activity_count[
                "PRE_CUTOFF_ACTIVITY_COUNT"
            ] == 0
        ).sum()
    )

    print(
        "Customers with no pre-cutoff activity:",
        no_activity_count
    )

    # --------------------------------------------------------
    # 5.14 Customer-level check file
    # --------------------------------------------------------

    customer_check_cols = [
        ID_COL,
        TARGET_COL,
        START_DATE_COL,
        CUTOFF_COL
    ]

    for optional_col in [
        "CUTOFF_TYPE",
        "PLACEBO_SEED",
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "SAMPLING_STATUS"
    ]:
        if optional_col in master_for_merge.columns:
            customer_check_cols.append(
                optional_col
            )

    customer_check = (
        master_for_merge[
            customer_check_cols
        ]
        .merge(
            activity_count,
            on=ID_COL,
            how="left",
            validate="one_to_one"
        )
    )

    customer_check[
        "PRE_CUTOFF_ACTIVITY_COUNT"
    ] = (
        customer_check[
            "PRE_CUTOFF_ACTIVITY_COUNT"
        ]
        .fillna(0)
        .astype(int)
    )

    customer_check[
        "NO_PRE_CUTOFF_ACTIVITY"
    ] = (
        customer_check[
            "PRE_CUTOFF_ACTIVITY_COUNT"
        ] == 0
    )

    # --------------------------------------------------------
    # 5.15 Summary by target
    # --------------------------------------------------------

    summary_agg = {
        "CUSTOMERS": (
            ID_COL,
            "nunique"
        ),
        "ACTIVITY_ROWS": (
            ACTIVITY_DATE_COL,
            "count"
        ),
        "EARLIEST_ACTIVITY": (
            ACTIVITY_DATE_COL,
            "min"
        ),
        "LATEST_ACTIVITY": (
            ACTIVITY_DATE_COL,
            "max"
        )
    }

    if "ASSIGNED_CUTOFF_DAYS" in df_filtered.columns:
        summary_agg[
            "MEAN_CUTOFF_DAYS"
        ] = (
            "ASSIGNED_CUTOFF_DAYS",
            "mean"
        )

        summary_agg[
            "MEDIAN_CUTOFF_DAYS"
        ] = (
            "ASSIGNED_CUTOFF_DAYS",
            "median"
        )

    summary = (
        df_filtered
        .groupby(TARGET_COL)
        .agg(
            **summary_agg
        )
        .reset_index()
    )

    summary["SEED"] = seed

    # --------------------------------------------------------
    # 5.16 Sort
    # --------------------------------------------------------

    df_filtered = (
        df_filtered
        .sort_values(
            by=[
                ID_COL,
                ACTIVITY_DATE_COL
            ],
            na_position="last"
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 5.17 Save outputs
    # --------------------------------------------------------

    df_filtered.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    customer_check.to_csv(
        customer_check_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== SAVED =====")
    print(output_file)
    print(summary_file)
    print(customer_check_file)

    return {
        "SEED": seed,
        "INPUT_CUSTOMERS": expected_customers,
        "FINAL_CUSTOMERS": final_customers,
        "OUTPUT_ROWS": len(df_filtered),
        "NO_ACTIVITY_CUSTOMERS": no_activity_count,
        "LEAKAGE_ROWS": len(leak),
        "OUTPUT_FILE": output_file
    }


# ============================================================
# 6. RUN ALL FIVE SEEDS
# ============================================================

run_results = []

for seed in SEEDS:

    result = process_activity_seed(
        seed
    )

    run_results.append(
        result
    )


# ============================================================
# 7. COMBINE SEED SUMMARIES
# ============================================================

run_summary = pd.DataFrame(
    run_results
)

run_summary_file = os.path.join(
    OUTPUT_DIR,
    "step4_activity_all_seeds_summary.csv"
)

run_summary.to_csv(
    run_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("ALL FIVE ACTIVITY SEEDS COMPLETED")
print("=" * 75)

print(run_summary)

print("\nSaved combined summary:")
print(run_summary_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/ELAPSPLACEBO
PLACEBO_DIR: /content/drive/MyDrive/ELAPSPLACEBO/placebo_cutoff_outputs
ACTIVITY_FILE: /content/drive/MyDrive/ELAPSPLACEBO/3.Data_MyVIB_Activity.csv
OUTPUT_DIR: /content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs

===== RAW ACTIVITY DATA =====
Rows: 16132675
Customers: 77741
Missing ACTIVITY_DATE: 0

PROCESSING ACTIVITY — SEED 1
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 14444693
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 2169048
Customers: 130302
Customers temporarily lost: 20106


/tmp/ipykernel_54151/2794546692.py:494: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff activity: 107895

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_before_placebo_cutoff_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_summary_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_customer_check_seed_1.csv

PROCESSING ACTIVITY — SEED 2
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 14444693
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 2170463
Customers: 130390
Customers temporarily lost: 20018


/tmp/ipykernel_54151/2794546692.py:494: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff activity: 107807

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_before_placebo_cutoff_seed_2.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_summary_seed_2.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_customer_check_seed_2.csv

PROCESSING ACTIVITY — SEED 3
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 14444693
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 2174031
Customers: 130300
Customers temporarily lost: 20108


...

Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff activity: 107897

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_before_placebo_cutoff_seed_3.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_summary_seed_3.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_customer_check_seed_3.csv

PROCESSING ACTIVITY — SEED 4
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 14444693
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 2144130
Customers: 130398
Customers temporarily lost: 20010


/tmp/ipykernel_54151/2794546692.py:494: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff activity: 107799

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_before_placebo_cutoff_seed_4.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_summary_seed_4.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_customer_check_seed_4.csv

PROCESSING ACTIVITY — SEED 5
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 14444693
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 2178009
Customers: 130295
Customers temporarily lost: 20113


/tmp/ipykernel_54151/2794546692.py:494: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff activity: 107902

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_before_placebo_cutoff_seed_5.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_summary_seed_5.csv
/content/drive/MyDrive/ELAPSPLACEBO/activity_placebo_outputs/step4_activity_customer_check_seed_5.csv

ALL FIVE ACTIVITY SEEDS COMPLETED
   SEED  INPUT_CUSTOMERS  FINAL_CUSTOMERS  OUTPUT_ROWS  NO_ACTIVITY_CUSTOMERS  \
0     1           150408           150408      2189154                 107895   
1     2           150408           150408      2190481                 107807   
2     3           150408           150408      2194139                 107897   
3     4           150408           150408      2164140                 107799   
4     5           150408           150408      2198122                 107902   

   LEAKAGE_ROWS                              

TẠO FEATURE CHO ACTIVITY

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive", force_remount=False)

# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

INPUT_DIR = os.path.join(
    BASE_DIR,
    "activity_placebo_outputs"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "activity_features"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

SEEDS = [1,2,3,4,5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"
DATE_COL = "ACTIVITY_DATE"
ACTIVITY_COL = "ACTIVITY_NAME"

# ============================================================
# PROCESS EACH SEED
# ============================================================

for seed in SEEDS:

    print("="*70)
    print(f"PROCESSING SEED {seed}")
    print("="*70)

    INPUT_FILE = os.path.join(
        INPUT_DIR,
        f"step4_activity_before_placebo_cutoff_seed_{seed}.csv"
    )

    OUTPUT_FILE = os.path.join(
        OUTPUT_DIR,
        f"feature_activity_2features_placebo_seed_{seed}.csv"
    )

    AUDIT_FILE = os.path.join(
        OUTPUT_DIR,
        f"feature_activity_audit_placebo_seed_{seed}.csv"
    )

    SUMMARY_FILE = os.path.join(
        OUTPUT_DIR,
        f"feature_activity_summary_placebo_seed_{seed}.csv"
    )

    if not Path(INPUT_FILE).exists():
        raise FileNotFoundError(INPUT_FILE)

    # ============================================================
    # LOAD DATA
    # ============================================================

    df = pd.read_csv(
        INPUT_FILE,
        low_memory=False
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )

    # ============================================================
    # CHECK REQUIRED COLUMNS
    # ============================================================

    required_cols = [
        ID_COL,
        TARGET_COL,
        DATE_COL,
        ACTIVITY_COL
    ]

    missing = [
        c for c in required_cols
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Seed {seed} thiếu cột {missing}"
        )

    # ============================================================
    # STANDARDIZE
    # ============================================================

    df[DATE_COL] = pd.to_datetime(
        df[DATE_COL],
        errors="coerce"
    )

    df[ACTIVITY_COL] = (
        df[ACTIVITY_COL]
        .astype("string")
        .str.upper()
        .str.strip()
    )

    # ============================================================
    # CUSTOMER BASE
    # ============================================================

    customer_base = (
        df[
            [ID_COL, TARGET_COL]
        ]
        .drop_duplicates(
            subset=[ID_COL]
        )
        .copy()
    )

    expected = customer_base[
        ID_COL
    ].nunique()

    # ============================================================
    # VALID ACTIVITY
    # ============================================================

    valid = df[
        df[DATE_COL].notna()
    ].copy()

    valid["IS_LOGIN"] = (
        valid[ACTIVITY_COL]
        .str.contains(
            "LOGIN",
            na=False
        )
    ).astype(int)

    valid["IS_INTEREST"] = (
        valid[ACTIVITY_COL]
        .str.contains(
            "INTEREST_RATE",
            na=False
        )
    ).astype(int)

    valid["HAS_ACTIVITY"] = 1

    # ============================================================
    # FEATURE ENGINEERING
    # ============================================================

    feat = (
        valid
        .groupby(ID_COL)
        .agg(
            LOGIN_COUNT=(
                "IS_LOGIN",
                "sum"
            ),
            INTEREST_COUNT=(
                "IS_INTEREST",
                "sum"
            ),
            TOTAL_ACTIVITY=(
                "HAS_ACTIVITY",
                "sum"
            ),
            ACTIVE_DAYS=(
                DATE_COL,
                lambda x:
                x.dt.normalize().nunique()
            )
        )
        .reset_index()
    )

    feat["LOGIN_PER_ACTIVE_DAY"] = (
        feat["LOGIN_COUNT"]
        /
        feat["ACTIVE_DAYS"]
        .replace(0,np.nan)
    ).fillna(0)

    feat["INTEREST_RATE_RATIO"] = (
        feat["INTEREST_COUNT"]
        /
        feat["TOTAL_ACTIVITY"]
        .replace(0,np.nan)
    ).fillna(0)

    # ============================================================
    # MERGE
    # ============================================================

    feature_full = customer_base.merge(
        feat,
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )

    count_cols = [
        "LOGIN_COUNT",
        "INTEREST_COUNT",
        "TOTAL_ACTIVITY",
        "ACTIVE_DAYS"
    ]

    feature_cols = [
        "LOGIN_PER_ACTIVE_DAY",
        "INTEREST_RATE_RATIO"
    ]

    feature_full[
        count_cols+feature_cols
    ] = feature_full[
        count_cols+feature_cols
    ].fillna(0)

    feature_full[count_cols] = (
        feature_full[count_cols]
        .astype(int)
    )

    # ============================================================
    # FINAL FEATURE
    # ============================================================

    feature_final = feature_full[
        [
            ID_COL,
            "LOGIN_PER_ACTIVE_DAY",
            "INTEREST_RATE_RATIO",
            TARGET_COL
        ]
    ].copy()

    # ============================================================
    # VALIDATION
    # ============================================================

    final = feature_final[
        ID_COL
    ].nunique()

    duplicate = (
        feature_final[
            ID_COL
        ]
        .duplicated()
        .sum()
    )

    if expected != final:
        raise ValueError(
            "Customer mismatch"
        )

    if duplicate > 0:
        raise ValueError(
            "Duplicate customer"
        )

    # ============================================================
    # SUMMARY
    # ============================================================

    summary = (
        feature_full
        .groupby(TARGET_COL)
        .agg(
            Customers=(
                ID_COL,
                "nunique"
            ),
            Mean_Login_Per_Day=(
                "LOGIN_PER_ACTIVE_DAY",
                "mean"
            ),
            Mean_Interest_Ratio=(
                "INTEREST_RATE_RATIO",
                "mean"
            ),
            Mean_Total_Activity=(
                "TOTAL_ACTIVITY",
                "mean"
            ),
            Customers_No_Activity=(
                "TOTAL_ACTIVITY",
                lambda x:(x==0).sum()
            )
        )
        .reset_index()
    )

    # ============================================================
    # SAVE
    # ============================================================

    feature_final.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    feature_full.to_csv(
        AUDIT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        SUMMARY_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", OUTPUT_FILE)

print("\nALL FIVE SEEDS COMPLETED.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROCESSING SEED 1
Saved: /content/drive/MyDrive/ELAPSPLACEBO/activity_features/feature_activity_2features_placebo_seed_1.csv
PROCESSING SEED 2
Saved: /content/drive/MyDrive/ELAPSPLACEBO/activity_features/feature_activity_2features_placebo_seed_2.csv
PROCESSING SEED 3
Saved: /content/drive/MyDrive/ELAPSPLACEBO/activity_features/feature_activity_2features_placebo_seed_3.csv
PROCESSING SEED 4
Saved: /content/drive/MyDrive/ELAPSPLACEBO/activity_features/feature_activity_2features_placebo_seed_4.csv
PROCESSING SEED 5
Saved: /content/drive/MyDrive/ELAPSPLACEBO/activity_features/feature_activity_2features_placebo_seed_5.csv

ALL FIVE SEEDS COMPLETED.


XỬ LÝ DEPOSITE

In [ ]:
import os
from pathlib import Path

import pandas as pd
from google.colab import drive


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

PLACEBO_DIR = os.path.join(
    BASE_DIR,
    "placebo_cutoff_outputs"
)

DEPOSIT_FILE = os.path.join(
    BASE_DIR,
    "4.Data_Deposit.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "deposit_placebo_outputs"
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"
START_DATE_COL = "CLIENT_CREATE_DATE"
CUTOFF_COL = "PLACEBO_CUTOFF_DATE"
DEPOSIT_DATE_COL = "MONTH"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. CHECK FILES
# ============================================================

if not Path(BASE_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục:\n{BASE_DIR}"
    )

if not Path(PLACEBO_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục placebo:\n{PLACEBO_DIR}"
    )

if not Path(DEPOSIT_FILE).exists():
    raise FileNotFoundError(
        f"Không tìm thấy file deposit:\n{DEPOSIT_FILE}"
    )

print("BASE_DIR:", BASE_DIR)
print("PLACEBO_DIR:", PLACEBO_DIR)
print("DEPOSIT_FILE:", DEPOSIT_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)


# ============================================================
# 4. LOAD RAW DEPOSIT DATA ONCE
# ============================================================

deposit = pd.read_csv(
    DEPOSIT_FILE,
    low_memory=False
)

deposit.columns = (
    deposit.columns
    .str.strip()
    .str.upper()
)

required_deposit_cols = [
    ID_COL,
    DEPOSIT_DATE_COL
]

missing_deposit_cols = [
    col for col in required_deposit_cols
    if col not in deposit.columns
]

if missing_deposit_cols:
    raise ValueError(
        f"File deposit thiếu cột: {missing_deposit_cols}\n"
        f"Các cột hiện có: {deposit.columns.tolist()}"
    )

# Chuẩn hóa mã khách hàng
deposit[ID_COL] = (
    deposit[ID_COL]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
)

# Chuẩn hóa ngày/tháng deposit
deposit[DEPOSIT_DATE_COL] = pd.to_datetime(
    deposit[DEPOSIT_DATE_COL],
    errors="coerce"
)

# Loại bản ghi thiếu mã khách hàng
deposit = deposit.dropna(
    subset=[ID_COL]
).copy()

print("\n===== RAW DEPOSIT DATA =====")
print("Rows:", len(deposit))
print("Customers:", deposit[ID_COL].nunique())
print(
    "Missing MONTH:",
    int(deposit[DEPOSIT_DATE_COL].isna().sum())
)


# ============================================================
# 5. FUNCTION: PROCESS ONE SEED
# ============================================================

def process_deposit_seed(seed: int) -> dict:

    print("\n" + "=" * 75)
    print(f"PROCESSING DEPOSIT — SEED {seed}")
    print("=" * 75)

    # --------------------------------------------------------
    # 5.1 File paths
    # --------------------------------------------------------

    master_file = os.path.join(
        PLACEBO_DIR,
        f"step2_customer_target_filtered_placebo_seed_{seed}.csv"
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"step5_deposit_before_placebo_cutoff_seed_{seed}.csv"
    )

    summary_file = os.path.join(
        OUTPUT_DIR,
        f"step5_deposit_summary_seed_{seed}.csv"
    )

    customer_check_file = os.path.join(
        OUTPUT_DIR,
        f"step5_deposit_customer_check_seed_{seed}.csv"
    )

    if not Path(master_file).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file master placebo seed {seed}:\n"
            f"{master_file}"
        )

    # --------------------------------------------------------
    # 5.2 Load master placebo
    # --------------------------------------------------------

    master = pd.read_csv(
        master_file,
        low_memory=False
    )

    master.columns = (
        master.columns
        .str.strip()
        .str.upper()
    )

    required_master_cols = [
        ID_COL,
        TARGET_COL,
        START_DATE_COL,
        CUTOFF_COL
    ]

    missing_master_cols = [
        col for col in required_master_cols
        if col not in master.columns
    ]

    if missing_master_cols:
        raise ValueError(
            f"Seed {seed}: file master thiếu cột: "
            f"{missing_master_cols}\n"
            f"Các cột hiện có: {master.columns.tolist()}"
        )

    # --------------------------------------------------------
    # 5.3 Standardize master data
    # --------------------------------------------------------

    master[ID_COL] = (
        master[ID_COL]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    master[START_DATE_COL] = pd.to_datetime(
        master[START_DATE_COL],
        errors="coerce"
    )

    master[CUTOFF_COL] = pd.to_datetime(
        master[CUTOFF_COL],
        errors="coerce"
    )

    master[TARGET_COL] = pd.to_numeric(
        master[TARGET_COL],
        errors="coerce"
    )

    for col in [
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "PLACEBO_SEED"
    ]:
        if col in master.columns:
            master[col] = pd.to_numeric(
                master[col],
                errors="coerce"
            )

    master = master.dropna(
        subset=[
            ID_COL,
            TARGET_COL
        ]
    ).copy()

    master[TARGET_COL] = (
        master[TARGET_COL]
        .astype(int)
    )

    invalid_target = ~master[
        TARGET_COL
    ].isin([0, 1])

    if invalid_target.any():
        raise ValueError(
            f"Seed {seed}: TARGET phải chỉ gồm 0 và 1."
        )

    # --------------------------------------------------------
    # 5.4 Check duplicate customers
    # --------------------------------------------------------

    duplicate_count = int(
        master[ID_COL]
        .duplicated()
        .sum()
    )

    print(
        "Duplicate customers in master:",
        duplicate_count
    )

    if duplicate_count > 0:

        duplicate_file = os.path.join(
            OUTPUT_DIR,
            f"duplicate_master_customers_seed_{seed}.csv"
        )

        master.loc[
            master[ID_COL].duplicated(
                keep=False
            )
        ].to_csv(
            duplicate_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: file master có "
            "CUSTOMER_NUMBER bị trùng."
        )

    # --------------------------------------------------------
    # 5.5 Check invalid cutoff
    # --------------------------------------------------------

    invalid_cutoff = master[
        master[CUTOFF_COL].isna()
        | master[START_DATE_COL].isna()
        | (
            master[CUTOFF_COL]
            < master[START_DATE_COL]
        )
    ].copy()

    print(
        "Customers with invalid cutoff:",
        len(invalid_cutoff)
    )

    if not invalid_cutoff.empty:

        invalid_file = os.path.join(
            OUTPUT_DIR,
            f"invalid_cutoff_seed_{seed}.csv"
        )

        invalid_cutoff.to_csv(
            invalid_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: có khách hàng thiếu cutoff "
            "hoặc cutoff trước ngày bắt đầu quan hệ."
        )

    # --------------------------------------------------------
    # 5.6 Remove overlapping deposit columns from master
    #
    # Tránh xuất hiện MONTH_X, MONTH_Y sau merge.
    # --------------------------------------------------------

    overlapping_cols = [
        col for col in deposit.columns
        if col in master.columns
        and col != ID_COL
    ]

    if overlapping_cols:
        print(
            "Các cột deposit trùng trong master sẽ bị bỏ trước merge:",
            overlapping_cols
        )

        master_for_merge = master.drop(
            columns=overlapping_cols
        ).copy()
    else:
        master_for_merge = master.copy()

    # --------------------------------------------------------
    # 5.7 LEFT JOIN
    # --------------------------------------------------------

    df = master_for_merge.merge(
        deposit,
        on=ID_COL,
        how="left",
        validate="one_to_many"
    )

    expected_customers = (
        master_for_merge[ID_COL]
        .nunique()
    )

    print("\n===== BEFORE FILTERING =====")
    print("Rows:", len(df))
    print(
        "Customers:",
        df[ID_COL].nunique()
    )
    print(
        "Expected customers:",
        expected_customers
    )
    print(
        "Adopters:",
        df.loc[
            df[TARGET_COL] == 1,
            ID_COL
        ].nunique()
    )
    print(
        "Non-adopters:",
        df.loc[
            df[TARGET_COL] == 0,
            ID_COL
        ].nunique()
    )

    # --------------------------------------------------------
    # 5.8 Filter deposit records before cutoff
    #
    # Giữ:
    # - khách không có deposit;
    # - record deposit trước PLACEBO_CUTOFF_DATE.
    # --------------------------------------------------------

    valid_deposit_mask = (
        df[DEPOSIT_DATE_COL].isna()
        |
        (
            df[CUTOFF_COL].notna()
            & (
                df[DEPOSIT_DATE_COL]
                < df[CUTOFF_COL]
            )
        )
    )

    df_filtered = df.loc[
        valid_deposit_mask
    ].copy()

    # --------------------------------------------------------
    # 5.9 Detect customers lost after filtering
    # --------------------------------------------------------

    customers_before = set(
        master_for_merge[
            ID_COL
        ].unique()
    )

    customers_after_filter = set(
        df_filtered[
            ID_COL
        ].unique()
    )

    lost_customers = (
        customers_before
        - customers_after_filter
    )

    print("\n===== AFTER RAW FILTERING =====")
    print("Rows:", len(df_filtered))
    print(
        "Customers:",
        df_filtered[ID_COL].nunique()
    )
    print(
        "Customers temporarily lost:",
        len(lost_customers)
    )

    # --------------------------------------------------------
    # 5.10 Restore lost customers
    #
    # Khách có deposit nhưng tất cả record đều sau cutoff.
    # --------------------------------------------------------

    if lost_customers:

        missing_rows = master_for_merge[
            master_for_merge[ID_COL].isin(
                lost_customers
            )
        ].copy()

        deposit_only_cols = [
            col for col in deposit.columns
            if col != ID_COL
        ]

        for col in deposit_only_cols:
            if col not in missing_rows.columns:
                missing_rows[col] = pd.NA

        missing_rows = missing_rows[
            df_filtered.columns
        ]

        df_filtered = pd.concat(
            [
                df_filtered,
                missing_rows
            ],
            ignore_index=True
        )

    # --------------------------------------------------------
    # 5.11 Final customer consistency
    # --------------------------------------------------------

    final_customers = (
        df_filtered[ID_COL]
        .nunique()
    )

    print(
        "Customers after restoration:",
        final_customers
    )

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách cuối không khớp. "
            f"Expected={expected_customers}, "
            f"Final={final_customers}"
        )

    # --------------------------------------------------------
    # 5.12 Leakage check
    # --------------------------------------------------------

    leak = df_filtered[
        df_filtered[
            DEPOSIT_DATE_COL
        ].notna()
        & df_filtered[
            CUTOFF_COL
        ].notna()
        & (
            df_filtered[DEPOSIT_DATE_COL]
            >= df_filtered[CUTOFF_COL]
        )
    ].copy()

    print(
        "Remaining leakage rows:",
        len(leak)
    )

    if not leak.empty:

        leak_file = os.path.join(
            OUTPUT_DIR,
            f"deposit_leakage_seed_{seed}.csv"
        )

        leak.to_csv(
            leak_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: vẫn còn deposit "
            "tại hoặc sau PLACEBO_CUTOFF_DATE."
        )

    # --------------------------------------------------------
    # 5.13 Count deposit rows per customer
    # --------------------------------------------------------

    deposit_count = (
        df_filtered
        .groupby(ID_COL)[DEPOSIT_DATE_COL]
        .count()
        .rename(
            "PRE_CUTOFF_DEPOSIT_COUNT"
        )
        .reset_index()
    )

    no_deposit_count = int(
        (
            deposit_count[
                "PRE_CUTOFF_DEPOSIT_COUNT"
            ] == 0
        ).sum()
    )

    print(
        "Customers with no pre-cutoff deposit:",
        no_deposit_count
    )

    # --------------------------------------------------------
    # 5.14 Customer-level check
    # --------------------------------------------------------

    customer_check_cols = [
        ID_COL,
        TARGET_COL,
        START_DATE_COL,
        CUTOFF_COL
    ]

    for optional_col in [
        "CUTOFF_TYPE",
        "PLACEBO_SEED",
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "SAMPLING_STATUS"
    ]:
        if optional_col in master_for_merge.columns:
            customer_check_cols.append(
                optional_col
            )

    customer_check = (
        master_for_merge[
            customer_check_cols
        ]
        .merge(
            deposit_count,
            on=ID_COL,
            how="left",
            validate="one_to_one"
        )
    )

    customer_check[
        "PRE_CUTOFF_DEPOSIT_COUNT"
    ] = (
        customer_check[
            "PRE_CUTOFF_DEPOSIT_COUNT"
        ]
        .fillna(0)
        .astype(int)
    )

    customer_check[
        "NO_PRE_CUTOFF_DEPOSIT"
    ] = (
        customer_check[
            "PRE_CUTOFF_DEPOSIT_COUNT"
        ] == 0
    )

    # --------------------------------------------------------
    # 5.15 Summary by target
    # --------------------------------------------------------

    summary_agg = {
        "CUSTOMERS": (
            ID_COL,
            "nunique"
        ),
        "DEPOSIT_ROWS": (
            DEPOSIT_DATE_COL,
            "count"
        ),
        "EARLIEST_DEPOSIT": (
            DEPOSIT_DATE_COL,
            "min"
        ),
        "LATEST_DEPOSIT": (
            DEPOSIT_DATE_COL,
            "max"
        )
    }

    if "ASSIGNED_CUTOFF_DAYS" in df_filtered.columns:
        summary_agg[
            "MEAN_CUTOFF_DAYS"
        ] = (
            "ASSIGNED_CUTOFF_DAYS",
            "mean"
        )

        summary_agg[
            "MEDIAN_CUTOFF_DAYS"
        ] = (
            "ASSIGNED_CUTOFF_DAYS",
            "median"
        )

    summary = (
        df_filtered
        .groupby(TARGET_COL)
        .agg(
            **summary_agg
        )
        .reset_index()
    )

    summary["SEED"] = seed

    # --------------------------------------------------------
    # 5.16 Sort
    # --------------------------------------------------------

    df_filtered = (
        df_filtered
        .sort_values(
            by=[
                ID_COL,
                DEPOSIT_DATE_COL
            ],
            na_position="last"
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 5.17 Save
    # --------------------------------------------------------

    df_filtered.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    customer_check.to_csv(
        customer_check_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== SAVED =====")
    print(output_file)
    print(summary_file)
    print(customer_check_file)

    return {
        "SEED": seed,
        "INPUT_CUSTOMERS": expected_customers,
        "FINAL_CUSTOMERS": final_customers,
        "OUTPUT_ROWS": len(df_filtered),
        "NO_DEPOSIT_CUSTOMERS": no_deposit_count,
        "LEAKAGE_ROWS": len(leak),
        "OUTPUT_FILE": output_file
    }


# ============================================================
# 6. RUN ALL FIVE SEEDS
# ============================================================

run_results = []

for seed in SEEDS:

    result = process_deposit_seed(
        seed
    )

    run_results.append(
        result
    )


# ============================================================
# 7. COMBINE SEED SUMMARIES
# ============================================================

run_summary = pd.DataFrame(
    run_results
)

run_summary_file = os.path.join(
    OUTPUT_DIR,
    "step5_deposit_all_seeds_summary.csv"
)

run_summary.to_csv(
    run_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("ALL FIVE DEPOSIT SEEDS COMPLETED")
print("=" * 75)

print(run_summary)

print("\nSaved combined summary:")
print(run_summary_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/ELAPSPLACEBO
PLACEBO_DIR: /content/drive/MyDrive/ELAPSPLACEBO/placebo_cutoff_outputs
DEPOSIT_FILE: /content/drive/MyDrive/ELAPSPLACEBO/4.Data_Deposit.csv
OUTPUT_DIR: /content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs

===== RAW DEPOSIT DATA =====
Rows: 1258424
Customers: 223817
Missing MONTH: 0

PROCESSING DEPOSIT — SEED 1
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 691238
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 110690
Customers: 73808
Customers temporarily lost: 76600


/tmp/ipykernel_54151/995751381.py:499: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff deposit: 111511

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_before_placebo_cutoff_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_summary_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_customer_check_seed_1.csv

PROCESSING DEPOSIT — SEED 2
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 691238
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 111016
Customers: 73890
Customers temporarily lost: 76518


/tmp/ipykernel_54151/995751381.py:499: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff deposit: 111429

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_before_placebo_cutoff_seed_2.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_summary_seed_2.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_customer_check_seed_2.csv

PROCESSING DEPOSIT — SEED 3
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 691238
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 110817
Customers: 73844
Customers temporarily lost: 76564


...

Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff deposit: 111475

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_before_placebo_cutoff_seed_3.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_summary_seed_3.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_customer_check_seed_3.csv

PROCESSING DEPOSIT — SEED 4
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 691238
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 111151
Customers: 73983
Customers temporarily lost: 76425


/tmp/ipykernel_54151/995751381.py:499: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff deposit: 111336

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_before_placebo_cutoff_seed_4.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_summary_seed_4.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_customer_check_seed_4.csv

PROCESSING DEPOSIT — SEED 5
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 691238
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 110335
Customers: 73919
Customers temporarily lost: 76489


/tmp/ipykernel_54151/995751381.py:499: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff deposit: 111400

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_before_placebo_cutoff_seed_5.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_summary_seed_5.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs/step5_deposit_customer_check_seed_5.csv

ALL FIVE DEPOSIT SEEDS COMPLETED
   SEED  INPUT_CUSTOMERS  FINAL_CUSTOMERS  OUTPUT_ROWS  NO_DEPOSIT_CUSTOMERS  \
0     1           150408           150408       187290                111511   
1     2           150408           150408       187534                111429   
2     3           150408           150408       187381                111475   
3     4           150408           150408       187576                111336   
4     5           150408           150408       186824                111400   

   LEAKAGE_ROWS                                        OUTP

TẠO FEATURE DEPOSITE

In [ ]:
import pandas as pd

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("step5_deposit_before_card_keep_all_customers.csv")

# =========================
# STANDARDIZE
# =========================
df.columns = df.columns.str.strip().str.upper()

df["MONTH"] = pd.to_datetime(df["MONTH"], errors="coerce")

# =========================
# SORT THEO THỜI GIAN
# =========================
df = df.sort_values(["CUSTOMER_NUMBER", "MONTH"])

# =========================
# LẤY SNAPSHOT GẦN NHẤT
# =========================
feat = df.groupby("CUSTOMER_NUMBER").tail(1).copy()

# =========================
# CHỌN 4 FEATURE
# =========================
feature_deposit = feat[[
    "CUSTOMER_NUMBER",
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE",
    "TARGET"
]]

# =========================
# FILL NA (KHÁCH KHÔNG CÓ DEPOSIT)
# =========================
feature_deposit = feature_deposit.fillna(0)

# =========================
# SAVE
# =========================
feature_deposit.to_csv("feature_deposit_4features.csv", index=False)

print("Saved: feature_deposit_4features.csv")
print("Shape:", feature_deposit.shape)
print(feature_deposit.head())

Saved: feature_deposit_4features.csv
Shape: (130925, 6)
    CUSTOMER_NUMBER  COUNT_CA_ACCT  AVG_CA_BALANCE  COUNT_TD_ACCT  \
2                 0            1.0      5280833.33            0.0   
9                 3            1.0       646748.67            0.0   
10                8            0.0            0.00            0.0   
12                9            1.0      5369682.93            0.0   
13               13            0.0            0.00            0.0   

    AVG_TD_BALANCE  TARGET  
2              0.0       0  
9              0.0       0  
10             0.0       1  
12             0.0       0  
13             0.0       0  


XỬ LÝ FILE CUSTOMER

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

INPUT_DIR = os.path.join(
    BASE_DIR,
    "deposit_placebo_outputs"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "deposit_features"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"
DATE_COL = "MONTH"

DEPOSIT_FEATURE_COLS = [
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE"
]


# ============================================================
# 3. CHECK INPUT DIRECTORY
# ============================================================

if not Path(INPUT_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục đầu vào:\n{INPUT_DIR}"
    )

print("INPUT_DIR :", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


# ============================================================
# 4. PROCESS ONE SEED
# ============================================================

def build_deposit_features(seed: int) -> dict:

    print("\n" + "=" * 75)
    print(f"PROCESSING DEPOSIT FEATURES — SEED {seed}")
    print("=" * 75)

    input_file = os.path.join(
        INPUT_DIR,
        f"step5_deposit_before_placebo_cutoff_seed_{seed}.csv"
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"feature_deposit_4features_placebo_seed_{seed}.csv"
    )

    audit_file = os.path.join(
        OUTPUT_DIR,
        f"feature_deposit_audit_placebo_seed_{seed}.csv"
    )

    summary_file = os.path.join(
        OUTPUT_DIR,
        f"feature_deposit_summary_placebo_seed_{seed}.csv"
    )

    if not Path(input_file).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file seed {seed}:\n{input_file}"
        )

    # ========================================================
    # 4.1 LOAD DATA
    # ========================================================

    df = pd.read_csv(
        input_file,
        low_memory=False
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )

    print("Input shape:", df.shape)

    # ========================================================
    # 4.2 CHECK REQUIRED COLUMNS
    # ========================================================

    required_cols = [
        ID_COL,
        TARGET_COL,
        DATE_COL
    ] + DEPOSIT_FEATURE_COLS

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Seed {seed} thiếu các cột: {missing_cols}\n"
            f"Các cột hiện có: {df.columns.tolist()}"
        )

    # ========================================================
    # 4.3 STANDARDIZE TYPES
    # ========================================================

    df[ID_COL] = (
        df[ID_COL]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce"
    )

    df[DATE_COL] = pd.to_datetime(
        df[DATE_COL],
        errors="coerce"
    )

    for col in DEPOSIT_FEATURE_COLS:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # Loại dòng thiếu mã khách hàng
    df = df.dropna(
        subset=[ID_COL]
    ).copy()

    # ========================================================
    # 4.4 CHECK TARGET CONSISTENCY
    # ========================================================

    target_nunique = (
        df.groupby(ID_COL)[TARGET_COL]
        .nunique(dropna=True)
    )

    inconsistent_target = target_nunique[
        target_nunique > 1
    ]

    print(
        "Customers with inconsistent TARGET:",
        len(inconsistent_target)
    )

    if not inconsistent_target.empty:
        raise ValueError(
            f"Seed {seed}: một số khách hàng có nhiều TARGET."
        )

    missing_target_customers = (
        df.groupby(ID_COL)[TARGET_COL]
        .apply(lambda x: x.isna().all())
    )

    if missing_target_customers.any():
        raise ValueError(
            f"Seed {seed}: có "
            f"{int(missing_target_customers.sum())} khách thiếu TARGET."
        )

    # ========================================================
    # 4.5 CREATE CUSTOMER BASE
    #
    # Một dòng cho mỗi khách, không phụ thuộc có deposit hay không
    # ========================================================

    base_cols = [
        ID_COL,
        TARGET_COL
    ]

    optional_audit_cols = [
        "CLIENT_CREATE_DATE",
        "PLACEBO_CUTOFF_DATE",
        "CUTOFF_TYPE",
        "PLACEBO_SEED",
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "SAMPLING_STATUS"
    ]

    for col in optional_audit_cols:
        if col in df.columns:
            base_cols.append(col)

    customer_base = (
        df[base_cols]
        .sort_values(ID_COL)
        .drop_duplicates(
            subset=[ID_COL],
            keep="first"
        )
        .copy()
    )

    expected_customers = (
        customer_base[ID_COL]
        .nunique()
    )

    print(
        "Expected customers:",
        expected_customers
    )

    # ========================================================
    # 4.6 SELECT VALID DEPOSIT SNAPSHOTS
    #
    # Chỉ dùng dòng MONTH hợp lệ.
    # Các dòng được thêm lại để giữ khách có MONTH = NaT
    # không được tham gia chọn snapshot.
    # ========================================================

    valid_snapshots = df.loc[
        df[DATE_COL].notna()
    ].copy()

    print(
        "Valid deposit rows:",
        len(valid_snapshots)
    )

    print(
        "Customers with at least one valid deposit snapshot:",
        valid_snapshots[ID_COL].nunique()
    )

    # ========================================================
    # 4.7 TAKE LATEST VALID SNAPSHOT PER CUSTOMER
    # ========================================================

    if not valid_snapshots.empty:

        valid_snapshots = (
            valid_snapshots
            .sort_values(
                by=[
                    ID_COL,
                    DATE_COL
                ],
                na_position="last"
            )
        )

        latest_snapshot = (
            valid_snapshots
            .groupby(
                ID_COL,
                as_index=False,
                sort=False
            )
            .tail(1)
            .copy()
        )

        latest_features = latest_snapshot[
            [
                ID_COL,
                DATE_COL
            ] + DEPOSIT_FEATURE_COLS
        ].copy()

    else:
        latest_features = pd.DataFrame(
            columns=[
                ID_COL,
                DATE_COL
            ] + DEPOSIT_FEATURE_COLS
        )

    # ========================================================
    # 4.8 MERGE WITH ALL CUSTOMERS
    # ========================================================

    feature_full = customer_base.merge(
        latest_features,
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )

    # Đánh dấu khách có snapshot hợp lệ hay không
    feature_full[
        "HAS_PRE_CUTOFF_DEPOSIT_SNAPSHOT"
    ] = (
        feature_full[DATE_COL].notna()
    ).astype(int)

    # Chỉ điền 0 cho bốn feature deposit
    feature_full[
        DEPOSIT_FEATURE_COLS
    ] = (
        feature_full[
            DEPOSIT_FEATURE_COLS
        ]
        .fillna(0)
    )

    # ========================================================
    # 4.9 CREATE FINAL MODELING TABLE
    # ========================================================

    feature_deposit = feature_full[
        [
            ID_COL
        ]
        + DEPOSIT_FEATURE_COLS
        + [
            TARGET_COL
        ]
    ].copy()

    # ========================================================
    # 4.10 VALIDATION
    # ========================================================

    final_customers = (
        feature_deposit[ID_COL]
        .nunique()
    )

    duplicate_count = int(
        feature_deposit[ID_COL]
        .duplicated()
        .sum()
    )

    missing_target_count = int(
        feature_deposit[TARGET_COL]
        .isna()
        .sum()
    )

    missing_feature_count = int(
        feature_deposit[
            DEPOSIT_FEATURE_COLS
        ]
        .isna()
        .sum()
        .sum()
    )

    print("\n===== VALIDATION =====")
    print("Expected customers :", expected_customers)
    print("Final customers    :", final_customers)
    print("Duplicate customers:", duplicate_count)
    print("Missing TARGET     :", missing_target_count)
    print("Missing features   :", missing_feature_count)
    print(
        "Customers without valid pre-cutoff deposit:",
        int(
            (
                feature_full[
                    "HAS_PRE_CUTOFF_DEPOSIT_SNAPSHOT"
                ] == 0
            ).sum()
        )
    )

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách hàng không khớp."
        )

    if duplicate_count > 0:
        raise ValueError(
            f"Seed {seed}: còn CUSTOMER_NUMBER bị trùng."
        )

    if missing_target_count > 0:
        raise ValueError(
            f"Seed {seed}: còn TARGET bị thiếu."
        )

    if missing_feature_count > 0:
        raise ValueError(
            f"Seed {seed}: còn feature deposit bị thiếu."
        )

    # ========================================================
    # 4.11 SUMMARY BY TARGET
    # ========================================================

    summary = (
        feature_full
        .groupby(TARGET_COL)
        .agg(
            CUSTOMERS=(
                ID_COL,
                "nunique"
            ),
            CUSTOMERS_WITH_SNAPSHOT=(
                "HAS_PRE_CUTOFF_DEPOSIT_SNAPSHOT",
                "sum"
            ),
            MEAN_COUNT_CA_ACCT=(
                "COUNT_CA_ACCT",
                "mean"
            ),
            MEAN_AVG_CA_BALANCE=(
                "AVG_CA_BALANCE",
                "mean"
            ),
            MEAN_COUNT_TD_ACCT=(
                "COUNT_TD_ACCT",
                "mean"
            ),
            MEAN_AVG_TD_BALANCE=(
                "AVG_TD_BALANCE",
                "mean"
            )
        )
        .reset_index()
    )

    summary[
        "CUSTOMERS_WITHOUT_SNAPSHOT"
    ] = (
        summary["CUSTOMERS"]
        - summary["CUSTOMERS_WITH_SNAPSHOT"]
    )

    summary["SEED"] = seed

    # ========================================================
    # 4.12 SORT OUTPUT
    # ========================================================

    feature_deposit = (
        feature_deposit
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    feature_full = (
        feature_full
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    # ========================================================
    # 4.13 SAVE FILES
    # ========================================================

    feature_deposit.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    feature_full.to_csv(
        audit_file,
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== SAVED =====")
    print(output_file)
    print(audit_file)
    print(summary_file)

    return {
        "SEED": seed,
        "INPUT_CUSTOMERS": expected_customers,
        "FINAL_CUSTOMERS": final_customers,
        "CUSTOMERS_WITH_SNAPSHOT": int(
            feature_full[
                "HAS_PRE_CUTOFF_DEPOSIT_SNAPSHOT"
            ].sum()
        ),
        "CUSTOMERS_WITHOUT_SNAPSHOT": int(
            (
                feature_full[
                    "HAS_PRE_CUTOFF_DEPOSIT_SNAPSHOT"
                ] == 0
            ).sum()
        ),
        "DUPLICATES": duplicate_count,
        "MISSING_FEATURES": missing_feature_count,
        "OUTPUT_FILE": output_file
    }


# ============================================================
# 5. RUN ALL FIVE SEEDS
# ============================================================

run_results = []

for seed in SEEDS:
    result = build_deposit_features(
        seed
    )
    run_results.append(
        result
    )


# ============================================================
# 6. SAVE COMBINED SUMMARY
# ============================================================

all_seed_summary = pd.DataFrame(
    run_results
)

all_seed_summary_file = os.path.join(
    OUTPUT_DIR,
    "feature_deposit_all_seeds_summary.csv"
)

all_seed_summary.to_csv(
    all_seed_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("ALL FIVE DEPOSIT FEATURE SEEDS COMPLETED")
print("=" * 75)

print(all_seed_summary)

print("\nSaved combined summary:")
print(all_seed_summary_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
INPUT_DIR : /content/drive/MyDrive/ELAPSPLACEBO/deposit_placebo_outputs
OUTPUT_DIR: /content/drive/MyDrive/ELAPSPLACEBO/deposit_features

PROCESSING DEPOSIT FEATURES — SEED 1
Input shape: (187290, 25)
Customers with inconsistent TARGET: 0
Expected customers: 150408
Valid deposit rows: 75779
Customers with at least one valid deposit snapshot: 38897

===== VALIDATION =====
Expected customers : 150408
Final customers    : 150408
Duplicate customers: 0
Missing TARGET     : 0
Missing features   : 0
Customers without valid pre-cutoff deposit: 111511

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/deposit_features/feature_deposit_4features_placebo_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_features/feature_deposit_audit_placebo_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/deposit_features/feature_deposit_summary_placebo_seed_1.csv

PROCESSING DE

LENDING → nối với TARGET

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

PLACEBO_DIR = os.path.join(
    BASE_DIR,
    "placebo_cutoff_outputs"
)

LENDING_FILE = os.path.join(
    BASE_DIR,
    "5.Data_Lending.xlsx"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "lending_placebo_outputs"
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"
START_DATE_COL = "CLIENT_CREATE_DATE"
CUTOFF_COL = "PLACEBO_CUTOFF_DATE"
LENDING_DATE_COL = "MONTH"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. CHECK FILES
# ============================================================

if not Path(BASE_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục:\n{BASE_DIR}"
    )

if not Path(PLACEBO_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục placebo:\n{PLACEBO_DIR}"
    )

if not Path(LENDING_FILE).exists():
    raise FileNotFoundError(
        f"Không tìm thấy file lending:\n{LENDING_FILE}"
    )

print("BASE_DIR:", BASE_DIR)
print("PLACEBO_DIR:", PLACEBO_DIR)
print("LENDING_FILE:", LENDING_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)


# ============================================================
# 4. LOAD RAW LENDING DATA ONCE
# ============================================================

lending = pd.read_excel(
    LENDING_FILE
)

lending.columns = (
    lending.columns
    .str.strip()
    .str.upper()
)

required_lending_cols = [
    ID_COL,
    LENDING_DATE_COL,
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT"
]

missing_lending_cols = [
    col for col in required_lending_cols
    if col not in lending.columns
]

if missing_lending_cols:
    raise ValueError(
        f"File lending thiếu cột: {missing_lending_cols}\n"
        f"Các cột hiện có: {lending.columns.tolist()}"
    )


# ============================================================
# 5. STANDARDIZE RAW LENDING DATA
# ============================================================

lending[ID_COL] = (
    lending[ID_COL]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
)

lending[LENDING_DATE_COL] = pd.to_datetime(
    lending[LENDING_DATE_COL],
    errors="coerce"
)

# Xử lý cột số có dấu phẩy
for col in [
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT",
    "COUNT_OF_LOAN"
]:
    if col in lending.columns:
        lending[col] = (
            lending[col]
            .astype("string")
            .str.replace(
                ",",
                "",
                regex=False
            )
            .str.strip()
        )

        lending[col] = pd.to_numeric(
            lending[col],
            errors="coerce"
        )

# Loại dòng thiếu CUSTOMER_NUMBER
lending = lending.dropna(
    subset=[ID_COL]
).copy()

print("\n===== RAW LENDING DATA =====")
print("Rows:", len(lending))
print("Customers:", lending[ID_COL].nunique())
print(
    "Missing MONTH:",
    int(lending[LENDING_DATE_COL].isna().sum())
)


# ============================================================
# 6. FUNCTION: PROCESS ONE SEED
# ============================================================

def process_lending_seed(seed: int) -> dict:

    print("\n" + "=" * 75)
    print(f"PROCESSING LENDING — SEED {seed}")
    print("=" * 75)

    # --------------------------------------------------------
    # 6.1 File paths
    # --------------------------------------------------------

    master_file = os.path.join(
        PLACEBO_DIR,
        f"step2_customer_target_filtered_placebo_seed_{seed}.csv"
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"step6_lending_before_placebo_cutoff_seed_{seed}.csv"
    )

    summary_file = os.path.join(
        OUTPUT_DIR,
        f"step6_lending_summary_seed_{seed}.csv"
    )

    customer_check_file = os.path.join(
        OUTPUT_DIR,
        f"step6_lending_customer_check_seed_{seed}.csv"
    )

    if not Path(master_file).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file master seed {seed}:\n"
            f"{master_file}"
        )

    # --------------------------------------------------------
    # 6.2 Load master placebo
    # --------------------------------------------------------

    master = pd.read_csv(
        master_file,
        low_memory=False
    )

    master.columns = (
        master.columns
        .str.strip()
        .str.upper()
    )

    required_master_cols = [
        ID_COL,
        TARGET_COL,
        START_DATE_COL,
        CUTOFF_COL
    ]

    missing_master_cols = [
        col for col in required_master_cols
        if col not in master.columns
    ]

    if missing_master_cols:
        raise ValueError(
            f"Seed {seed}: file master thiếu cột: "
            f"{missing_master_cols}\n"
            f"Các cột hiện có: {master.columns.tolist()}"
        )

    # --------------------------------------------------------
    # 6.3 Standardize master data types
    # --------------------------------------------------------

    master[ID_COL] = (
        master[ID_COL]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    master[START_DATE_COL] = pd.to_datetime(
        master[START_DATE_COL],
        errors="coerce"
    )

    master[CUTOFF_COL] = pd.to_datetime(
        master[CUTOFF_COL],
        errors="coerce"
    )

    master[TARGET_COL] = pd.to_numeric(
        master[TARGET_COL],
        errors="coerce"
    )

    for col in [
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "PLACEBO_SEED"
    ]:
        if col in master.columns:
            master[col] = pd.to_numeric(
                master[col],
                errors="coerce"
            )

    master = master.dropna(
        subset=[
            ID_COL,
            TARGET_COL
        ]
    ).copy()

    master[TARGET_COL] = (
        master[TARGET_COL]
        .astype(int)
    )

    invalid_target = ~master[
        TARGET_COL
    ].isin([0, 1])

    if invalid_target.any():
        raise ValueError(
            f"Seed {seed}: TARGET phải chỉ gồm 0 và 1."
        )

    # --------------------------------------------------------
    # 6.4 Check duplicate customers
    # --------------------------------------------------------

    duplicate_count = int(
        master[ID_COL]
        .duplicated()
        .sum()
    )

    print(
        "Duplicate customers in master:",
        duplicate_count
    )

    if duplicate_count > 0:

        duplicate_file = os.path.join(
            OUTPUT_DIR,
            f"duplicate_master_customers_seed_{seed}.csv"
        )

        master.loc[
            master[ID_COL].duplicated(
                keep=False
            )
        ].to_csv(
            duplicate_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: file master có CUSTOMER_NUMBER bị trùng."
        )

    # --------------------------------------------------------
    # 6.5 Check invalid cutoff
    # --------------------------------------------------------

    invalid_cutoff = master[
        master[CUTOFF_COL].isna()
        | master[START_DATE_COL].isna()
        | (
            master[CUTOFF_COL]
            < master[START_DATE_COL]
        )
    ].copy()

    print(
        "Customers with invalid cutoff:",
        len(invalid_cutoff)
    )

    if not invalid_cutoff.empty:

        invalid_file = os.path.join(
            OUTPUT_DIR,
            f"invalid_cutoff_seed_{seed}.csv"
        )

        invalid_cutoff.to_csv(
            invalid_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: có khách hàng thiếu cutoff "
            "hoặc cutoff trước ngày bắt đầu quan hệ."
        )

    # --------------------------------------------------------
    # 6.6 Remove overlapping lending columns from master
    # --------------------------------------------------------

    overlapping_cols = [
        col for col in lending.columns
        if col in master.columns
        and col != ID_COL
    ]

    if overlapping_cols:
        print(
            "Các cột lending trùng trong master sẽ bị bỏ trước merge:",
            overlapping_cols
        )

        master_for_merge = master.drop(
            columns=overlapping_cols
        ).copy()
    else:
        master_for_merge = master.copy()

    # --------------------------------------------------------
    # 6.7 LEFT JOIN
    # --------------------------------------------------------

    df = master_for_merge.merge(
        lending,
        on=ID_COL,
        how="left",
        validate="one_to_many"
    )

    expected_customers = (
        master_for_merge[ID_COL]
        .nunique()
    )

    print("\n===== BEFORE FILTERING =====")
    print("Rows:", len(df))
    print(
        "Customers:",
        df[ID_COL].nunique()
    )
    print(
        "Expected customers:",
        expected_customers
    )
    print(
        "Adopters:",
        df.loc[
            df[TARGET_COL] == 1,
            ID_COL
        ].nunique()
    )
    print(
        "Non-adopters:",
        df.loc[
            df[TARGET_COL] == 0,
            ID_COL
        ].nunique()
    )

    # --------------------------------------------------------
    # 6.8 Filter lending records before cutoff
    #
    # Giữ:
    # - khách không có lending;
    # - record lending trước PLACEBO_CUTOFF_DATE.
    # --------------------------------------------------------

    valid_lending_mask = (
        df[LENDING_DATE_COL].isna()
        |
        (
            df[CUTOFF_COL].notna()
            & (
                df[LENDING_DATE_COL]
                < df[CUTOFF_COL]
            )
        )
    )

    df_filtered = df.loc[
        valid_lending_mask
    ].copy()

    # --------------------------------------------------------
    # 6.9 Detect customers lost after filtering
    # --------------------------------------------------------

    customers_before = set(
        master_for_merge[
            ID_COL
        ].unique()
    )

    customers_after_filter = set(
        df_filtered[
            ID_COL
        ].unique()
    )

    lost_customers = (
        customers_before
        - customers_after_filter
    )

    print("\n===== AFTER RAW FILTERING =====")
    print("Rows:", len(df_filtered))
    print(
        "Customers:",
        df_filtered[ID_COL].nunique()
    )
    print(
        "Customers temporarily lost:",
        len(lost_customers)
    )

    # --------------------------------------------------------
    # 6.10 Restore lost customers
    # --------------------------------------------------------

    if lost_customers:

        missing_rows = master_for_merge[
            master_for_merge[ID_COL].isin(
                lost_customers
            )
        ].copy()

        lending_only_cols = [
            col for col in lending.columns
            if col != ID_COL
        ]

        for col in lending_only_cols:
            if col not in missing_rows.columns:
                missing_rows[col] = pd.NA

        missing_rows = missing_rows[
            df_filtered.columns
        ]

        df_filtered = pd.concat(
            [
                df_filtered,
                missing_rows
            ],
            ignore_index=True
        )

    # --------------------------------------------------------
    # 6.11 Final customer consistency
    # --------------------------------------------------------

    final_customers = (
        df_filtered[ID_COL]
        .nunique()
    )

    print(
        "Customers after restoration:",
        final_customers
    )

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách cuối không khớp. "
            f"Expected={expected_customers}, "
            f"Final={final_customers}"
        )

    # --------------------------------------------------------
    # 6.12 Leakage check
    # --------------------------------------------------------

    leak = df_filtered[
        df_filtered[
            LENDING_DATE_COL
        ].notna()
        & df_filtered[
            CUTOFF_COL
        ].notna()
        & (
            df_filtered[LENDING_DATE_COL]
            >= df_filtered[CUTOFF_COL]
        )
    ].copy()

    print(
        "Remaining leakage rows:",
        len(leak)
    )

    if not leak.empty:

        leak_file = os.path.join(
            OUTPUT_DIR,
            f"lending_leakage_seed_{seed}.csv"
        )

        leak.to_csv(
            leak_file,
            index=False,
            encoding="utf-8-sig"
        )

        raise ValueError(
            f"Seed {seed}: vẫn còn lending tại hoặc sau cutoff."
        )

    # --------------------------------------------------------
    # 6.13 Count lending rows per customer
    # --------------------------------------------------------

    lending_count = (
        df_filtered
        .groupby(ID_COL)[LENDING_DATE_COL]
        .count()
        .rename(
            "PRE_CUTOFF_LENDING_COUNT"
        )
        .reset_index()
    )

    no_lending_count = int(
        (
            lending_count[
                "PRE_CUTOFF_LENDING_COUNT"
            ] == 0
        ).sum()
    )

    print(
        "Customers with no pre-cutoff lending:",
        no_lending_count
    )

    # --------------------------------------------------------
    # 6.14 Customer-level check
    # --------------------------------------------------------

    customer_check_cols = [
        ID_COL,
        TARGET_COL,
        START_DATE_COL,
        CUTOFF_COL
    ]

    for optional_col in [
        "CUTOFF_TYPE",
        "PLACEBO_SEED",
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "SAMPLING_STATUS"
    ]:
        if optional_col in master_for_merge.columns:
            customer_check_cols.append(
                optional_col
            )

    customer_check = (
        master_for_merge[
            customer_check_cols
        ]
        .merge(
            lending_count,
            on=ID_COL,
            how="left",
            validate="one_to_one"
        )
    )

    customer_check[
        "PRE_CUTOFF_LENDING_COUNT"
    ] = (
        customer_check[
            "PRE_CUTOFF_LENDING_COUNT"
        ]
        .fillna(0)
        .astype(int)
    )

    customer_check[
        "NO_PRE_CUTOFF_LENDING"
    ] = (
        customer_check[
            "PRE_CUTOFF_LENDING_COUNT"
        ] == 0
    )

    # --------------------------------------------------------
    # 6.15 Summary by target
    # --------------------------------------------------------

    summary_agg = {
        "CUSTOMERS": (
            ID_COL,
            "nunique"
        ),
        "LENDING_ROWS": (
            LENDING_DATE_COL,
            "count"
        ),
        "EARLIEST_LENDING": (
            LENDING_DATE_COL,
            "min"
        ),
        "LATEST_LENDING": (
            LENDING_DATE_COL,
            "max"
        )
    }

    if "ASSIGNED_CUTOFF_DAYS" in df_filtered.columns:
        summary_agg[
            "MEAN_CUTOFF_DAYS"
        ] = (
            "ASSIGNED_CUTOFF_DAYS",
            "mean"
        )

        summary_agg[
            "MEDIAN_CUTOFF_DAYS"
        ] = (
            "ASSIGNED_CUTOFF_DAYS",
            "median"
        )

    summary = (
        df_filtered
        .groupby(TARGET_COL)
        .agg(
            **summary_agg
        )
        .reset_index()
    )

    summary["SEED"] = seed

    # --------------------------------------------------------
    # 6.16 Sort
    # --------------------------------------------------------

    df_filtered = (
        df_filtered
        .sort_values(
            by=[
                ID_COL,
                LENDING_DATE_COL
            ],
            na_position="last"
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 6.17 Save outputs
    # --------------------------------------------------------

    df_filtered.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    customer_check.to_csv(
        customer_check_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== SAVED =====")
    print(output_file)
    print(summary_file)
    print(customer_check_file)

    return {
        "SEED": seed,
        "INPUT_CUSTOMERS": expected_customers,
        "FINAL_CUSTOMERS": final_customers,
        "OUTPUT_ROWS": len(df_filtered),
        "NO_LENDING_CUSTOMERS": no_lending_count,
        "LEAKAGE_ROWS": len(leak),
        "OUTPUT_FILE": output_file
    }


# ============================================================
# 7. RUN ALL FIVE SEEDS
# ============================================================

run_results = []

for seed in SEEDS:

    result = process_lending_seed(
        seed
    )

    run_results.append(
        result
    )


# ============================================================
# 8. COMBINE SEED SUMMARIES
# ============================================================

run_summary = pd.DataFrame(
    run_results
)

run_summary_file = os.path.join(
    OUTPUT_DIR,
    "step6_lending_all_seeds_summary.csv"
)

run_summary.to_csv(
    run_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("ALL FIVE LENDING SEEDS COMPLETED")
print("=" * 75)

print(run_summary)

print("\nSaved combined summary:")
print(run_summary_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/ELAPSPLACEBO
PLACEBO_DIR: /content/drive/MyDrive/ELAPSPLACEBO/placebo_cutoff_outputs
LENDING_FILE: /content/drive/MyDrive/ELAPSPLACEBO/5.Data_Lending.xlsx
OUTPUT_DIR: /content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)



===== RAW LENDING DATA =====
Rows: 576431
Customers: 102014
Missing MONTH: 0

PROCESSING LENDING — SEED 1
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 346834
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 133827
Customers: 115583
Customers temporarily lost: 34825


/tmp/ipykernel_54151/2350237105.py:522: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff lending: 142040

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_before_placebo_cutoff_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_summary_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_customer_check_seed_1.csv

PROCESSING LENDING — SEED 2
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 346834
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 133900
Customers: 115661
Customers temporarily lost: 34747


...

Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff lending: 142040

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_before_placebo_cutoff_seed_3.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_summary_seed_3.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_customer_check_seed_3.csv

PROCESSING LENDING — SEED 4
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 346834
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 134011
Customers: 115702
Customers temporarily lost: 34706


/tmp/ipykernel_54151/2350237105.py:522: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff lending: 141921

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_before_placebo_cutoff_seed_4.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_summary_seed_4.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_customer_check_seed_4.csv

PROCESSING LENDING — SEED 5
Duplicate customers in master: 0
Customers with invalid cutoff: 0

===== BEFORE FILTERING =====
Rows: 346834
Customers: 150408
Expected customers: 150408
Adopters: 42323
Non-adopters: 108085

===== AFTER RAW FILTERING =====
Rows: 133899
Customers: 115648
Customers temporarily lost: 34760


/tmp/ipykernel_54151/2350237105.py:522: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_filtered = pd.concat(


Customers after restoration: 150408
Remaining leakage rows: 0
Customers with no pre-cutoff lending: 141975

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_before_placebo_cutoff_seed_5.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_summary_seed_5.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs/step6_lending_customer_check_seed_5.csv

ALL FIVE LENDING SEEDS COMPLETED
   SEED  INPUT_CUSTOMERS  FINAL_CUSTOMERS  OUTPUT_ROWS  NO_LENDING_CUSTOMERS  \
0     1           150408           150408       168652                142040   
1     2           150408           150408       168647                141962   
2     3           150408           150408       168612                142040   
3     4           150408           150408       168717                141921   
4     5           150408           150408       168659                141975   

   LEAKAGE_ROWS                                        OUTP

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

INPUT_DIR = os.path.join(
    BASE_DIR,
    "lending_placebo_outputs"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "lending_features"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "TARGET"
DATE_COL = "MONTH"

LENDING_FEATURE_COLS = [
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT"
]


# ============================================================
# 3. CHECK INPUT DIRECTORY
# ============================================================

if not Path(INPUT_DIR).exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục đầu vào:\n{INPUT_DIR}"
    )

print("INPUT_DIR :", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


# ============================================================
# 4. PROCESS ONE SEED
# ============================================================

def build_lending_features(seed: int) -> dict:

    print("\n" + "=" * 75)
    print(f"PROCESSING LENDING FEATURES — SEED {seed}")
    print("=" * 75)

    input_file = os.path.join(
        INPUT_DIR,
        f"step6_lending_before_placebo_cutoff_seed_{seed}.csv"
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"feature_lending_placebo_seed_{seed}.csv"
    )

    audit_file = os.path.join(
        OUTPUT_DIR,
        f"feature_lending_audit_placebo_seed_{seed}.csv"
    )

    summary_file = os.path.join(
        OUTPUT_DIR,
        f"feature_lending_summary_placebo_seed_{seed}.csv"
    )

    if not Path(input_file).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file seed {seed}:\n{input_file}"
        )

    # ========================================================
    # 4.1 LOAD DATA
    # ========================================================

    df = pd.read_csv(
        input_file,
        low_memory=False
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )

    print("Input shape:", df.shape)

    # ========================================================
    # 4.2 CHECK REQUIRED COLUMNS
    # ========================================================

    required_cols = [
        ID_COL,
        TARGET_COL,
        DATE_COL,
        "COUNT_OF_LOAN",
        "AVG_LOAN_AMOUNT"
    ]

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Seed {seed} thiếu các cột: {missing_cols}\n"
            f"Các cột hiện có: {df.columns.tolist()}"
        )

    # ========================================================
    # 4.3 STANDARDIZE DATA TYPES
    # ========================================================

    df[ID_COL] = (
        df[ID_COL]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce"
    )

    df[DATE_COL] = pd.to_datetime(
        df[DATE_COL],
        errors="coerce"
    )

    # Làm sạch số có dấu phẩy, khoảng trắng hoặc chuỗi rỗng
    for col in LENDING_FEATURE_COLS:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.strip()
        )

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    df = df.dropna(
        subset=[ID_COL]
    ).copy()

    # ========================================================
    # 4.4 CHECK TARGET CONSISTENCY
    # ========================================================

    target_nunique = (
        df.groupby(ID_COL)[TARGET_COL]
        .nunique(dropna=True)
    )

    inconsistent_target = target_nunique[
        target_nunique > 1
    ]

    print(
        "Customers with inconsistent TARGET:",
        len(inconsistent_target)
    )

    if not inconsistent_target.empty:
        raise ValueError(
            f"Seed {seed}: có khách hàng mang nhiều TARGET."
        )

    missing_target_customers = (
        df.groupby(ID_COL)[TARGET_COL]
        .apply(lambda x: x.isna().all())
    )

    if missing_target_customers.any():
        raise ValueError(
            f"Seed {seed}: có "
            f"{int(missing_target_customers.sum())} khách thiếu TARGET."
        )

    # ========================================================
    # 4.5 CREATE CUSTOMER BASE
    # ========================================================

    base_cols = [
        ID_COL,
        TARGET_COL
    ]

    optional_audit_cols = [
        "CLIENT_CREATE_DATE",
        "PLACEBO_CUTOFF_DATE",
        "CUTOFF_TYPE",
        "PLACEBO_SEED",
        "ASSIGNED_CUTOFF_DAYS",
        "ACTUAL_D_STAR",
        "AVAILABLE_DAYS",
        "SAMPLING_STATUS"
    ]

    for col in optional_audit_cols:
        if col in df.columns:
            base_cols.append(col)

    customer_base = (
        df[base_cols]
        .sort_values(ID_COL)
        .drop_duplicates(
            subset=[ID_COL],
            keep="first"
        )
        .copy()
    )

    expected_customers = (
        customer_base[ID_COL]
        .nunique()
    )

    print(
        "Expected customers:",
        expected_customers
    )

    # ========================================================
    # 4.6 KEEP ONLY VALID LENDING SNAPSHOTS
    #
    # Dòng MONTH = NaT chỉ dùng để giữ khách,
    # không được dùng để chọn snapshot gần nhất.
    # ========================================================

    valid_snapshots = df.loc[
        df[DATE_COL].notna()
    ].copy()

    print(
        "Valid lending rows:",
        len(valid_snapshots)
    )

    print(
        "Customers with at least one valid lending snapshot:",
        valid_snapshots[ID_COL].nunique()
    )

    # ========================================================
    # 4.7 TAKE LATEST VALID SNAPSHOT PER CUSTOMER
    # ========================================================

    if not valid_snapshots.empty:

        valid_snapshots = (
            valid_snapshots
            .sort_values(
                by=[
                    ID_COL,
                    DATE_COL
                ]
            )
        )

        latest_snapshot = (
            valid_snapshots
            .groupby(
                ID_COL,
                as_index=False,
                sort=False
            )
            .tail(1)
            .copy()
        )

        latest_features = latest_snapshot[
            [
                ID_COL,
                DATE_COL,
                "COUNT_OF_LOAN",
                "AVG_LOAN_AMOUNT"
            ]
        ].copy()

    else:
        latest_features = pd.DataFrame(
            columns=[
                ID_COL,
                DATE_COL,
                "COUNT_OF_LOAN",
                "AVG_LOAN_AMOUNT"
            ]
        )

    # ========================================================
    # 4.8 MERGE WITH ALL CUSTOMERS
    # ========================================================

    feature_full = customer_base.merge(
        latest_features,
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )

    feature_full[
        "HAS_PRE_CUTOFF_LENDING_SNAPSHOT"
    ] = (
        feature_full[DATE_COL].notna()
    ).astype(int)

    # Chỉ điền 0 cho feature khoản vay
    feature_full[
        [
            "COUNT_OF_LOAN",
            "AVG_LOAN_AMOUNT"
        ]
    ] = (
        feature_full[
            [
                "COUNT_OF_LOAN",
                "AVG_LOAN_AMOUNT"
            ]
        ]
        .fillna(0)
    )

    # ========================================================
    # 4.9 CREATE TOTAL_LOAN_AMOUNT
    #
    # Chỉ hợp lý nếu AVG_LOAN_AMOUNT thực sự là giá trị
    # trung bình trên mỗi khoản vay tại snapshot.
    # ========================================================

    feature_full["TOTAL_LOAN_AMOUNT"] = (
        feature_full["COUNT_OF_LOAN"]
        * feature_full["AVG_LOAN_AMOUNT"]
    )

    # ========================================================
    # 4.10 FINAL MODELING TABLE
    # ========================================================

    feature_lending = feature_full[
        [
            ID_COL,
            "COUNT_OF_LOAN",
            "AVG_LOAN_AMOUNT",
            "TOTAL_LOAN_AMOUNT",
            TARGET_COL
        ]
    ].copy()

    # ========================================================
    # 4.11 VALIDATION
    # ========================================================

    final_customers = (
        feature_lending[ID_COL]
        .nunique()
    )

    duplicate_count = int(
        feature_lending[ID_COL]
        .duplicated()
        .sum()
    )

    missing_target_count = int(
        feature_lending[TARGET_COL]
        .isna()
        .sum()
    )

    missing_feature_count = int(
        feature_lending[
            [
                "COUNT_OF_LOAN",
                "AVG_LOAN_AMOUNT",
                "TOTAL_LOAN_AMOUNT"
            ]
        ]
        .isna()
        .sum()
        .sum()
    )

    print("\n===== VALIDATION =====")
    print("Expected customers :", expected_customers)
    print("Final customers    :", final_customers)
    print("Duplicate customers:", duplicate_count)
    print("Missing TARGET     :", missing_target_count)
    print("Missing features   :", missing_feature_count)
    print(
        "Customers without valid pre-cutoff lending:",
        int(
            (
                feature_full[
                    "HAS_PRE_CUTOFF_LENDING_SNAPSHOT"
                ] == 0
            ).sum()
        )
    )

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách hàng không khớp."
        )

    if duplicate_count > 0:
        raise ValueError(
            f"Seed {seed}: còn CUSTOMER_NUMBER bị trùng."
        )

    if missing_target_count > 0:
        raise ValueError(
            f"Seed {seed}: còn TARGET bị thiếu."
        )

    if missing_feature_count > 0:
        raise ValueError(
            f"Seed {seed}: còn feature lending bị thiếu."
        )

    # ========================================================
    # 4.12 SUMMARY BY TARGET
    # ========================================================

    summary = (
        feature_full
        .groupby(TARGET_COL)
        .agg(
            CUSTOMERS=(
                ID_COL,
                "nunique"
            ),
            CUSTOMERS_WITH_SNAPSHOT=(
                "HAS_PRE_CUTOFF_LENDING_SNAPSHOT",
                "sum"
            ),
            MEAN_COUNT_OF_LOAN=(
                "COUNT_OF_LOAN",
                "mean"
            ),
            MEAN_AVG_LOAN_AMOUNT=(
                "AVG_LOAN_AMOUNT",
                "mean"
            ),
            MEAN_TOTAL_LOAN_AMOUNT=(
                "TOTAL_LOAN_AMOUNT",
                "mean"
            )
        )
        .reset_index()
    )

    summary[
        "CUSTOMERS_WITHOUT_SNAPSHOT"
    ] = (
        summary["CUSTOMERS"]
        - summary["CUSTOMERS_WITH_SNAPSHOT"]
    )

    summary["SEED"] = seed

    # ========================================================
    # 4.13 SORT OUTPUT
    # ========================================================

    feature_lending = (
        feature_lending
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    feature_full = (
        feature_full
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    # ========================================================
    # 4.14 SAVE
    # ========================================================

    feature_lending.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    feature_full.to_csv(
        audit_file,
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== SAVED =====")
    print(output_file)
    print(audit_file)
    print(summary_file)

    return {
        "SEED": seed,
        "INPUT_CUSTOMERS": expected_customers,
        "FINAL_CUSTOMERS": final_customers,
        "CUSTOMERS_WITH_SNAPSHOT": int(
            feature_full[
                "HAS_PRE_CUTOFF_LENDING_SNAPSHOT"
            ].sum()
        ),
        "CUSTOMERS_WITHOUT_SNAPSHOT": int(
            (
                feature_full[
                    "HAS_PRE_CUTOFF_LENDING_SNAPSHOT"
                ] == 0
            ).sum()
        ),
        "DUPLICATES": duplicate_count,
        "MISSING_FEATURES": missing_feature_count,
        "OUTPUT_FILE": output_file
    }


# ============================================================
# 5. RUN ALL FIVE SEEDS
# ============================================================

run_results = []

for seed in SEEDS:

    result = build_lending_features(
        seed
    )

    run_results.append(
        result
    )


# ============================================================
# 6. SAVE ALL-SEED SUMMARY
# ============================================================

all_seed_summary = pd.DataFrame(
    run_results
)

all_seed_summary_file = os.path.join(
    OUTPUT_DIR,
    "feature_lending_all_seeds_summary.csv"
)

all_seed_summary.to_csv(
    all_seed_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("ALL FIVE LENDING FEATURE SEEDS COMPLETED")
print("=" * 75)

print(all_seed_summary)

print("\nSaved combined summary:")
print(all_seed_summary_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
INPUT_DIR : /content/drive/MyDrive/ELAPSPLACEBO/lending_placebo_outputs
OUTPUT_DIR: /content/drive/MyDrive/ELAPSPLACEBO/lending_features

PROCESSING LENDING FEATURES — SEED 1
Input shape: (168652, 23)
Customers with inconsistent TARGET: 0
Expected customers: 150408
Valid lending rows: 26612
Customers with at least one valid lending snapshot: 8368

===== VALIDATION =====
Expected customers : 150408
Final customers    : 150408
Duplicate customers: 0
Missing TARGET     : 0
Missing features   : 0
Customers without valid pre-cutoff lending: 142040

===== SAVED =====
/content/drive/MyDrive/ELAPSPLACEBO/lending_features/feature_lending_placebo_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_features/feature_lending_audit_placebo_seed_1.csv
/content/drive/MyDrive/ELAPSPLACEBO/lending_features/feature_lending_summary_placebo_seed_1.csv

PROCESSING LENDING FEATU

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

CUSTOMER_DIR = os.path.join(
    BASE_DIR,
    "placebo_cutoff_outputs"
)

ACTIVITY_DIR = os.path.join(
    BASE_DIR,
    "activity_features"
)

TRANSACTION_DIR = os.path.join(
    BASE_DIR,
    "transaction_features"
)

DEPOSIT_DIR = os.path.join(
    BASE_DIR,
    "deposit_features"
)

LENDING_DIR = os.path.join(
    BASE_DIR,
    "lending_features"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "final_placebo_datasets"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
SOURCE_TARGET_COL = "TARGET"
FINAL_TARGET_COL = "COUNT_CREDITCARD"


# ============================================================
# 3. FILE PATTERNS
# ============================================================

CUSTOMER_PATTERN = (
    "step2_customer_target_filtered_placebo_seed_{seed}.csv"
)

ACTIVITY_PATTERN = (
    "feature_activity_2features_placebo_seed_{seed}.csv"
)

TRANSACTION_PATTERN = (
    "feature_transaction_mode_placebo_seed_{seed}.csv"
)

DEPOSIT_PATTERN = (
    "feature_deposit_4features_placebo_seed_{seed}.csv"
)

LENDING_PATTERN = (
    "feature_lending_placebo_seed_{seed}.csv"
)


# ============================================================
# 4. EXACT BASELINE FEATURE SET
# ============================================================

CUSTOMER_FEATURES = [
    "CLIENT_SEX",
    "EB_REGISTER_CHANNEL",
    "SMS",
    "VERIFY_METHOD"
]

ACTIVITY_FEATURES = [
    "LOGIN_PER_ACTIVE_DAY",
    "INTEREST_RATE_RATIO"
]

TRANSACTION_FEATURES = [
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
    "TRANS_LV1_MODE_CODE",
    "TRANS_LV2_MODE_CODE"
]

DEPOSIT_FEATURES = [
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE"
]

LENDING_FEATURES = [
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT"
]

FINAL_COLUMNS = [
    ID_COL,
    "CLIENT_SEX",
    "EB_REGISTER_CHANNEL",
    "SMS",
    "VERIFY_METHOD",
    FINAL_TARGET_COL,
    "AGE",
    "LOGIN_PER_ACTIVE_DAY",
    "INTEREST_RATE_RATIO",
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
    "TRANS_LV1_MODE_CODE",
    "TRANS_LV2_MODE_CODE",
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE",
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT"
]

NUMERIC_FEATURES = [
    "AGE",
    "LOGIN_PER_ACTIVE_DAY",
    "INTEREST_RATE_RATIO",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
    "TRANS_LV1_MODE_CODE",
    "TRANS_LV2_MODE_CODE",
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE",
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT"
]

CATEGORICAL_FEATURES = [
    "CLIENT_SEX",
    "EB_REGISTER_CHANNEL",
    "SMS",
    "VERIFY_METHOD",
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE"
]


# ============================================================
# 5. HELPER: STANDARDIZE CUSTOMER NUMBER
# ============================================================

def standardize_customer_number(
    df: pd.DataFrame
) -> pd.DataFrame:

    result = df.copy()

    result[ID_COL] = (
        result[ID_COL]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    return result


# ============================================================
# 6. HELPER: LOAD CUSTOMER-LEVEL FILE
# ============================================================

def load_customer_level_file(
    file_path: str,
    table_name: str
) -> pd.DataFrame:

    if not Path(file_path).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file {table_name}:\n"
            f"{file_path}"
        )

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )

    if ID_COL not in df.columns:
        raise ValueError(
            f"{table_name} thiếu cột {ID_COL}.\n"
            f"Các cột hiện có:\n{df.columns.tolist()}"
        )

    df = standardize_customer_number(
        df
    )

    df = df.dropna(
        subset=[ID_COL]
    ).copy()

    duplicate_count = int(
        df[ID_COL]
        .duplicated()
        .sum()
    )

    print(
        f"{table_name} shape:",
        df.shape
    )

    print(
        f"{table_name} duplicates:",
        duplicate_count
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{table_name} có {duplicate_count} "
            "CUSTOMER_NUMBER bị trùng."
        )

    return df


# ============================================================
# 7. HELPER: REQUIRE COLUMNS
# ============================================================

def require_columns(
    df: pd.DataFrame,
    required_cols: list,
    table_name: str
):

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"{table_name} thiếu các cột:\n"
            f"{missing_cols}\n\n"
            f"Các cột hiện có:\n"
            f"{df.columns.tolist()}"
        )


# ============================================================
# 8. PREPARE CUSTOMER TABLE
# ============================================================

def prepare_customer_table(
    customer: pd.DataFrame,
    seed: int
) -> pd.DataFrame:

    require_columns(
        customer,
        [
            ID_COL,
            SOURCE_TARGET_COL,
            "CLIENT_SEX",
            "EB_REGISTER_CHANNEL",
            "SMS",
            "VERIFY_METHOD"
        ],
        f"customer seed {seed}"
    )

    # --------------------------------------------------------
    # Tạo AGE = 2019 - năm sinh
    # --------------------------------------------------------

    if "AGE" not in customer.columns:

        if "DATE_OF_BIRTH" not in customer.columns:
            raise ValueError(
                f"Customer seed {seed} thiếu cả AGE "
                "và DATE_OF_BIRTH."
            )

        date_of_birth = pd.to_datetime(
            customer["DATE_OF_BIRTH"],
            errors="coerce"
        )

        customer["AGE"] = (
            2019
            - date_of_birth.dt.year
        )

    customer["AGE"] = pd.to_numeric(
        customer["AGE"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # TARGET -> COUNT_CREDITCARD
    # --------------------------------------------------------

    customer[FINAL_TARGET_COL] = pd.to_numeric(
        customer[SOURCE_TARGET_COL],
        errors="coerce"
    )

    invalid_target = (
        customer[FINAL_TARGET_COL].isna()
        | ~customer[FINAL_TARGET_COL].isin([0, 1])
    )

    if invalid_target.any():
        raise ValueError(
            f"Seed {seed}: TARGET phải chỉ gồm 0 và 1."
        )

    customer[FINAL_TARGET_COL] = (
        customer[FINAL_TARGET_COL]
        .astype(int)
    )

    # --------------------------------------------------------
    # Chỉ giữ đúng thuộc tính customer của baseline
    # Không giữ DATE_OF_BIRTH, STAFF_VIB, IB_REGISTER_DATE...
    # --------------------------------------------------------

    customer_final = customer[
        [
            ID_COL,
            "CLIENT_SEX",
            "EB_REGISTER_CHANNEL",
            "SMS",
            "VERIFY_METHOD",
            FINAL_TARGET_COL,
            "AGE"
        ]
    ].copy()

    return customer_final


# ============================================================
# 9. PREPARE FEATURE TABLE
# ============================================================

def prepare_feature_table(
    df: pd.DataFrame,
    expected_features: list,
    table_name: str
) -> pd.DataFrame:

    require_columns(
        df,
        [ID_COL] + expected_features,
        table_name
    )

    result = df[
        [ID_COL] + expected_features
    ].copy()

    # Không ép các biến MODE sang numeric
    for col in expected_features:

        if col in [
            "TRANS_LV1_MODE",
            "TRANS_LV2_MODE"
        ]:

            result[col] = (
                result[col]
                .astype("string")
                .str.strip()
                .str.upper()
            )

            result[col] = (
                result[col]
                .replace({
                    "": pd.NA,
                    "NAN": pd.NA,
                    "<NA>": pd.NA,
                    "NONE": pd.NA,
                    "NULL": pd.NA
                })
                .fillna("NO_TRANSACTION")
            )

        else:

            result[col] = (
                result[col]
                .astype("string")
                .str.replace(
                    ",",
                    "",
                    regex=False
                )
                .str.strip()
            )

            result[col] = pd.to_numeric(
                result[col],
                errors="coerce"
            )

    return result


# ============================================================
# 10. BUILD ONE FINAL DATASET
# ============================================================

def build_final_dataset(
    seed: int
) -> dict:

    print("\n" + "=" * 80)
    print(f"BUILDING FINAL PLACEBO DATASET — SEED {seed}")
    print("=" * 80)

    # --------------------------------------------------------
    # 10.1 File paths
    # --------------------------------------------------------

    customer_file = os.path.join(
        CUSTOMER_DIR,
        CUSTOMER_PATTERN.format(
            seed=seed
        )
    )

    activity_file = os.path.join(
        ACTIVITY_DIR,
        ACTIVITY_PATTERN.format(
            seed=seed
        )
    )

    transaction_file = os.path.join(
        TRANSACTION_DIR,
        TRANSACTION_PATTERN.format(
            seed=seed
        )
    )

    deposit_file = os.path.join(
        DEPOSIT_DIR,
        DEPOSIT_PATTERN.format(
            seed=seed
        )
    )

    lending_file = os.path.join(
        LENDING_DIR,
        LENDING_PATTERN.format(
            seed=seed
        )
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"final_dataset_placebo_seed_{seed}.csv"
    )

    no_auto_job_file = os.path.join(
        OUTPUT_DIR,
        f"final_dataset_placebo_seed_{seed}_no_auto_job.csv"
    )

    missing_check_file = os.path.join(
        OUTPUT_DIR,
        f"final_dataset_missing_check_seed_{seed}.csv"
    )

    # --------------------------------------------------------
    # 10.2 Load files
    # --------------------------------------------------------

    customer_raw = load_customer_level_file(
        customer_file,
        f"customer seed {seed}"
    )

    activity_raw = load_customer_level_file(
        activity_file,
        f"activity seed {seed}"
    )

    transaction_raw = load_customer_level_file(
        transaction_file,
        f"transaction seed {seed}"
    )

    deposit_raw = load_customer_level_file(
        deposit_file,
        f"deposit seed {seed}"
    )

    lending_raw = load_customer_level_file(
        lending_file,
        f"lending seed {seed}"
    )

    # --------------------------------------------------------
    # 10.3 Prepare tables
    # --------------------------------------------------------

    customer = prepare_customer_table(
        customer_raw,
        seed
    )

    activity = prepare_feature_table(
        activity_raw,
        ACTIVITY_FEATURES,
        f"activity seed {seed}"
    )

    transaction = prepare_feature_table(
        transaction_raw,
        TRANSACTION_FEATURES,
        f"transaction seed {seed}"
    )

    deposit = prepare_feature_table(
        deposit_raw,
        DEPOSIT_FEATURES,
        f"deposit seed {seed}"
    )

    lending = prepare_feature_table(
        lending_raw,
        LENDING_FEATURES,
        f"lending seed {seed}"
    )

    expected_customers = (
        customer[ID_COL]
        .nunique()
    )

    print(
        "Expected customers:",
        expected_customers
    )

    # --------------------------------------------------------
    # 10.4 Merge
    # --------------------------------------------------------

    final_df = customer.copy()

    feature_tables = [
        ("activity", activity),
        ("transaction", transaction),
        ("deposit", deposit),
        ("lending", lending)
    ]

    for table_name, feature_df in feature_tables:

        rows_before = len(
            final_df
        )

        final_df = final_df.merge(
            feature_df,
            on=ID_COL,
            how="left",
            validate="one_to_one"
        )

        print(
            f"After merging {table_name}:",
            final_df.shape
        )

        if len(final_df) != rows_before:
            raise ValueError(
                f"Seed {seed}: số dòng thay đổi "
                f"sau merge {table_name}."
            )

    # --------------------------------------------------------
    # 10.5 Missing report before fill
    # --------------------------------------------------------

    missing_check = (
        final_df
        .isna()
        .sum()
        .rename(
            "MISSING_BEFORE_FILL"
        )
        .reset_index()
        .rename(
            columns={
                "index": "COLUMN"
            }
        )
    )

    # --------------------------------------------------------
    # 10.6 Fill only model features
    # --------------------------------------------------------

    for col in NUMERIC_FEATURES:

        if col in final_df.columns:

            final_df[col] = pd.to_numeric(
                final_df[col],
                errors="coerce"
            )

            final_df[col] = (
                final_df[col]
                .fillna(0)
            )

    for col in CATEGORICAL_FEATURES:

        if col in final_df.columns:

            final_df[col] = (
                final_df[col]
                .astype("string")
                .str.strip()
                .str.upper()
            )

            final_df[col] = (
                final_df[col]
                .replace({
                    "": pd.NA,
                    "NAN": pd.NA,
                    "<NA>": pd.NA,
                    "NONE": pd.NA,
                    "NULL": pd.NA
                })
                .fillna("UNKNOWN")
            )

    # --------------------------------------------------------
    # 10.7 Exact baseline schema
    # --------------------------------------------------------

    require_columns(
        final_df,
        FINAL_COLUMNS,
        f"final dataset seed {seed}"
    )

    final_df = final_df[
        FINAL_COLUMNS
    ].copy()

    # --------------------------------------------------------
    # 10.8 Validation
    # --------------------------------------------------------

    final_customers = (
        final_df[ID_COL]
        .nunique()
    )

    duplicate_count = int(
        final_df[ID_COL]
        .duplicated()
        .sum()
    )

    missing_count = int(
        final_df
        .isna()
        .sum()
        .sum()
    )

    print("\n===== FINAL VALIDATION =====")
    print("Shape:", final_df.shape)
    print(
        "Expected customers:",
        expected_customers
    )
    print(
        "Final customers:",
        final_customers
    )
    print(
        "Duplicate customers:",
        duplicate_count
    )
    print(
        "Total missing values:",
        missing_count
    )
    print(
        "Columns:",
        final_df.columns.tolist()
    )

    if final_df.shape[1] != 22:
        raise ValueError(
            f"Seed {seed}: phải có 22 cột, "
            f"nhưng hiện có {final_df.shape[1]}."
        )

    if final_customers != expected_customers:
        raise ValueError(
            f"Seed {seed}: số khách cuối không khớp."
        )

    if duplicate_count > 0:
        raise ValueError(
            f"Seed {seed}: còn CUSTOMER_NUMBER bị trùng."
        )

    if missing_count > 0:
        raise ValueError(
            f"Seed {seed}: còn dữ liệu thiếu."
        )

    # --------------------------------------------------------
    # 10.9 Sort
    # --------------------------------------------------------

    final_df = (
        final_df
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 10.10 Save full dataset
    # --------------------------------------------------------

    final_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # 10.11 Remove AUTO-JOB
    # --------------------------------------------------------

    auto_job_mask = (
        final_df[
            "EB_REGISTER_CHANNEL"
        ]
        .astype("string")
        .str.strip()
        .str.upper()
        .eq("AUTO-JOB")
    )

    auto_job_count = int(
        auto_job_mask.sum()
    )

    final_no_auto_job = final_df.loc[
        ~auto_job_mask
    ].copy()

    final_no_auto_job.to_csv(
        no_auto_job_file,
        index=False,
        encoding="utf-8-sig"
    )

    missing_check.to_csv(
        missing_check_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== SAVED =====")
    print("Full dataset:", output_file)
    print(
        "No AUTO-JOB dataset:",
        no_auto_job_file
    )
    print(
        "AUTO-JOB removed:",
        auto_job_count
    )

    return {
        "SEED": seed,
        "ROWS_FULL": len(
            final_df
        ),
        "ROWS_NO_AUTO_JOB": len(
            final_no_auto_job
        ),
        "COLUMNS": final_df.shape[1],
        "TARGET_0": int(
            (
                final_df[
                    FINAL_TARGET_COL
                ] == 0
            ).sum()
        ),
        "TARGET_1": int(
            (
                final_df[
                    FINAL_TARGET_COL
                ] == 1
            ).sum()
        ),
        "AUTO_JOB_REMOVED": auto_job_count,
        "DUPLICATES": duplicate_count,
        "MISSING_VALUES": missing_count,
        "OUTPUT_FILE": (
            no_auto_job_file
        )
    }


# ============================================================
# 11. RUN ALL FIVE SEEDS
# ============================================================

run_results = []

for seed in SEEDS:

    result = build_final_dataset(
        seed
    )

    run_results.append(
        result
    )


# ============================================================
# 12. SAVE SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    run_results
)

summary_file = os.path.join(
    OUTPUT_DIR,
    "final_dataset_all_seeds_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 80)
print("ALL FIVE PLACEBO DATASETS COMPLETED")
print("=" * 80)

print(summary_df)

print("\nSummary file:")
print(summary_file)

Mounted at /content/drive

BUILDING FINAL PLACEBO DATASET — SEED 1
customer seed 1 shape: (150408, 20)
customer seed 1 duplicates: 0
activity seed 1 shape: (150408, 4)
activity seed 1 duplicates: 0
transaction seed 1 shape: (150408, 7)
transaction seed 1 duplicates: 0
deposit seed 1 shape: (150408, 6)
deposit seed 1 duplicates: 0
lending seed 1 shape: (150408, 5)
lending seed 1 duplicates: 0
Expected customers: 150408
After merging activity: (150408, 9)
After merging transaction: (150408, 15)
After merging deposit: (150408, 19)
After merging lending: (150408, 22)

===== FINAL VALIDATION =====
Shape: (150408, 22)
Expected customers: 150408
Final customers: 150408
Duplicate customers: 0
Total missing values: 0
Columns: ['CUSTOMER_NUMBER', 'CLIENT_SEX', 'EB_REGISTER_CHANNEL', 'SMS', 'VERIFY_METHOD', 'COUNT_CREDITCARD', 'AGE', 'LOGIN_PER_ACTIVE_DAY', 'INTEREST_RATE_RATIO', 'TRANS_LV1_MODE', 'TRANS_LV2_MODE', 'TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN', 'TRANS_LV1_MODE_CODE', 'TRANS_LV2_MODE_CO

  XÓA CÁC BẢN GHI CÓ EB_REGISTER_CHANEL LÀ AUTO_JOB

In [ ]:
import os
import pandas as pd
from google.colab import drive

# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive", force_remount=False)

# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets"

SEEDS = [1, 2, 3, 4, 5]

# ============================================================
# PROCESS EACH SEED
# ============================================================

for seed in SEEDS:

    print("=" * 70)
    print(f"PROCESSING SEED {seed}")
    print("=" * 70)

    INPUT_FILE = os.path.join(
        BASE_DIR,
        f"final_dataset_placebo_seed_{seed}.csv"
    )

    OUTPUT_FILE = os.path.join(
        BASE_DIR,
        f"final_dataset_placebo_seed_{seed}_no_auto_job.csv"
    )

    # --------------------------------------------------------
    # LOAD DATA
    # --------------------------------------------------------

    df = pd.read_csv(
        INPUT_FILE,
        low_memory=False
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )

    if "EB_REGISTER_CHANNEL" not in df.columns:
        raise ValueError(
            f"{INPUT_FILE} không có cột EB_REGISTER_CHANNEL"
        )

    before = len(df)

    # --------------------------------------------------------
    # REMOVE AUTO-JOB
    # --------------------------------------------------------

    mask = (
        df["EB_REGISTER_CHANNEL"]
        .astype(str)
        .str.strip()
        .str.upper()
        != "AUTO-JOB"
    )

    df = df.loc[mask].copy()

    after = len(df)

    removed = before - after

    print("Before :", before)
    print("After  :", after)
    print("Removed:", removed)

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", OUTPUT_FILE)

print("\nCompleted all seeds.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROCESSING SEED 1
Before : 150408
After  : 127460
Removed: 22948
Saved: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_1_no_auto_job.csv
PROCESSING SEED 2
Before : 150408
After  : 127460
Removed: 22948
Saved: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_2_no_auto_job.csv
PROCESSING SEED 3
Before : 150408
After  : 127460
Removed: 22948
Saved: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_3_no_auto_job.csv
PROCESSING SEED 4
Before : 150408
After  : 127460
Removed: 22948
Saved: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_4_no_auto_job.csv
PROCESSING SEED 5
Before : 150408
After  : 127460
Removed: 22948
Saved: /content/drive/MyDrive/ELAPSPLACEBO/final_placebo_datasets/final_dataset_placebo_seed_5_no_auto

CHẠY PIPLINE

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from google.colab import drive

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
    brier_score_loss
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler


warnings.filterwarnings("ignore")


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/ELAPSPLACEBO"

INPUT_DIR = os.path.join(
    BASE_DIR,
    "final_placebo_datasets"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "placebo_training_results"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

SEEDS = [1, 2, 3, 4, 5]

ID_COL = "CUSTOMER_NUMBER"
TARGET_COL = "COUNT_CREDITCARD"

TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
MODEL_RANDOM_STATE = 42
DECISION_THRESHOLD = 0.50

# Chạy SHAP cho cả 5 seed sẽ khá lâu.
# Có thể để False khi kiểm tra pipeline lần đầu.
RUN_SHAP = True

# Giới hạn số mẫu SHAP để tránh Colab kiệt sức.
SHAP_SAMPLE_SIZE = 5000


# ============================================================
# 3. FILE PATTERN
# ============================================================

INPUT_PATTERN = (
    "final_dataset_placebo_seed_{seed}_no_auto_job.csv"
)


# ============================================================
# 4. CÁC CỘT KHÔNG ĐƯA VÀO MÔ HÌNH
# ============================================================

EXCLUDE_COLS = [
    ID_COL,
    TARGET_COL,

    # Biến liên quan trực tiếp đến target
    "COUNT_CREDITCARD",
    "MAX_CARD",
    "FIRST_CARD_MONTH",

    # Biến tạo cutoff/placebo
    "TARGET_MONTH",
    "CLIENT_CREATE_DATE",
    "ACTUAL_D_STAR",
    "AVAILABLE_DAYS",
    "ASSIGNED_CUTOFF_DAYS",
    "PLACEBO_CUTOFF_DATE",
    "CUTOFF_TYPE",
    "ELIGIBLE_DSTAR_COUNT",
    "SAMPLING_STATUS",
    "PLACEBO_SEED",
    "TENURE_AT_CUTOFF",
    "TENURE_DAYS_AT_TARGET"
]


# ============================================================
# 5. MODEL
# Giữ nguyên tham số pipeline gốc
# ============================================================

MODEL_NAME = "XGBoost + RandomOverSampling (ROS)"


def get_xgb_model():

    return XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=MODEL_RANDOM_STATE,
        n_jobs=-1
    )


# ============================================================
# 6. HELPER: CHUẨN HÓA CUSTOMER_NUMBER
# ============================================================

def standardize_customer_id(
    series: pd.Series
) -> pd.Series:

    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )


# ============================================================
# 7. LOAD ONE SEED
# ============================================================

def load_seed_data(
    seed: int
) -> pd.DataFrame:

    input_file = os.path.join(
        INPUT_DIR,
        INPUT_PATTERN.format(
            seed=seed
        )
    )

    if not Path(input_file).exists():
        raise FileNotFoundError(
            f"Không tìm thấy file seed {seed}:\n"
            f"{input_file}"
        )

    df = pd.read_csv(
        input_file,
        low_memory=False
    )

    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )

    required_cols = [
        ID_COL,
        TARGET_COL
    ]

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Seed {seed} thiếu cột: {missing_cols}\n"
            f"Các cột hiện có: {df.columns.tolist()}"
        )

    df[ID_COL] = standardize_customer_id(
        df[ID_COL]
    )

    df[TARGET_COL] = pd.to_numeric(
        df[TARGET_COL],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            ID_COL,
            TARGET_COL
        ]
    ).copy()

    df[TARGET_COL] = (
        df[TARGET_COL]
        .astype(int)
    )

    invalid_target = ~df[
        TARGET_COL
    ].isin([0, 1])

    if invalid_target.any():
        raise ValueError(
            f"Seed {seed}: TARGET phải chỉ gồm 0 và 1."
        )

    duplicate_count = int(
        df[ID_COL]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"Seed {seed}: có {duplicate_count} "
            "CUSTOMER_NUMBER bị trùng."
        )

    # Chuẩn hóa missing
    df = df.replace(
        [
            "",
            " ",
            "NA",
            "N/A",
            "na",
            "null",
            "NULL",
            "None",
            "none"
        ],
        np.nan
    )

    df = (
        df
        .sort_values(ID_COL)
        .reset_index(drop=True)
    )

    print(
        f"Seed {seed} | shape={df.shape} | "
        f"positive rate={df[TARGET_COL].mean():.6f}"
    )

    return df


# ============================================================
# 8. KIỂM TRA 5 SEED CÓ CÙNG TẬP KHÁCH HÀNG KHÔNG
# ============================================================

seed_data = {}

for seed in SEEDS:
    seed_data[seed] = load_seed_data(
        seed
    )

reference_ids = set(
    seed_data[1][ID_COL]
)

reference_target = (
    seed_data[1]
    .set_index(ID_COL)[TARGET_COL]
    .sort_index()
)

for seed in SEEDS[1:]:

    current_ids = set(
        seed_data[seed][ID_COL]
    )

    missing_ids = reference_ids - current_ids
    extra_ids = current_ids - reference_ids

    print(
        f"Seed {seed} | missing IDs={len(missing_ids)} | "
        f"extra IDs={len(extra_ids)}"
    )

    if missing_ids or extra_ids:
        raise ValueError(
            f"Seed {seed} không có cùng tập khách hàng với seed 1."
        )

    current_target = (
        seed_data[seed]
        .set_index(ID_COL)[TARGET_COL]
        .sort_index()
    )

    target_difference = int(
        (
            current_target
            != reference_target
        ).sum()
    )

    if target_difference > 0:
        raise ValueError(
            f"Seed {seed}: có {target_difference} "
            "khách hàng có TARGET khác seed 1."
        )


# ============================================================
# 9. COMMON TRAIN / TEST SPLIT
#
# Tạo split từ seed 1 rồi giữ nguyên cho 5 seed
# ============================================================

split_source = seed_data[1][
    [
        ID_COL,
        TARGET_COL
    ]
].copy()

train_ids_array, test_ids_array = train_test_split(
    split_source[ID_COL],
    test_size=TEST_SIZE,
    stratify=split_source[TARGET_COL],
    random_state=SPLIT_RANDOM_STATE
)

TRAIN_IDS = set(
    train_ids_array.tolist()
)

TEST_IDS = set(
    test_ids_array.tolist()
)

print("\n===== COMMON TRAIN/TEST SPLIT =====")
print("Train customers:", len(TRAIN_IDS))
print("Test customers :", len(TEST_IDS))
print(
    "Overlap:",
    len(TRAIN_IDS.intersection(TEST_IDS))
)

split_table = split_source.copy()

split_table["SPLIT"] = np.where(
    split_table[ID_COL].isin(TEST_IDS),
    "TEST",
    "TRAIN"
)

split_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "common_train_test_split.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. PREPARE X, Y
# ============================================================

def prepare_features(
    df: pd.DataFrame
):

    feature_cols = [
        col for col in df.columns
        if col not in EXCLUDE_COLS
    ]

    X = df[
        feature_cols
    ].copy()

    y = df[
        TARGET_COL
    ].copy()

    # Giữ logic harmonize type từ pipeline gốc
    for col in X.columns:

        if X[col].dtype == "object":

            converted = pd.to_numeric(
                X[col],
                errors="coerce"
            )

            non_null_original = (
                X[col]
                .notna()
                .sum()
            )

            non_null_converted = (
                converted
                .notna()
                .sum()
            )

            if (
                non_null_original > 0
                and (
                    non_null_converted
                    / non_null_original
                ) >= 0.8
            ):
                X[col] = converted

            else:
                X[col] = (
                    X[col]
                    .astype("string")
                )

    cat_cols = (
        X.select_dtypes(
            include=[
                "object",
                "string",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    num_cols = [
        col for col in X.columns
        if col not in cat_cols
    ]

    return (
        X,
        y,
        feature_cols,
        cat_cols,
        num_cols
    )


# ============================================================
# 11. PREPROCESS FUNCTION
# Giữ nguyên logic pipeline gốc
# ============================================================

def fit_preprocess(
    X_tr,
    X_val,
    cat_cols
):

    X_tr = X_tr.copy()
    X_val = X_val.copy()

    for col in cat_cols:

        X_tr[col] = (
            X_tr[col]
            .astype("string")
        )

        X_val[col] = (
            X_val[col]
            .astype("string")
        )

    encoder = None

    if len(cat_cols) > 0:

        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-1
        )

        X_tr[cat_cols] = encoder.fit_transform(
            X_tr[cat_cols]
        )

        X_val[cat_cols] = encoder.transform(
            X_val[cat_cols]
        )

    imputer = SimpleImputer(
        strategy="median"
    )

    X_tr = pd.DataFrame(
        imputer.fit_transform(
            X_tr
        ),
        columns=X_tr.columns,
        index=X_tr.index
    )

    X_val = pd.DataFrame(
        imputer.transform(
            X_val
        ),
        columns=X_val.columns,
        index=X_val.index
    )

    return (
        X_tr,
        X_val,
        imputer,
        encoder
    )


# ============================================================
# 12. DECILE TABLE
# ============================================================

def create_decile_table(
    y_true,
    y_prob,
    n_groups=10
):

    decile_df = pd.DataFrame({
        "Y_TRUE": np.asarray(
            y_true
        ),
        "Y_PROB": np.asarray(
            y_prob
        )
    })

    decile_df = (
        decile_df
        .sort_values(
            "Y_PROB",
            ascending=False
        )
        .reset_index(drop=True)
    )

    decile_df["DECILE"] = pd.qcut(
        decile_df.index + 1,
        q=n_groups,
        labels=[
            f"D{i}"
            for i in range(
                1,
                n_groups + 1
            )
        ]
    )

    decile_summary = (
        decile_df
        .groupby(
            "DECILE",
            observed=False
        )
        .agg(
            CUSTOMERS=(
                "Y_TRUE",
                "count"
            ),
            POSITIVES=(
                "Y_TRUE",
                "sum"
            ),
            ADOPTION_RATE=(
                "Y_TRUE",
                "mean"
            ),
            AVG_PRED_PROB=(
                "Y_PROB",
                "mean"
            )
        )
        .reset_index()
    )

    overall_rate = (
        decile_df["Y_TRUE"]
        .mean()
    )

    decile_summary["LIFT"] = (
        decile_summary[
            "ADOPTION_RATE"
        ]
        / overall_rate
    )

    return (
        decile_summary,
        overall_rate
    )


# ============================================================
# 13. PRECISION@K
# ============================================================

def precision_at_k(
    y_true,
    y_prob,
    top_rates=[
        0.10,
        0.20,
        0.30
    ]
):

    result_df = pd.DataFrame({
        "Y_TRUE": np.asarray(
            y_true
        ),
        "Y_PROB": np.asarray(
            y_prob
        )
    })

    result_df = (
        result_df
        .sort_values(
            "Y_PROB",
            ascending=False
        )
        .reset_index(drop=True)
    )

    rows = []
    n = len(
        result_df
    )

    total_positives = (
        result_df["Y_TRUE"]
        .sum()
    )

    base_rate = (
        result_df["Y_TRUE"]
        .mean()
    )

    for rate in top_rates:

        k = int(
            np.ceil(
                n * rate
            )
        )

        top_df = result_df.iloc[
            :k
        ]

        precision_value = (
            top_df["Y_TRUE"]
            .mean()
        )

        captured_positives = (
            top_df["Y_TRUE"]
            .sum()
        )

        capture_rate = (
            captured_positives
            / total_positives
            if total_positives > 0
            else np.nan
        )

        lift = (
            precision_value
            / base_rate
            if base_rate > 0
            else np.nan
        )

        rows.append({
            "TOP_K": (
                f"Top {int(rate * 100)}%"
            ),
            "TOP_RATE": rate,
            "CUSTOMERS": k,
            "PRECISION": precision_value,
            "LIFT": lift,
            "CAPTURED_POSITIVES": (
                captured_positives
            ),
            "CAPTURE_RATE": capture_rate
        })

    return pd.DataFrame(
        rows
    )


# ============================================================
# 14. KS
# ============================================================

def calculate_ks(
    y_true,
    y_prob
):

    fpr, tpr, _ = roc_curve(
        y_true,
        y_prob
    )

    return float(
        np.max(
            tpr - fpr
        )
    )


# ============================================================
# 15. RUN ONE SEED
# ============================================================

def run_one_seed(
    seed: int
):

    print("\n" + "=" * 90)
    print(f"TRAINING PLACEBO DATASET — SEED {seed}")
    print("=" * 90)

    seed_output_dir = os.path.join(
        OUTPUT_DIR,
        f"seed_{seed}"
    )

    os.makedirs(
        seed_output_dir,
        exist_ok=True
    )

    df = seed_data[
        seed
    ].copy()

    train_df = df.loc[
        df[ID_COL].isin(
            TRAIN_IDS
        )
    ].copy()

    test_df = df.loc[
        df[ID_COL].isin(
            TEST_IDS
        )
    ].copy()

    (
        X_all,
        y_all,
        feature_cols,
        cat_cols,
        num_cols
    ) = prepare_features(
        df
    )

    X_train = X_all.loc[
        train_df.index
    ].copy()

    y_train = y_all.loc[
        train_df.index
    ].copy()

    X_test = X_all.loc[
        test_df.index
    ].copy()

    y_test = y_all.loc[
        test_df.index
    ].copy()

    print(
        "Dataset shape:",
        df.shape
    )

    print(
        "Number of model features:",
        len(feature_cols)
    )

    print(
        "Categorical columns:",
        cat_cols
    )

    print(
        "Numerical columns:",
        num_cols
    )

    print(
        "Train shape:",
        X_train.shape
    )

    print(
        "Test shape:",
        X_test.shape
    )

    print(
        "Train target rate:",
        y_train.mean()
    )

    print(
        "Test target rate:",
        y_test.mean()
    )

    # --------------------------------------------------------
    # 15.1 CROSS-VALIDATION
    # --------------------------------------------------------

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    recall_scores = []
    precision_scores = []
    f1_scores = []
    auc_scores = []
    acc_scores = []

    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        skf.split(
            X_train,
            y_train
        ),
        start=1
    ):

        X_tr = (
            X_train
            .iloc[train_idx]
            .copy()
        )

        X_val = (
            X_train
            .iloc[val_idx]
            .copy()
        )

        y_tr = (
            y_train
            .iloc[train_idx]
            .copy()
        )

        y_val = (
            y_train
            .iloc[val_idx]
            .copy()
        )

        (
            X_tr_prep,
            X_val_prep,
            _,
            _
        ) = fit_preprocess(
            X_tr,
            X_val,
            cat_cols
        )

        ros = RandomOverSampler(
            random_state=42
        )

        X_tr_res, y_tr_res = (
            ros.fit_resample(
                X_tr_prep,
                y_tr
            )
        )

        model = get_xgb_model()

        model.fit(
            X_tr_res,
            y_tr_res
        )

        y_val_prob = (
            model.predict_proba(
                X_val_prep
            )[:, 1]
        )

        y_val_pred = (
            y_val_prob
            >= DECISION_THRESHOLD
        ).astype(int)

        recall_value = recall_score(
            y_val,
            y_val_pred,
            zero_division=0
        )

        precision_value = precision_score(
            y_val,
            y_val_pred,
            zero_division=0
        )

        f1_value = f1_score(
            y_val,
            y_val_pred,
            zero_division=0
        )

        auc_value = roc_auc_score(
            y_val,
            y_val_prob
        )

        acc_value = accuracy_score(
            y_val,
            y_val_pred
        )

        recall_scores.append(
            recall_value
        )

        precision_scores.append(
            precision_value
        )

        f1_scores.append(
            f1_value
        )

        auc_scores.append(
            auc_value
        )

        acc_scores.append(
            acc_value
        )

        print(
            f"Fold {fold} | "
            f"Recall={recall_value:.4f} | "
            f"Precision={precision_value:.4f} | "
            f"F1={f1_value:.4f} | "
            f"AUC={auc_value:.4f} | "
            f"Accuracy={acc_value:.4f}"
        )

    cv_summary = pd.DataFrame([
        {
            "SEED": seed,
            "MODEL": MODEL_NAME,
            "CV_RECALL": np.mean(
                recall_scores
            ),
            "CV_PRECISION": np.mean(
                precision_scores
            ),
            "CV_F1": np.mean(
                f1_scores
            ),
            "CV_AUC": np.mean(
                auc_scores
            ),
            "CV_ACCURACY": np.mean(
                acc_scores
            ),
            "CV_AUC_STD": np.std(
                auc_scores,
                ddof=1
            )
        }
    ])

    cv_summary.to_csv(
        os.path.join(
            seed_output_dir,
            f"xgboost_ros_cv_seed_{seed}.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # 15.2 FINAL TRAINING
    # --------------------------------------------------------

    (
        X_train_prep,
        X_test_prep,
        final_imputer,
        final_encoder
    ) = fit_preprocess(
        X_train,
        X_test,
        cat_cols
    )

    ros = RandomOverSampler(
        random_state=42
    )

    X_train_res, y_train_res = (
        ros.fit_resample(
            X_train_prep,
            y_train
        )
    )

    final_model = get_xgb_model()

    final_model.fit(
        X_train_res,
        y_train_res
    )

    # --------------------------------------------------------
    # 15.3 TEST
    # --------------------------------------------------------

    y_test_prob = (
        final_model
        .predict_proba(
            X_test_prep
        )[:, 1]
    )

    y_test_pred = (
        y_test_prob
        >= DECISION_THRESHOLD
    ).astype(int)

    test_auc = roc_auc_score(
        y_test,
        y_test_prob
    )

    test_recall = recall_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    test_precision = precision_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    test_f1 = f1_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    test_acc = accuracy_score(
        y_test,
        y_test_pred
    )

    test_brier = brier_score_loss(
        y_test,
        y_test_prob
    )

    test_ks = calculate_ks(
        y_test,
        y_test_prob
    )

    cm = confusion_matrix(
        y_test,
        y_test_pred
    )

    tn, fp, fn, tp = cm.ravel()

    print("\n===== TEST RESULTS =====")

    print(
        f"AUC       : {test_auc:.6f}"
    )

    print(
        f"Recall    : {test_recall:.6f}"
    )

    print(
        f"Precision : {test_precision:.6f}"
    )

    print(
        f"F1        : {test_f1:.6f}"
    )

    print(
        f"Accuracy  : {test_acc:.6f}"
    )

    print(
        f"Brier     : {test_brier:.6f}"
    )

    print(
        f"KS        : {test_ks:.6f}"
    )

    print(
        "\nConfusion Matrix:\n",
        cm
    )

    print(
        "\nClassification Report:\n"
    )

    print(
        classification_report(
            y_test,
            y_test_pred,
            digits=4
        )
    )

    # --------------------------------------------------------
    # 15.4 SAVE PREDICTIONS
    # --------------------------------------------------------

    predictions = pd.DataFrame({
        ID_COL: test_df[
            ID_COL
        ].values,
        "Y_TRUE": y_test.values,
        "Y_PROB": y_test_prob,
        "Y_PRED": y_test_pred,
        "SEED": seed
    })

    predictions = (
        predictions
        .sort_values(
            "Y_PROB",
            ascending=False
        )
        .reset_index(drop=True)
    )

    predictions.to_csv(
        os.path.join(
            seed_output_dir,
            f"test_predictions_seed_{seed}.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # 15.5 DECILE
    # --------------------------------------------------------

    decile_df, overall_rate = (
        create_decile_table(
            y_test,
            y_test_prob
        )
    )

    decile_df["SEED"] = seed

    decile_df.to_csv(
        os.path.join(
            seed_output_dir,
            f"decile_table_seed_{seed}.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # 15.6 PRECISION@K
    # --------------------------------------------------------

    precision_k_df = precision_at_k(
        y_test,
        y_test_prob
    )

    precision_k_df[
        "SEED"
    ] = seed

    precision_k_df.to_csv(
        os.path.join(
            seed_output_dir,
            f"precision_at_k_seed_{seed}.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    top10 = precision_k_df.loc[
        precision_k_df[
            "TOP_RATE"
        ] == 0.10
    ].iloc[0]

    top20 = precision_k_df.loc[
        precision_k_df[
            "TOP_RATE"
        ] == 0.20
    ].iloc[0]

    top30 = precision_k_df.loc[
        precision_k_df[
            "TOP_RATE"
        ] == 0.30
    ].iloc[0]

    # --------------------------------------------------------
    # 15.7 ROC PLOT
    # --------------------------------------------------------

    fpr, tpr, _ = roc_curve(
        y_test,
        y_test_prob
    )

    plt.figure(
        figsize=(7, 5)
    )

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=(
            f"XGBoost + ROS "
            f"(AUC = {test_auc:.3f})"
        )
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1,
        label="Random baseline"
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.title(
        f"ROC Curve — Placebo Seed {seed}"
    )

    plt.legend(
        loc="lower right"
    )

    plt.grid(
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            seed_output_dir,
            f"roc_curve_seed_{seed}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 15.8 SHAP
    # --------------------------------------------------------

    top_shap_feature = None
    count_ca_rank = np.nan
    age_rank = np.nan

    if RUN_SHAP:

        if len(
            X_test_prep
        ) > SHAP_SAMPLE_SIZE:

            shap_indices = (
                X_test_prep
                .sample(
                    n=SHAP_SAMPLE_SIZE,
                    random_state=seed
                )
                .index
            )

            X_shap = (
                X_test_prep
                .loc[shap_indices]
                .copy()
            )

        else:
            X_shap = (
                X_test_prep
                .copy()
            )

        explainer = shap.TreeExplainer(
            final_model
        )

        shap_values = (
            explainer
            .shap_values(
                X_shap
            )
        )

        if isinstance(
            shap_values,
            list
        ):
            shap_values = (
                shap_values[1]
            )

        mean_abs_shap = (
            np.abs(
                shap_values
            )
            .mean(
                axis=0
            )
        )

        shap_importance = pd.DataFrame({
            "FEATURE": X_shap.columns,
            "MEAN_ABS_SHAP": (
                mean_abs_shap
            )
        })

        shap_importance = (
            shap_importance
            .sort_values(
                "MEAN_ABS_SHAP",
                ascending=False
            )
            .reset_index(drop=True)
        )

        shap_importance[
            "SHAP_RANK"
        ] = np.arange(
            1,
            len(
                shap_importance
            ) + 1
        )

        shap_importance.to_csv(
            os.path.join(
                seed_output_dir,
                f"shap_importance_seed_{seed}.csv"
            ),
            index=False,
            encoding="utf-8-sig"
        )

        if not shap_importance.empty:

            top_shap_feature = (
                shap_importance
                .iloc[0][
                    "FEATURE"
                ]
            )

        count_match = (
            shap_importance.loc[
                shap_importance[
                    "FEATURE"
                ] == "COUNT_CA_ACCT"
            ]
        )

        if not count_match.empty:

            count_ca_rank = int(
                count_match
                .iloc[0][
                    "SHAP_RANK"
                ]
            )

        age_match = (
            shap_importance.loc[
                shap_importance[
                    "FEATURE"
                ] == "AGE"
            ]
        )

        if not age_match.empty:

            age_rank = int(
                age_match
                .iloc[0][
                    "SHAP_RANK"
                ]
            )

        plt.figure()

        shap.summary_plot(
            shap_values,
            X_shap,
            show=False,
            max_display=20
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                seed_output_dir,
                f"shap_summary_seed_{seed}.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

        plt.close()

    # --------------------------------------------------------
    # 15.9 RETURN SUMMARY
    # --------------------------------------------------------

    result = {
        "SEED": seed,

        "N_TRAIN": len(
            X_train
        ),
        "N_TEST": len(
            X_test
        ),
        "N_FEATURES": len(
            feature_cols
        ),
        "TEST_BASE_RATE": (
            y_test.mean()
        ),

        "CV_RECALL": np.mean(
            recall_scores
        ),
        "CV_PRECISION": np.mean(
            precision_scores
        ),
        "CV_F1": np.mean(
            f1_scores
        ),
        "CV_AUC": np.mean(
            auc_scores
        ),
        "CV_AUC_STD": np.std(
            auc_scores,
            ddof=1
        ),
        "CV_ACCURACY": np.mean(
            acc_scores
        ),

        "TEST_AUC": test_auc,
        "TEST_RECALL": test_recall,
        "TEST_PRECISION": test_precision,
        "TEST_F1": test_f1,
        "TEST_ACCURACY": test_acc,
        "TEST_BRIER": test_brier,
        "TEST_KS": test_ks,

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,

        "PRECISION_AT_10": (
            top10["PRECISION"]
        ),
        "LIFT_AT_10": (
            top10["LIFT"]
        ),
        "CAPTURE_RATE_AT_10": (
            top10["CAPTURE_RATE"]
        ),

        "PRECISION_AT_20": (
            top20["PRECISION"]
        ),
        "LIFT_AT_20": (
            top20["LIFT"]
        ),
        "CAPTURE_RATE_AT_20": (
            top20["CAPTURE_RATE"]
        ),

        "PRECISION_AT_30": (
            top30["PRECISION"]
        ),
        "LIFT_AT_30": (
            top30["LIFT"]
        ),
        "CAPTURE_RATE_AT_30": (
            top30["CAPTURE_RATE"]
        ),

        "TOP_SHAP_FEATURE": (
            top_shap_feature
        ),
        "COUNT_CA_ACCT_SHAP_RANK": (
            count_ca_rank
        ),
        "AGE_SHAP_RANK": age_rank
    }

    return result


# ============================================================
# 16. RUN 5 SEEDS
# ============================================================

all_results = []

for seed in SEEDS:

    seed_result = run_one_seed(
        seed
    )

    all_results.append(
        seed_result
    )


# ============================================================
# 17. DETAILED RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_results
)

detailed_file = os.path.join(
    OUTPUT_DIR,
    "placebo_xgboost_ros_results_5_seeds.csv"
)

results_df.to_csv(
    detailed_file,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 18. MEAN ± SD
# ============================================================

SUMMARY_METRICS = [
    "CV_AUC",
    "CV_RECALL",
    "CV_PRECISION",
    "CV_F1",
    "CV_ACCURACY",

    "TEST_AUC",
    "TEST_RECALL",
    "TEST_PRECISION",
    "TEST_F1",
    "TEST_ACCURACY",
    "TEST_BRIER",
    "TEST_KS",

    "PRECISION_AT_10",
    "LIFT_AT_10",
    "CAPTURE_RATE_AT_10",

    "PRECISION_AT_20",
    "LIFT_AT_20",
    "CAPTURE_RATE_AT_20",

    "PRECISION_AT_30",
    "LIFT_AT_30",
    "CAPTURE_RATE_AT_30"
]

summary_rows = []

for metric in SUMMARY_METRICS:

    values = (
        results_df[
            metric
        ]
        .astype(float)
    )

    mean_value = (
        values.mean()
    )

    std_value = (
        values.std(
            ddof=1
        )
    )

    summary_rows.append({
        "METRIC": metric,
        "MEAN": mean_value,
        "STD": std_value,
        "MIN": values.min(),
        "MAX": values.max(),
        "MEAN_PLUS_MINUS_SD": (
            f"{mean_value:.4f} ± "
            f"{std_value:.4f}"
        )
    })

summary_df = pd.DataFrame(
    summary_rows
)

summary_file = os.path.join(
    OUTPUT_DIR,
    "placebo_xgboost_ros_mean_sd.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. PAPER TABLE
# ============================================================

paper_table = results_df[
    [
        "SEED",
        "TEST_AUC",
        "TEST_RECALL",
        "TEST_F1",
        "TEST_ACCURACY",
        "TEST_BRIER",
        "TEST_KS",
        "PRECISION_AT_10",
        "LIFT_AT_10",
        "CAPTURE_RATE_AT_10",
        "PRECISION_AT_20",
        "LIFT_AT_20",
        "PRECISION_AT_30",
        "LIFT_AT_30",
        "TOP_SHAP_FEATURE",
        "COUNT_CA_ACCT_SHAP_RANK",
        "AGE_SHAP_RANK"
    ]
].copy()

mean_row = {
    "SEED": "Mean"
}

sd_row = {
    "SEED": "SD"
}

for col in paper_table.columns:

    if col == "SEED":
        continue

    if col == "TOP_SHAP_FEATURE":
        mean_row[col] = ""
        sd_row[col] = ""
        continue

    mean_row[col] = (
        pd.to_numeric(
            paper_table[col],
            errors="coerce"
        )
        .mean()
    )

    sd_row[col] = (
        pd.to_numeric(
            paper_table[col],
            errors="coerce"
        )
        .std(
            ddof=1
        )
    )

paper_table = pd.concat(
    [
        paper_table,
        pd.DataFrame(
            [
                mean_row,
                sd_row
            ]
        )
    ],
    ignore_index=True
)

paper_file = os.path.join(
    OUTPUT_DIR,
    "placebo_paper_table.csv"
)

paper_table.to_csv(
    paper_file,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 20. EXCEL RESULTS
# ============================================================

excel_file = os.path.join(
    OUTPUT_DIR,
    "placebo_xgboost_ros_results.xlsx"
)

with pd.ExcelWriter(
    excel_file
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="Seed results",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Mean SD",
        index=False
    )

    paper_table.to_excel(
        writer,
        sheet_name="Paper table",
        index=False
    )

    split_table.to_excel(
        writer,
        sheet_name="Common split",
        index=False
    )


# ============================================================
# 21. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 90)
print("PLACEBO XGBOOST + ROS EXPERIMENT COMPLETED")
print("=" * 90)

print("\n===== TEST RESULTS =====")

print(
    results_df[
        [
            "SEED",
            "TEST_AUC",
            "TEST_RECALL",
            "TEST_F1",
            "TEST_ACCURACY",
            "TEST_BRIER",
            "TEST_KS",
            "PRECISION_AT_10",
            "LIFT_AT_10",
            "CAPTURE_RATE_AT_10"
        ]
    ]
)

print("\n===== MEAN ± SD =====")

print(
    summary_df[
        [
            "METRIC",
            "MEAN_PLUS_MINUS_SD"
        ]
    ]
)

print("\nSaved:")
print(detailed_file)
print(summary_file)
print(paper_file)
print(excel_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Seed 1 | shape=(127460, 22) | positive rate=0.154668
Seed 2 | shape=(127460, 22) | positive rate=0.154668
Seed 3 | shape=(127460, 22) | positive rate=0.154668
Seed 4 | shape=(127460, 22) | positive rate=0.154668
Seed 5 | shape=(127460, 22) | positive rate=0.154668
Seed 2 | missing IDs=0 | extra IDs=0
Seed 3 | missing IDs=0 | extra IDs=0
Seed 4 | missing IDs=0 | extra IDs=0
Seed 5 | missing IDs=0 | extra IDs=0

===== COMMON TRAIN/TEST SPLIT =====
Train customers: 101968
Test customers : 25492
Overlap: 0

TRAINING PLACEBO DATASET — SEED 1
Dataset shape: (127460, 22)
Number of model features: 20
Categorical columns: ['CLIENT_SEX', 'EB_REGISTER_CHANNEL', 'SMS', 'VERIFY_METHOD', 'TRANS_LV1_MODE', 'TRANS_LV2_MODE']
Numerical columns: ['AGE', 'LOGIN_PER_ACTIVE_DAY', 'INTEREST_RATE_RATIO', 'TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN', 'TRANS_LV1_MODE_CODE', 'TRANS_LV2_MOD